# Section 3 : Le transport de la donnée

## Présentation de la section

#### Présentation de la section

# Le transport de la donnée

![Data transport](images/noun_IoT_3265860.png)

Durant cette troisième semaine, nous allons continuer notre exploration du transport des données dans l'internet des objets. Vous verrez en particulier l'architecture logicielle REST (*REpresentational State Transfer*) et le protocole de transport CoAP (*Constrained Application Protocol*).

Vous pousserez un peu plus loin le TP de la section précédente en utilisant ce protocole pour structurer et envoyer des données depuis notre objet connecté vers un serveur.

### À la fin de cette semaine, vous serez capable de :





+ décrire l'architecture logicielle REST,
+ décrire le protocole CoAP,
+ émettre des données structurées sur un réseau IoT avec un objet et recevoir ces données sur un serveur en utilisant REST et CoAP.

Bonne semaine !

*L'équipe pédagogique*

## Découvrons CoAP

### Pourquoi CoAP ?

#### Pourquoi CoAP ?

# Pourquoi CoAP ?

![](images/CoAP.png)

*Les principes REST avec leur représentation de l'information par des ressources pointées par des identifiants globalement unique est une des clés du succès de l'Internet et de la composition de services distribués. Même si ce n'est pas l'unique solution, il est indispensable que les informations provenant d'objets contraints puissent s'intégrer dans cette toile d'araignée mondiale. CoAP, pour* Contraint Application Protocol, *permet cette intégration à un meilleur coût en termes d'empreinte mémoire ou protocolaire que ne le permettrait le protocole HTTP.*

Dans la section précédente, vous avez appris que CBOR permettait d’envoyer des données structurées de manière efficace, que le récepteur pouvait faire la différence entre un entier ou une chaîne de caractères et également savoir combien d’éléments composaient un dictionnaire ou un tableau. En plus d’un gain de place par rapport à JSON, la complexité pour sérialiser ou désérialiser était limitée, conduisant à des implémentations peu gourmandes en mémoire.

Mais CBOR n’est pas suffisant pour une bonne interopérabilité. Quand un récepteur reçoit les données, il faut qu'il sache qu’il s’agit d’un codage CBOR et pas d'une autre structure. De plus, il faut que le récepteur sache quoi faire de ces données. Dans les exercices, nous avons transmis des séries temporelles correspondant à des relevés de température. Mais si nous voulions également transmettre l’humidité et la pression, comment le récepteur ferait la différence ?

Nous avons une solution pour répondre à ces questions : **l’utilisation de ressources**.

Nous avons vu qu’avec le paradigme REST, les ressources étaient nommées. Donc, pour distinguer les différentes séries temporelles, il suffit d’utiliser un nom (ou un URI) différent. Les ressources contiennent également des méta-informations et il est donc possible de transporter le format de codage pour indiquer qu’il s’agit de CBOR, de JSON, de CSV...

Malgré son universalité, ce modèle pose un problème pour les objets contraints :

* HTTP utilise TCP pour fiabiliser les communications entre le client et le serveur. Or, TCP est gourmand en ressources. Il faut de la mémoire pour stocker les paquets non acquittés ou hors séquence, un grand nombre d’heuristiques doivent être mises en œuvre pour améliorer ses performances ;
* la souplesse pour créer des en-têtes peut s'avérer être un désavantage pour un objet contraint. Ainsi, la requête HTTP vers le serveur www.arduino.cc peut avoir cette forme :

```
GET / HTTP/1.1\ r \n
Host: www.arduino.cc\r\n
User−Agent: Mozilla/5.0 (X11;Ubuntu;Linuxx8664;rv:25.0) Gecko /20100101Firefox/25.0\ r\n
Accept: text/html,application/xhtml+xml,application/xml;q=0.9,∗/∗;q=0.8\r\n
Accept−Language: en−US,en;q=0.5\r\n
Accept−Encoding:gzip,deflate\r\n
Connection: keep-alive\r\n
\r \n
```

En dessous de la première ligne qui demande la ressource à la racine (généralement index.html), on trouve un certain nombre de lignes sous le format :

Nom du champ : valeur du champ\r\n

qui vont indiquer au serveur le nom du serveur que l’on veut atteindre, la description du navigateur et les formats que celui-ci peut accepter.

Le but de CoAP est de définir un protocole beaucoup plus strict qui sera donc plus facile à mettre en œuvre mais qui pourra interopérer avec HTTP afin de préserver les principes définis par l’architecture REST et profiter du nommage des ressources pour que les ressources contraintes participent à la grande toile d'araignée mondiale.

### Format des messages CoAP

#### Vidéo



In [ ]:
from IPython.display import YouTubeVideo
YouTubeVideo("MqBAJ-qwbnU", width=700, height=394)

#### En-tête CoAP

L’en-tête des messages CoAP est de taille variable mais très structurée, comme le montre la figure suivante :

![](images/im29.png)

Le premier mot de 32 bits est présent dans tous les messages CoAP :

* le champ Ver, sur 2 bits, contient le numéro de version du protocole, qui vaut 01 dans la version actuelle ;
* le champ T pour Type, également sur 2 bits, indique la nature du message (00 : CONfirmable, 01 : NON confirmable, 10 : Acquittement, 11 : Reset) ;
* le champ TKL, sur 4 bits, donne la longueur en octets du champ *token* démarrant au deuxième mot de 32 bits. Si la valeur est 0, ce champ est absent. Les valeurs de 1 à 8 indiquent la longueur. Les valeurs de 9 à 15 ne sont pas autorisées ;
* le champ Code, sur 1 octet, permet un codage assez subtil de la nature de la requête ou de la réponse (cf. chapitre suivant) ;
* le champ Message ID, sur 2 octets, identifie les requêtes.

### Codage de code

Dans beaucoup de protocoles applicatifs comme FTP ou HTTP, le serveur renvoie un code sur 3 caractères indiquant si la requête s'est exécutée correctement ou non. Les codes commençant par le chiffre :

* 1 informent que la requête est en train d’être traitée normalement. Ce type de notification n’est pas pris en compte avec CoAP ;
* 2 indiquent que la requête a été acceptée et traitée correctement ;
* 3 permettent d’indiquer une indirection ;
* 4 font référence à une erreur du coté client, due à une mauvaise syntaxe ou une requête qui ne peut être traitée. Ainsi la célèbre erreur 404 indique que le client a demandé une page qui n’existe pas sur le serveur ;
* 5 désignent une erreur du coté du serveur.

Le site web de l'IANA (<http://www.iana.org/assignments/http-status-codes/http-status-codes.xhtml\#http-status-codes-1>) donne les erreurs que l’on retrouve dans le protocole HTTP. Comme indiqué précédemment, le chiffre de gauche varie entre 1 et 5 tandis que les deux chiffres de droite, précisant la raison de la notification, varient généralement entre 0 et 31.

Pour permettre une représentation plus compacte, CoAP va coder cette chaîne de caractères dans un octet. Les trois bits de gauche désignent la nature du code et les 5 à droite donneront la raison. Ainsi le code d’erreur HTTP 415 (*Unsupported Media Type*) se note en CoAP 4.15, s’écrit en binaire 100.01111 et en décimal 143. Cette notation concerne les réponses aux requêtes mais elle laisse de la place pour coder également les requêtes. En effet, le code avec les trois premiers bits à 0 n’est pas utilisé pour coder les notifications. Plusieurs requêtes compatibles avec l’architecture REST peuvent être codées : 

* GET, codé 0x01, retrouve le contenu d’une ressource présente sur le serveur et désignée par un URI ;
* POST, codé 0x02, stocke une valeur sur une ressource existante présente sur le serveur ;
* PUT, codé 0x03, crée une ressource sur le serveur et lui affecte une valeur ;
* DELETE, codé 0x04, supprime une ressource sur le serveur.

Notez que la valeur 0x00 peut être utilisée dans certains cas.

### **Message ID**

Le champ Message ID sur 2 octets sert à identifier les messages CoAP afin de détecter les duplicatas. Cette valeur est recopiée dans les acquittements pour permettre de savoir quel message est acquitté. Ils ne doivent pas être réutilisés pendant une période fixée.

### Tokens

CoAP utilise le protocole UDP pour communiquer. Contrairement à TCP, il n’y a pas de notion d’établissement de connexion. Il est donc difficile de faire le lien entre les établissements et les réponses, surtout si elles ne sont pas immédiates. La figure suivante illustre ce phénomène. Une requête GET est envoyée par un client à un serveur.

![](images/im31.png)

La réponse ne peut pas être immédiate (par exemple il faut lire une valeur sur un capteur qui demande d’être activé). Le message d'acquittement ne peut pas être différé sinon, le client ne voyant pas sa requête acquittée, la retransmettrait. Le serveur acquitte avec un message Ack vide. Quand le serveur peut envoyer la ressource, il le fait à son tour dans un message de type CON qui sera à son tour acquitté. Vous pouvez remarquer que les valeurs du champ Message ID sont complètement décorrélées. Pour faire le lien entre la requête et la réponse, un *token* fourni par le client est recopié par le serveur. C'est pour cela que l'on peut considérer une valeur de token comme une "connexion" entre le client et le serveur.

Le token est une séquence binaire facultative dont la taille est comprise entre 0 (pas de token) et 8 octets. La longueur est indiquée au début de l’en-tête dans le champ ToKen Length (TKL) et la valeur suit immédiatement l'en-tête obligatoire avant les options.

#### Options

# Les options CoAP

Le champ Option va contenir des options qui vont soit servir à améliorer le protocole de transfert des données entre le client et le serveur, soit servir à coder les en-têtes des requêtes et des réponses en garantissant une certaine compatibilité avec les en-têtes HTTP.

La structure utilisée est dite TLV pour Type Longueur Valeur. Chaque champ contient au moins ces deux informations :  
- Type indiquant la nature de l’option ;  
- Longueur indiquant la taille des données en octets. Si ce dernier n’est pas nul, les données vont se trouver après.

![](images/im32.png)

CoAP complique un peu la chose en optant pour un codage différentiel de la valeur de l’option. Ainsi, si l’on doit envoyer une option de type 5 puis deux de type 6, le codage contiendra ∆T = 5, ∆T = 1, ∆T = 0.

Mais comme le champ ∆T ne fait que 4 bits, on ne peut pas aller bien loin pour coder ces valeurs. Un mécanisme d'échappement est mis en place pour les différences supérieures à 13. Dans ce cas, la valeur 13 est mise dans le champ ∆T et l'octet suivant code la différence moins 13.

Par exemple, si l'on doit coder deux options de valeurs 5 et 20, la différence est de 15. La première option est codée normalement avec le ∆T à 5. Pour la seconde option, le ∆T est mis à 13 et l'octet suivant prendra la valeur 2.

Notez que la valeur 14 mise dans le champ ∆T indique que la différence nécessite deux octets pour être codée.

Pour la longueur on retrouve le même principe : les longueurs inférieures à 13 sont codées directement ; si elles sont supérieures ou égale à 13, la valeur moins 13 est codée dans un octet supplémentaire. Une valeur de 14 indique que deux octets sont utilisés pour coder la longueur moins 269.

La figure en haut de page illustre le codage d’une option dans l’en-tête d’un message CoAP.

Il se peut qu’il y ait des données après les options. Dans ce cas, un séparateur avec la valeur 0xFF est inséré. Il ne peut pas être confondu avec le codage d’une option puisque les champs ∆T et Longueur n'évoluent qu'entre 0 et 14.

S’il n’y a pas de données à transmettre (par exemple dans le cas d’une requête GET), le message CoAP se termine après les options.

#### Liste des options

### Options CoAP

Le premier bit sert à coder le type décrit. Quand il est positionné à 1, ce type doit être connu du récepteur (critique). Dans ce cas, si un destinataire reçoit une option de ce type et qu’il ne la connaît pas, il doit produire un message d’erreur. Dans le cas contraire, cette option est ignorée du récepteur qui poursuit le traitement des options suivantes. Ainsi, les options paires sont facultatives et les impaires, critiques. Les options apparaissant en gras seront traitées plus en détail dans la suite de ce chapitre.

| Valeur | Nom | Type | Nature | répété | Commentaire |
| --- | --- | --- | --- | --- | --- |
| 0 | Réserve |  |  |  |  |
| 1 | If-Match | opaque | critique | oui | Utilisé pour indiquer au serveur de n'effectuer la requête que sous certaines conditions. |
| 3 | Uri-Host | string | critique |  | Contient le nom du serveur d'une URI (nom, adresse IPv4 ou IPv6). Généralement, il n'est pas nécessaire de le préciser puisque les messages CoAP sont envoyés à cette adresse. |
| 4 | ETag | opaque | facultative | oui | Utilisé pour gérer la mise en cache des ressources. |
| 5 | If-None-Match | vide | critique |  | Utilisé pour indiquer au serveur de n'effectuer la requête que sous certaines conditions. |
| **6** | **Observe** | **entier** | **facultative** |  | **Permet à un serveur d'envoyer une requête aux changements d'état d'une ressource. Dans la réponse, la valeur doit toujours augmenter.** |
| 7 | Uri-Port | entier | critique |  | Contient le numéro du port UDP sur lequel CoAP est lancé. Généralement, ce champ n'est pas nécessaire vu que le serveur CoAP attend déjà des messages sur ce port. |
| 8 | Location-Path | string | facultative | oui | Utilisé en réponse à une requête POST pour indiquer un segment du chemin de la ressource. |
| **11** | **Uri-Path** | **string** | **facultative** | **oui** | **Contient un des segments de l'URI.** |
| **12** | **Content-Format** | **entier** | **facultative** |  | **Définit le format dans lequel sont codées des données.** |
| 14 | Max-Age | entier | facultative |  | Durée pendant laquelle la ressource peut être mise en cache. |
| **15** | **Uri-Query** | **string** | **critique** | **oui** | **Contient les segments d'interrogation que l'on retrouve dans les URI.** |
| **17** | **Accept** | **entier** | **critique** |  | **Indique les formats que le client peut accepter.** |
| 20 | Location-Query | string | facultative | oui | Utilisé en réponse à une requête POST pour indiquer le chemin de la ressource. |
| 35 | Proxy-Uri | string | critique |  | Contient une URI qui doit être prise en compte par le proxy. |
| 35 | Proxy-Scheme | string | critique |  | Indique le format de l'URI et le protocole dans une URL. |
| 60 | Size1 | entier | facultative |  | Indique la taille de la ressource. |
| **258** | **No-Response** | **entier** | **facultative** |  | **Limite les notifications REST.** |

### Le champ Message ID

#### Vidéo



In [ ]:
from IPython.display import YouTubeVideo
YouTubeVideo("BbMt49T9EV4", width=700, height=394)

#### Le champ Message ID

**Message ID**

Le champ Message ID, sur 2 octets, sert à identifier les messages CoAP afin de détecter les duplicatas liés à des retransmissions, ou à fiabiliser la transmission des messages confirmables. Dans ce cas, la valeur est recopiée dans les acquittements.

Dans l'exemple suivant, la première transaction se déroule correctement. L'émetteur envoie un message confirmable avec la valeur de message ID 0x1A8E, arme un temporisateur, et reçoit un acquittement avec la même valeur de Message ID avant que le temporisateur n'expire. La donnée est donc acquittée côté émetteur.

Dans le deuxième échange, la donnée n'est pas acquittée dans la durée allouée par le temporisateur ; elle est donc retransmise.

Le standard [RFC 7252](https://tools.ietf.org/html/rfc7252) suggère un temporisateur initial de 2 secondes dont la valeur double à chaque retransmission, et 4 transmissions sont proposées. Ces valeurs peuvent être changées pour s'adapter au contexte.

![](images/Capture28.png)

Les messages non confirmés possèdent également un message ID unique qui permet surtout d'éviter les duplications qui pourraient survenir lors du transport du message.

Pour rejeter les doublons, le récepteur doit garder une copie des Message ID émis par une source. Mais pour éviter de saturer la mémoire du récepteur avec ces valeurs, un simple calcul permet de définir la période de rétention des valeurs.

Un peu d’algèbre élémentaire permet de calculer cette durée. La figure suivante montre le calcul du pire cas. On prend le délai de propagation maximum dans les deux directions et on suppose que toutes les requêtes se perdent sauf la dernière qui est complète et on ajoute un temps de traitement important dans le récepteur.

![](https://studio.fun-mooc.fr/asset-v1:MinesTelecom+04038+session01+type@asset+block/im30.png)

Le standard prévoit que, par défaut, la durée d’activité d’un *Message ID* est d’environ 5 minutes (247 s) pour les messages confirmés, et 2,5 minutes (145 s) pour les messages non confirmés.

Avoir cette notion en tête peut vous éviter des heures de débogage. Supposons qu'un client commence toujours par numéroter ses messages à 1. Le récepteur va donc garder les valeurs des messages ID pendant 5 minutes. Si vous redémarrez l'objet, il va émettre de nouveaux messages, mais avec les mêmes Messages ID qui seront ignorés par le récepteur.

Finalement, la notion d'émetteur/récepteur est dissociée du concept REST de client/serveur. Un client peut être émetteur ou récepteur ; de même pour le serveur.

### Les Tokens

#### Vidéo



In [ ]:
from IPython.display import YouTubeVideo
YouTubeVideo("whg3BsUkxkE", width=700, height=394)

#### Les tokens

### Tokens

CoAP utilise le protocole UDP pour communiquer. Contrairement à TCP, il n’y a pas de notion d’établissement de connexion. Il est donc difficile de faire le lien entre les établissements et les réponses, surtout si elles ne sont pas immédiates. La figure suivante illustre ce phénomène. Une requête GET est envoyée par un client à un serveur.

![](https://studio.fun-mooc.fr/asset-v1:MinesTelecom+04038+session01+type@asset+block/im31.png)

La réponse ne peut pas être immédiate (par exemple il faut lire une valeur sur un capteur qui demande d’être activé). Le message d'acquittement ne peut pas être différé sinon, le client ne voyant pas sa requête acquittée, il la retransmettrait. Le serveur acquitte avec un message Ack vide.

Quand le serveur peut envoyer la ressource, il le fait à son tour dans un message de type CON qui sera à son tour acquitté. Vous pouvez remarquer que les valeurs du champ Message ID sont complètement décorrélées et que client et serveur sont à tour de rôle émetteur et récepteur.

Pour faire le lien entre la requête et la réponse, un *token* fourni par le client est recopié par le serveur. On peut considérer une valeur de *token* comme une connexion entre le client et le serveur.

Le *token* est une séquence binaire dont la taille est comprise entre 1 et 8 octets. La longueur est indiquée au début de l’en-tête obligatoire et la valeur suit immédiatement cet en-tête obligatoire.

### Des options "REST"

#### Vidéo



In [ ]:
from IPython.display import YouTubeVideo
YouTubeVideo("k8ml9PEy5t0", width=700, height=394)

#### Les URI

### Représentation des URI

On comprend mieux la signification de certaines options données dans le tableau précédent quand la syntaxe d’un URI est connue. (voir [RFC2396](https://tools.ietf.org/html/rfc2396)) . On a déjà vu que la syntaxe générale est :

<scheme>:<scheme-specific-part>

où *scheme* va définir le schéma de notation. De manière générale, le schéma va indiquer comment est structuré la suite de l'URI. Quand l'URI est aussi un localisateur (donc un URL), le schéma fait référence au protocole qui pourra être utilisé pour retrouver la ressource comme http ou https voire coap. Après le schéma, on trouve deux zones, l'autorité qui va indiquer qui est responsable de nommer les ressources. Dans le cas d'un URL, il peut s’écrire de la manière suivante :

<scheme>://*userinfo@*host*:port*/path?query

où les champs en italique *userinfo@* et *:port* sont facultatifs. Ils contiennent respectivement le nom de l’utilisateur et le numéro de port sur lequel tourne le service.

path va être composé d’une série de segments séparés par des caractères / qui identifient la ressource sur le serveur. L’URI peut se terminer par des questions, c’est-à-dire une chaîne de caractères qui sera interprétée par le serveur précédemment désigné. Une question, c'est-à-dire des paramètres fournis pour construire la ressource, peut contenir plusieurs parties séparées par le caractère &.

Ceci peut être vérifié par le programme de désassemblage. L’URI :

coap://192.168.1.52/capteur1/temperature?max value&date=20131206

utilise le schéma de nommage de coap. Le serveur est 192.168.1.52, le chemin *(path)*est composé de deux segments, suivi par deux questions. Le serveur reçoit la requête suivante : 

```
Received packet of size 50
40 01 BE BF|B8 63 61 70 74 - 65 75 72 31|0B 74 65 @.?¿¸capt eur1.te
6D 70 65 72 61 74 75 72 65 -|49 6D 61 78 5F 76 61 mperature Imax_va
6C 75 65|0D 00 64 61 74 65 - 3D 32 30 31 33 31 32 lue..date =201312
30 36 06
ver:1 Type = 0 (CON) Token Length = 0 code 1 (GET) Msg id = BEBF
Option = 11 (+11) length = 8
Uri-Path capteur1
Option = 11 (+0) length = 11
Uri-Path temperature
Option = 15 (+4) length = 9
Uri-Query max_value
Option = 15 (+0) length = 13
Uri-Query date=20131206
```

Le listing précédent montre ce que reçoit le serveur. L'URI n'est pas complète. Seules les parties "chemin" et "question" sont indiquées. Le schéma coap: n’est pas précisé ; de même que Uri-Host et Uri-Port car le serveur connaît son adresse IP et le numéro de port sur lequel s’exécute le serveur CoAP.

Ils pourraient être utiles en cas de virtualisation du serveur, c’est-à-dire si plusieurs instances de CoAP tournaient, soit à des noms différents, soit sur des numéros de port différents. Si cette possibilité existe, pour l’instant, la faible capacité des ressources ne pousse pas vers une virtualisation.

Si on reprend la partie optionnelle représentée en gras dans le listing précédent, la première option qui commence par l'octet 0xb8. 0xb (=11) indique qu'il s'agit d'une option *Uri-path* (comme c'est la première option, le delta se confond avec la valeur de l'option) et de longueur 8 octets qui correspondent à la valeur "capteur1". La seconde option débute par 0x0B. Le delta est nul. On reste sur une option *Uri-path* de longueur de 11 octets. La troisième option s'ouvre avec l'octet 0x49. L'incrément étant de 4, le numéro de l'option passe à 15 soit *Uri-query* sur 9 octets. L'option suivante démarre par 0x0D. On reste sur une option *Uri-query* mais la longueur 0xd informe que l'octet suivant contient la longueur. La valeur vaut étrangement 00 car elle est diminuée de 13 pour respecter le codage défini par CoAP.

**Quiz : Quizz**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Que représente ce message ?**"))
q0_w = widgets.RadioButtons(options=['Une requête GET', 'Une requête POST', 'Une requête PUT', 'Une requête DELETE', 'Une réponse'], description="", layout=widgets.Layout(width="90%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value == 'Une requête POST', ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)

display(Markdown("**Quelle est la valeur du champ token ?**"))
q1_w = widgets.RadioButtons(options=['Vide', '0xb4', '0xb474', '0xb47465', '0xb474656d'], description="", layout=widgets.Layout(width="90%"))
q1_b = widgets.Button(description="Verifier")
q1_o = widgets.Output()
def q1_check(_):
    q1_o.clear_output()
    with q1_o:
        display(_feedback(q1_w.value == 'Vide', ""))
q1_b.on_click(q1_check)
display(q1_w, q1_b, q1_o)

display(Markdown("**Quel élément de l'URI contient ce message ?**"))
q2_w = widgets.RadioButtons(options=['Aucun', '/temp', '/temp/sens1 et /max', '/temp/sens1/max', '/temp/sens1?max'], description="", layout=widgets.Layout(width="90%"))
q2_b = widgets.Button(description="Verifier")
q2_o = widgets.Output()
def q2_check(_):
    q2_o.clear_output()
    with q2_o:
        display(_feedback(q2_w.value == '/temp/sens1?max', ""))
q2_b.on_click(q2_check)
display(q2_w, q2_b, q2_o)



#### Content-format

### Représentation des données

Pour que le client puisse interpréter les données, il faut qu’il puisse comprendre comment elles sont représentées. Cela peut dépendre de la police de caractères. Ainsi, une lettre accentuée ne sera pas représentée de la même manière suivant le type de code. Il en va de même pour la représentation. Le plus simple consiste à envoyer en ASCII la valeur demandée, par exemple la chaîne de caractères 18 indique 18 degrés. Il faut donc indiquer le type de codage/sérialisation utilisé pour décrire le contenu de la ressource. Là où HTTP utiliserait un nom, c'est-à-dire une chaîne de caractères, CoAP va utiliser une valeur numérique.

| HTTP | CoAP | Signification |
| --- | --- | --- |
| text/plain ; charset=utf-8 | 0 | Valeur par défaut |
| application/link-format | 40 | Description des caractéristiques d'un objet |
| application/json | 50 | Ressource au format JSON |
| application/cbor | 60 | Ressource au format CBOR |
| application/senml+json | 110 | Mesures de capteur au format SenML codé en JSON |
| application/senml+cbor | 112 | Mesures de capteur au format SenML codé en CBOR |

Le tableau précédent donne un extrait des valeurs utilisées pour représenter les formats. La liste complète peut être trouvée sur le site de l'IANA <https://www.iana.org/assignments/core-parameters/core-parameters.xhtml#content-formats>.

Deux options CoAP utilisent ces codes :

* *Content-format* (12) indiquant comment la ressource est codée ;
* *Accept* (17) indiquant dans une requête le format dans lequel la réponse doit être codée.

Il existe beaucoup de valeurs pour le *content-format*, elles permettent de spécifier de manière très économique le type de ressource et par conséquent le traitement à effectuer.

Chaque protocole utilisant CoAP aura tendance à définir de nouveaux codes. Le tableau précédent illustre ce phénomène pour SenML. La valeur va servir à la fois pour indiquer la structure de données et le format de codage.

**Quiz : Quizz**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Quelle valeur Content-type utiliser pour une réponse en CoAP ?**"))
q0_w = widgets.RadioButtons(options=['text/plain', '0', '50'], description="", layout=widgets.Layout(width="90%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value == '0', ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)

display(Markdown("**Vous voulez recevoir une ressource dans le format SenML; CBOR. Quelle valeur doit transporter l'option Accept dans la requête ?**"))
q1_w = widgets.Text(layout=widgets.Layout(width="40%"))
q1_b = widgets.Button(description="Verifier")
q1_o = widgets.Output()
def q1_check(_):
    q1_o.clear_output()
    with q1_o:
        display(_feedback(_num_ok(q1_w.value, "112"), ""))
q1_b.on_click(q1_check)
display(q1_w, q1_b, q1_o)

display(Markdown("**Quel code d'erreur retourne le serveur s'il ne peut pas envoyer une réponse dans ce format ?**"))
q2_w = widgets.RadioButtons(options=['4.04 (Not Found)', '4.02 (Bad Option)', '4.06 (Not Acceptable)', '5.01 (Not Implemented)'], description="", layout=widgets.Layout(width="90%"))
q2_b = widgets.Button(description="Verifier")
q2_o = widgets.Output()
def q2_check(_):
    q2_o.clear_output()
    with q2_o:
        display(_feedback(q2_w.value == '4.06 (Not Acceptable)', "4.06 n'est pas facile à trouver. Il faut aller sur le site de l'IANA, aller sur le RFC de CoAP qui pointe sur celui de HTTP pour la définition de ce code qui est rarement utilisé en HTTP. 4.04 n'est pas possible car la ressource existe mais pas au bon format. 4.02 n'est également pas possible, l'option Accept est critique donc doit être connue du serveur. Finalement, il s'agit d'une erreur du client et pas du serveur ; donc l'erreur 5.01 n'est pas possible non plus."))
q2_b.on_click(q2_check)
display(q2_w, q2_b, q2_o)



### Un peu plus de CoAP

#### CoAP++

# Un peu plus de CoAP

Il existe d'autres options dans CoAP comme :

* block qui permet de faire une fragmentation d'une ressource de grande taille et en faire des petits blocs transmissibles sur le réseau,
* des options pour gérer les données dans les proxy ou les caches que nous ne détaillerons pas mais qui sont très similaires à ce que l'on fait avec HTTP.

Nous allons finir cette section avec l'option Observe car elle est peut être très utile pour suivre l'évolution de la valeur d'une ressource donné.

### Observe

Avec l’architecture REST, le serveur répond toujours aux requêtes d’un client. Si l'on veut suivre l'évolution d'une ressource, le client doit demander périodiquement la valeur ; le serveur ne gardant pas d'état sur les requêtes passées. Cela n'est pas toujours compatible avec les contraintes énergétiques des capteurs. Supposons que l'on ait une alarme d'incendie qui doit informer quand le taux de fumée atteint un certain seuil. Il existe deux possibilités :

* l'alarme est un client REST et envoie un POST vers un serveur quand l'alerte est déclenchée. Pour que cela fonctionne, il faut que l'alarme ait été préalablement configurée avec l'adresse du serveur vers où envoyer ses requêtes POST ;
* l'alarme est un serveur REST qui possède une ressource donnant le taux de fumée. Elle n'a pas besoin d'être configurée. Les clients l'interrogent et elle répond à leur adresse. En revanche, si l'on veut déterminer quand un seuil est atteint, il faut continuellement interroger la ressource à une fréquence élevée si l'on ne veut pas rater une information.

L'option Observe permet à un serveur d'envoyer périodiquement la valeur d'une ressource vers le client qui en a fait la demande. La période d'émission (ou les règles d'émission comme envoyer quand un seuil est atteint) est définie par le comportement du serveur.

![](/static/asset-v1_IMTx_101_2018_type_asset_block_Capture_d_e_cran_2018-09-22_a__20.16.28.png)

La figure précédente illustre ce fonctionnement. Le client envoie une requête GET avec le type Observe vide. Si le serveur l’accepte, il retourne la ressource en indiquant une valeur dans le type Observe, et il répète le *token* donné par le client. La valeur donnée dans le champ Observe doit croître pour permettre au client de reséquencer les réponses qui peuvent être plus ou moins retardées sur le réseau. Dans l’exemple, la troisième réponse avec la valeur d'Observe de 30 est retardée et arrive chez le client après la quatrième. Le client la rejette et ne traite pas les données qu'elle transporte.

Bien entendu, l'instauration d'un état dans le serveur peut poser des problèmes ; non pas pour émettre les données vers les clients, mais plutôt pour arrêter d'émettre :

* le client peut indiquer au serveur d'arrêter d'émettre en renvoyant une requête au serveur sans l'option Observe ;
* le client peut aussi avoir perdu le contexte par suite d'un redémarrage. Dans ce cas, le client retourne un message RST ;
* le serveur peut aussi périodiquement tester la disponibilité du client en envoyant un message CON à la place d'un NON. S'il ne reçoit pas d'acquittements, le client n'est plus présent sur le réseau.

### LwM2M

#### Introduction

Il est temps de voir une vrai plateforme IoT qui met en œuvre ce que nous venons d'apprendre sur CoAP et REST. L'environnement de travail est configuré pour que vous puissiez démarrer directement cet atelier. Mais vous pouvez également l'installer sur votre machine en téléchargeant les deux exécutables java:

```
wget https://ci.eclipse.org/leshan/job/leshan/lastSuccessfulBuild/artifact/leshan-client-demo.jar
et
wget https://ci.eclipse.org/leshan/job/leshan/lastSuccessfulBuild/artifact/leshan-server-demo.jar
```

*Lightweight Machine to Machine* ou LwM2M est une architecture définie par l'OMA (Open Mobile Alliance) à l'origine pour permettre aux opérateurs de gérer certaines ressources sur les téléphones mobiles. Mais cette architecture peut être étendu à d'autres environnement.

Les spécifications sont accessibles sur le site de l'OMA (<https://omaspecworks.org/what-is-oma-specworks/iot/lightweight-m2m-lwm2m/>). Nous allons utiliser une mise en œuvre en java appelé *leshan.* Pas de panique nous n'allons pas programmer nous aurons juste besoin d'un navigateur web et de *wireshark.*

LwM2M comme son nom l'indique se veut léger, c'est-à-dire que les mises en œuvre ne doivent pas être trop complexes et que le trafic engendrer ne doit pas non plus être trop important. LwM2M est une plate-forme et elle va donc faire plus qu'un simple trafic REST. En particulier, l'un des rôles est de permettre aux objets de s'enregistrer et de décrire leurs caractéristiques. LmM2M va également structurer fortement les ressources en imposant des règles de nommages relativement contraintes et le format des ressources.

### Architecture

Comme tout système qui se respecte, LwM2M fonctionne en mode client/serveur. Le serveur est la plate-forme de gestion des objets et les clients sont les objets connectés sur le réseau.

Dans un premier temps, lancez *wireshark* sur l'interface loopback et filtrez le trafic en le limitant au port 5683 pour UDP (celui de CoAP) en tapant udp.port==5683. Comme le montre la figure suivante :

![](/static/Capture_d_e_cran_2020-10-22_a__10.29.35.png)

Nous allons maintenant lancer le serveur LwM2M, tapez:

|  |  |
| --- | --- |
| ``` ``` | ``` > java -jar ./leshan-server-demo.jar 2020-10-22 01:34:09,365 INFO LeshanServer - LWM2M server started at coap://0.0.0.0/0.0.0.0:5683 coaps://0.0.0.0/0.0.0.0:5684 2020-10-22 01:34:09,553 INFO LeshanServerDemo - Web server started at http://0.0.0.0:8080/. ``` |

Il nous indique qu'il utilise le port 5683 pour CoAP et que l'on peut superviser le serveur avec un navigateur sur le port 8080.

Lancer le navigateur (**specifier l'adresse**), la page suivante apparaît :

![](/static/Capture_d_e_cran_2020-10-22_a__10.46.45.png)

Vous pouvez remarquer que l'ouverture du serveur LwM2M n'a provoqué aucun trafic CoAP sur l'analyseur réseau.

Lancez maintenant dans une autre fenêtre, le client LwM2M:

|  |  |
| --- | --- |
| ``` ``` | ``` > java -jar ./leshan-client-demo.jar 2020-10-22 01:49:51,063 INFO LeshanClientDemo - Commands available : - create <objectId> : to enable a new object. - delete <objectId> : to disable a new object. - update : to trigger a registration update. - w : to move to North. - a : to move to East. - s : to move to South. - d : to move to West. 2020-10-22 01:49:51,064 INFO LeshanClient - Starting Leshan client ... 2020-10-22 01:49:51,158 INFO CaliforniumEndpointsManager - New endpoint created for server coap://localhost:5683 at coap://0.0.0.0:48274 2020-10-22 01:49:51,159 INFO LeshanClient - Leshan client[endpoint:ubuntu] started. 2020-10-22 01:49:51,160 INFO DefaultRegistrationEngine - Trying to register to coap://localhost:5683 ... 2020-10-22 01:49:51,252 INFO DefaultRegistrationEngine - Registered with location '/rd/BOr5Pg7yW8'. 2020-10-22 01:49:51,252 INFO DefaultRegistrationEngine - Next registration update to coap://localhost:5683 in 53s... ``` |

Comme nous utilisons une adresse loopback, il est un peu plus difficile de repérer qui est le client et qui est le serveur. Le serveur attend sur le port 5683.

**Quiz : Quizz**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Quelle est la nature de la requête émise par le client:**"))
q0_w = widgets.RadioButtons(options=['GET', 'POST', 'PUT', 'DELETE'], description="", layout=widgets.Layout(width="90%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value == 'POST', ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)

display(Markdown("**Quelle est le chemin de l'URI**"))
q1_w = widgets.RadioButtons(options=['vide', '/rd', '/lwm2m'], description="", layout=widgets.Layout(width="90%"))
q1_b = widgets.Button(description="Verifier")
q1_o = widgets.Output()
def q1_check(_):
    q1_o.clear_output()
    with q1_o:
        display(_feedback(q1_w.value == '/rd', ""))
q1_b.on_click(q1_check)
display(q1_w, q1_b, q1_o)

display(Markdown("**Quelle est le format du contenu (content-format):**"))
q2_w = widgets.RadioButtons(options=['text', 'XML', 'JSON', 'link-format', 'CBOR'], description="", layout=widgets.Layout(width="90%"))
q2_b = widgets.Button(description="Verifier")
q2_o = widgets.Output()
def q2_check(_):
    q2_o.clear_output()
    with q2_o:
        display(_feedback(q2_w.value == 'link-format', ""))
q2_b.on_click(q2_check)
display(q2_w, q2_b, q2_o)

display(Markdown("**A quelle periode voyez vous d'afficher des messages CoAP ?**"))
q3_w = widgets.Text(layout=widgets.Layout(width="40%"))
q3_b = widgets.Button(description="Verifier")
q3_o = widgets.Output()
def q3_check(_):
    q3_o.clear_output()
    with q3_o:
        display(_feedback(_num_ok(q3_w.value, "53"), ""))
q3_b.on_click(q3_check)
display(q3_w, q3_b, q3_o)



#### URI CoAP

Nous allons décoder l'Uri-Query de la requête CoAP à l'aide du document [LwM2M Core Specification](http://www.openmobilealliance.org/release/LightweightM2M/V1_1_1-20190617-A/OMA-TS-LightweightM2M_Core-V1_1_1-20190617-A.pdf) et de sa fichier 6.2.1 page 40.

**Quiz : Quizz**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Que signifie b=U**"))
q0_w = widgets.RadioButtons(options=['Les données seront émises avec les unités (Units)', 'Le protocole sous jascent est en mode datagramme (UDP)', 'Le référenciel des unités est la représentation Universelle', "Le client n'est pas référencé (Unreferenced)"], description="", layout=widgets.Layout(width="90%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value == None, ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)

display(Markdown("**Que signifie lwm2m=1.1. Il s'agit:**"))
q1_w = widgets.RadioButtons(options=['de la version lwm2m du client', "de la taille mémoire de l'implémentation (1.1 Ko)", 'de la version lwm2m du serveur'], description="", layout=widgets.Layout(width="90%"))
q1_b = widgets.Button(description="Verifier")
q1_o = widgets.Output()
def q1_check(_):
    q1_o.clear_output()
    with q1_o:
        display(_feedback(q1_w.value == 'de la version lwm2m du client', ""))
q1_b.on_click(q1_check)
display(q1_w, q1_b, q1_o)

display(Markdown("**Que signifie lt=300:**"))
q2_w = widgets.RadioButtons(options=["le nombre maximal d'objets (less than 300)", "la taille maximale d'un échange (300 octets)", "la durée de vie de l'enregistrement d'un objet (lifetime)"], description="", layout=widgets.Layout(width="90%"))
q2_b = widgets.Button(description="Verifier")
q2_o = widgets.Output()
def q2_check(_):
    q2_o.clear_output()
    with q2_o:
        display(_feedback(q2_w.value == "la durée de vie de l'enregistrement d'un objet (lifetime)", ""))
q2_b.on_click(q2_check)
display(q2_w, q2_b, q2_o)

display(Markdown("**Que signifie ep=ubuntu:**"))
q3_w = widgets.RadioButtons(options=["le système d'exploitation de l'objet", "le nom de l'équipement", "l'emplacement de l'objet"], description="", layout=widgets.Layout(width="90%"))
q3_b = widgets.Button(description="Verifier")
q3_o = widgets.Output()
def q3_check(_):
    q3_o.clear_output()
    with q3_o:
        display(_feedback(q3_w.value == "le nom de l'équipement", ""))
q3_b.on_click(q3_check)
display(q3_w, q3_b, q3_o)



#### CoAP

Nous allons maintenant regarder les spécifications concernant le [transport](http://www.openmobilealliance.org/release/LightweightM2M/V1_1_1-20190617-A/OMA-TS-LightweightM2M_Transport-V1_1_1-20190617-A.pdf), c'est-à-dire comment utiliser CoAP sur différents médias comme traditionnellement sur UDP, mais il est également possible d'envoyer CoAP via des SMS. Allez page 37 au chapitre 6.4.3.

**Quiz : Quizz**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Que répond le serveur à une requête à l'URI /rd ?**"))
q0_w = widgets.RadioButtons(options=['rien', '4.04', "un nouveau URI identifiant la ressource grâce à l'option Location-Path", "une ressource contenant les informations concernant la vision du server de l'objet", "les informations nécessaire à l'échange de clés pour le chiffrement."], description="", layout=widgets.Layout(width="90%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value == "un nouveau URI identifiant la ressource grâce à l'option Location-Path", ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)



#### Link format

Revenons au contenu du message de la requête POST émise initialement par le client. Que veut dire link-format, nous n'avons pas encore vu ce format. Heureusement l'IANA est notre ami et en allant chercher à quoi correspond cette valeur, sur cette page <https://www.iana.org/assignments/core-parameters/core-parameters.xhtml#content-formats> on trouve que le [rfc 6690](https://tools.ietf.org/html/rfc6690) définit ce contenu.

Il utilise un format particulier qui est utilisé initialement par le web pour définir des relations entre pages. Ce qui est important de remarquer, et après la lecture devient plus clair, c'est que les URI sont notées entre <>. Ensuite on trouve des attributs liés à cet URI séparés par des points-virgules. Deux URI sont séparés par des virgules.

En suivant ces règles de notation, les données émises par le client peuvent être formatés de la manière suivante :

```
</>;rt="oma.lwm2m",
</1>;ver=1.1,
</1/0>,
</3>;ver=1.1,
</3/0>,
</6/0>,
</3303/0>
```

Le client utilise ce format pour décrire les 7 ressources qu'il possède et qui sont identifiées par ces 7 chemins d'URI. Le nommage des ressources peut paraître étrange, nous verrons par la suite à quoi il correspond.

**Quiz : Quizz**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**A l'aide du rfc 6690, à quoi correspond l'attribut \"rt\".**"))
q0_w = widgets.RadioButtons(options=['resource type', 'retwit', 'retransmission timer', 'reassembly timer'], description="", layout=widgets.Layout(width="90%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value == 'resource type', ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)

display(Markdown("**A quoi sert l'attribut rt ?**"))
q1_w = widgets.RadioButtons(options=['à donner la sémantique (ie. comment interpréter) de la ressource', 'aider à déboguer les échanges', 'à indiquer la taille de la ressource en faisant référence à lwm2m', 'à indiquer la version de lwm2m utilisée'], description="", layout=widgets.Layout(width="90%"))
q1_b = widgets.Button(description="Verifier")
q1_o = widgets.Output()
def q1_check(_):
    q1_o.clear_output()
    with q1_o:
        display(_feedback(q1_w.value == 'à donner la sémantique (ie. comment interpréter) de la ressource', ""))
q1_b.on_click(q1_check)
display(q1_w, q1_b, q1_o)



#### Ressources

LwM2M représente les ressources d'une manière assez originales pour être à la fois compact, universel (ce qui est quelque fois oxymorique) et flexible. Pour ce faire tout est désigné par des chiffres.  Le chapitre 7 page 63 du document principal (<http://www.openmobilealliance.org/release/LightweightM2M/V1_1_1-20190617-A/OMA-TS-LightweightM2M_Core-V1_1_1-20190617-A.pdf>) contient le schéma suivant illustrant cette hiérarchie :

![](/static/Capture_d_e_cran_2020-10-22_a__14.45.47.png)

Un objet physique (ie notre client) contient une liste d'objets numériques identifié par un numéro. Ce numéro est attribué par l'OMA. Par exemple, un capteur de température aura la valeur 3303. Une liste des valeurs déjà attribuées peut être trouvée à cette adresse : <http://www.openmobilealliance.org/wp/OMNA/LwM2M/LwM2MRegistry.html#resources>.

Ainsi pour l'objet Température, on trouve la définition suivante :

![](/static/Capture_d_e_cran_2020-10-22_a__14.50.53.png)

Un client peut évidemment posséder plusieurs capteurs de température. Le deuxième chiffre du chemin de l'URI indique cette instance.

Finalement, l'objet peut être complexe et contenir plusieurs informations. En cliquant sur le chiffre 3303 dans la page web précédente, on obtient la description de l'objet. On peut voir que :

* 5700 représente la valeur mesurée par le capteur
* 5601 la valeur minimale
* 5602 la valeur maximale
* 5701 indique les unités
* 5605 permet de remettre à zéro le calcul des minima et maxima

Ainsi 3303/3/5601 représentera la valeur minimale (5601) du 4ieme capteur (3 on commence à 0) de la température (3303).

**Quiz : Quizz**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Que représente l'URI 3301/0/5602**"))
q0_w = widgets.RadioButtons(options=['la température maximale du premier capteur', "l'humiditité minimale du capteur 0", 'la luminosité maximale du capteur 0'], description="", layout=widgets.Layout(width="90%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value == 'la luminosité maximale du capteur 0', ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)

display(Markdown("**Que représente l'URI 10340/0/2**"))
q1_w = widgets.RadioButtons(options=["la temperature de fahrenheit de l'objet 10340", 'la longitude dans des coordonnées GPS', "le status actif ou non d'une caméra."], description="", layout=widgets.Layout(width="90%"))
q1_b = widgets.Button(description="Verifier")
q1_o = widgets.Output()
def q1_check(_):
    q1_o.clear_output()
    with q1_o:
        display(_feedback(q1_w.value == None, ""))
q1_b.on_click(q1_check)
display(q1_w, q1_b, q1_o)

display(Markdown("**/1**"))
q2_w = widgets.RadioButtons(options=['Device', 'LwM2M Server', 'Location', 'Temperature'], description="", layout=widgets.Layout(width="90%"))
q2_b = widgets.Button(description="Verifier")
q2_o = widgets.Output()
def q2_check(_):
    q2_o.clear_output()
    with q2_o:
        display(_feedback(q2_w.value == 'LwM2M Server', ""))
q2_b.on_click(q2_check)
display(q2_w, q2_b, q2_o)

display(Markdown("**/3**"))
q3_w = widgets.RadioButtons(options=['Device', 'LwM2M Server', 'Location', 'Temperature'], description="", layout=widgets.Layout(width="90%"))
q3_b = widgets.Button(description="Verifier")
q3_o = widgets.Output()
def q3_check(_):
    q3_o.clear_output()
    with q3_o:
        display(_feedback(q3_w.value == 'Device', ""))
q3_b.on_click(q3_check)
display(q3_w, q3_b, q3_o)

display(Markdown("**/6**"))
q4_w = widgets.RadioButtons(options=['Device', 'LwM2M Server', 'Location', 'Temperature'], description="", layout=widgets.Layout(width="90%"))
q4_b = widgets.Button(description="Verifier")
q4_o = widgets.Output()
def q4_check(_):
    q4_o.clear_output()
    with q4_o:
        display(_feedback(q4_w.value == 'Location', ""))
q4_b.on_click(q4_check)
display(q4_w, q4_b, q4_o)

display(Markdown("**/3303**"))
q5_w = widgets.RadioButtons(options=['Device', 'LwM2M Server', 'Location', 'Temperature'], description="", layout=widgets.Layout(width="90%"))
q5_b = widgets.Button(description="Verifier")
q5_o = widgets.Output()
def q5_check(_):
    q5_o.clear_output()
    with q5_o:
        display(_feedback(q5_w.value == 'Temperature', ""))
q5_b.on_click(q5_check)
display(q5_w, q5_b, q5_o)

display(Markdown("**Combien d'instances maximum pour un objet ?**"))
q6_w = widgets.Text(layout=widgets.Layout(width="40%"))
q6_b = widgets.Button(description="Verifier")
q6_o = widgets.Output()
def q6_check(_):
    q6_o.clear_output()
    with q6_o:
        display(_feedback(_num_ok(q6_w.value, "1"), ""))
q6_b.on_click(q6_check)
display(q6_w, q6_b, q6_o)



**Quiz : Quizz**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Which URI gives access to the Dimmer element of a light (Light Control)**"))
q0_w = widgets.Text(layout=widgets.Layout(width="40%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(_num_ok(q0_w.value, "3311/0/5851"), ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)

display(Markdown("**What is the maximum value of this resource?**"))
q1_w = widgets.Text(layout=widgets.Layout(width="40%"))
q1_b = widgets.Button(description="Verifier")
q1_o = widgets.Output()
def q1_check(_):
    q1_o.clear_output()
    with q1_o:
        display(_feedback(_num_ok(q1_w.value, "100"), ""))
q1_b.on_click(q1_check)
display(q1_w, q1_b, q1_o)



#### Object ID

L'OMA attribue des numéros pour les objets. Ces numéros sont uniques et réfèrent toujours au même objet. De cette manière, il n'y a pas d'ambiguïté possible, quand un serveur reçoit une valeur contenue dans le premier élément du chemin de l'URI, il connait à la fois la nature de l'objet et son format, c'est-à-dire les ressources que le composent, leur signification et leur codage.

Si l'on reprend l'objet Température (3303), nous avons vu qu'il est composé de différents ressources également  représenter par un entier identifiant la ressource (ressource ID), par exemple:

* 5700 pour la valeur courante,
* 5601 pour la valeur minimale, ...

Ces dernières valeurs peuvent être communes à différent objets (comme l'objet Humidité, ....). De même un annuaire permet de connaître leur signification et leur représentation. Si l'on va sur la page <https://devtoolkit.openmobilealliance.org/OEditor/Default>, on peut avoir la liste des *resourceID* attribués:

* |  |  |  |  |  |  |  |  |
  | --- | --- | --- | --- | --- | --- | --- | --- |
  | 5700 | Sensor Value | R | Float |  |  | IPSO | Last or Current Measured Value from the Sensor. |

* |  |  |  |  |  |  |  |  |
  | --- | --- | --- | --- | --- | --- | --- | --- |
  | 5601 | Min Measured Value | R | Float |  |  | IPSO | The minimum value measured by the sensor since power ON or reset. |
  |
* |  |  |  |  |  |  |  |  |
  | --- | --- | --- | --- | --- | --- | --- | --- |
  | 5701 | Sensor Units | R | String |  |  | IPSO | Measurement Units Definition |

Les valeurs mesurées par les capteurs seront représentées par des nombres flottant, et les unités par une chaîne de caractères.

On voit aussi que les données sont de type R (Read), le serveur peut donc les lire mais ne peut pas les changer.

#### Configuration

Nous allons nous focaliser sur le premier échange qui nous avons commencé à analyser en regardant le message émis par le client vers le serveur LwM2M. Comme nous avons vu, ce premier message contient la description des ressources présentes sur le client. Les valeurs des identifiants d'objets (Object ID) et de ressources (Resource ID) permettent au serveur de savoir ce que peut faire le client, vu qu'elles sont normalisées. Le client LwM2M envoie ces informations sur un chemin d'URI bien connu /rd (pour r*esource description*).

**Quiz : Quizz**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Que répond le serveur ?**"))
q0_w = widgets.RadioButtons(options=['il acquitte juste le message', 'rien', "il renvoie une URI qui servira par la suite à identifier l'objet.", 'il retourne les données de chiffrement des messages'], description="", layout=widgets.Layout(width="90%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value == "il renvoie une URI qui servira par la suite à identifier l'objet.", ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)



**Quiz : Quizz**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Si on laisse la plate-forme fonctionner sans intervenir que voit-on sur l'analyseur réseau.**"))
q0_w = widgets.RadioButtons(options=['rien', 'le capteur envoie des informations indiquant un changement de valeur mesurée', "le client envoie les valuers mesurées même s'il elle n'ont pas changées", "le client envoie un message vide vers le serveur pour indiquer qu'il est toujours présent", "le serveur envoie un message vide vers ses clients pour savoir s'ils sont toujours présents"], description="", layout=widgets.Layout(width="90%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value == "le client envoie un message vide vers le serveur pour indiquer qu'il est toujours présent", ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)



#### Plate-forme

Il est temps de revenir à l'interface de la plate-forme en ouvrant la page http://127.0.0.1:8080 depuis le navigateur. Vous devez avoir quelque chose de ce genre :

![](images/im34.png)

On voit qu'un objet s'y est enregistré et l'on retrouve des informations que l'on avait en disséquant l'échange. Le nom ubun était présent dans l'URI Query ep=ubun (ep est pour *end point*). Il est suivi d'un chiffre en hexadécimal (*Registration ID*) qui correspond au chemin d'URI retourné par le serveur en réponse à la demande d'enregistrement. En mettant la souris sur le i en fin de ligne, on retrouve tous les paramètres envoyés par le client.

Cliquez sur le nom de l'objet qui apparait en couleur et en gras. Une nouvelle page apparaît :

![](images/im35.png)

Le contenu ne devrait maintenant pas vous étonner. On retrouve les objets ID que l'on avait vu en analysant le message d'enregistrement et en allant sur la description des objets. On retrouve dans la deuxième colonne les URI correspondantes. En jetant un œil sur le trafic capturé par l'analyseur, on voit que l'affichage de cette page n'a produit aucun échange sur le réseau. Le bouton coap message en bas à droite de la fenêtre du navigateur nous permet d'avoir les mêmes informations que depuis l'analyseur.

#### Lecture

Cliquez sur le bouton Read de la ligne Short Server ID (qui correspond à l'URI /1/0/0). Sur l'analyseur vous pouvez voir un échange.

**Quiz : Quizz**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Quel type de requête envoie le serveur ?**"))
q0_w = widgets.RadioButtons(options=['GET', 'POST', 'PUT', 'DELETE'], description="", layout=widgets.Layout(width="90%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value == 'GET', ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)

display(Markdown("**Quel Content-format est demandé par le serveur ?**"))
q1_w = widgets.RadioButtons(options=['Texte', 'JSON', 'CBOR', 'SenML+CBOR', 'application/vnd.oma.lwm2m+tlv', 'application/vnd.oma.lwm2m+json'], description="", layout=widgets.Layout(width="90%"))
q1_b = widgets.Button(description="Verifier")
q1_o = widgets.Output()
def q1_check(_):
    q1_o.clear_output()
    with q1_o:
        display(_feedback(q1_w.value == 'application/vnd.oma.lwm2m+tlv', ""))
q1_b.on_click(q1_check)
display(q1_w, q1_b, q1_o)



#### Décodage

Si on regarde le contenu de la réponse du client, la représentation de la donnée est différente de ce que l'on a vu jusqu'à présent. En fait, LwM2M à défini son propre format appelé TLV (Type Longueur Valeur) dont on retrouve la définition dans le document principal (<http://www.openmobilealliance.org/release/LightweightM2M/V1_1_1-20190617-A/OMA-TS-LightweightM2M_Core-V1_1_1-20190617-A.pdf>) au chapitre 7.4.4 page 72.

Ainsi pour un Short Server ID valant 123, la réponse du client est C1 00 7B que l'on retrouve dans les traces de *wireshark*, le standard nous aide à décoder cette valeur. Pour information :

* le premier octet C1 ou 0b11_0_00_001 : 11 indique une ressource avec une valeur, 0 que la ressource ID est codé sur 1 octet (1 indique 2 octets), 00 que la longueur est codée sur les 3 bits suivants (sinon des octets supplémentaires sont nécessaire pour coder la longueur), 001 que la valeur est sur un octet.
* le second octet 00 donne l'instance (pour rappel, le chiffre du milieu dans l'URI) qui vaut ici 0.
* le troisième octet donne la valeur 0x7B = 123.

On peut aussi sélectionner la représentation JSON en sélectionnant JSON dans le menu Single-value en haut de l'écran. En analysant le trafic vous verrez que le champ content-format a changé et que la réponse se fait avec le format JSON :

```
{"bn": "/1/0/0",
"e":[{"v":123}]}
```

Ce format ne nous est pas complètement inconnu, il s'agit de SenML([RFC 8428](https://tools.ietf.org/html/rfc8428)), où du moins une ancienne version que l'on retrouve :

* bn indique la *base name*, c'est-à-dire la partie de l'URI commune à tout le reste, ici /1/0/0 que l'on a plus besoin de commenter.
* e est un reste de l'ancienne version de SenML qui indique un tableau de mesure. Dans les nouvelles versions, la structure est directement un tableau. Ici, la valeur (clé 'v') vaut 123.

Comme on peut s'y attendre JSON est beaucoup plus bavard que le format spécifique TLV.

Il est également possible de récupérer l'ensemble d'un objet. Cliquez sur *read* pour l'instance 0; De même en jouant sur le paramètre *multi-value* en haut de l'écran vous pouvez avoir la représentation en TLV ou en JSON.

```
{"bn":"/1/0/",
"e":[{"n":"0","v":123},
{"n":"1","v":300},
{"n":"6","bv":false},
{"n":"7","sv":"U"}
]
}
```

Comme on peut le voir la réponse indique un *base name* s'arrêtant à l'instance et les différents éléments dans l'objet.

#### Observe

### Observe

Nous allons maintenant utiliser Observe pour recevoir régulièrement la valeur d'un Objet. Allez dans le menu Temperature et cliquez sur Observe de Sensor Value.

###

**Quiz : Quizz**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Quelle est la valeur de l'option Observe dans la requête?**"))
q0_w = widgets.RadioButtons(options=['aucune', '0', '15.2', '54'], description="", layout=widgets.Layout(width="90%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value == 'aucune', ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)

display(Markdown("**Comment évolue la valeur de l'option Observe dans les réponses ?**"))
q1_w = widgets.RadioButtons(options=["il n'y en a pas", "Elle s'incrémete de 2 à chaque envoi", "Elle s'incrémente du nombre de secondes entre chaque envoi", 'Elle contient la température.'], description="", layout=widgets.Layout(width="90%"))
q1_b = widgets.Button(description="Verifier")
q1_o = widgets.Output()
def q1_check(_):
    q1_o.clear_output()
    with q1_o:
        display(_feedback(q1_w.value == "Elle s'incrémente du nombre de secondes entre chaque envoi", ""))
q1_b.on_click(q1_check)
display(q1_w, q1_b, q1_o)

display(Markdown("**Pourquoi voit-on des messages CON de temps en temps dans les réponses ?**"))
q2_w = widgets.RadioButtons(options=['pour être certain que le serveur est toujours sur le réseau', "ces messages sont plus importants que les autres du fait d'une variation plus importante", 'les messages NON sout juste des estimations de la température, les autres sont des mesures'], description="", layout=widgets.Layout(width="90%"))
q2_b = widgets.Button(description="Verifier")
q2_o = widgets.Output()
def q2_check(_):
    q2_o.clear_output()
    with q2_o:
        display(_feedback(q2_w.value == 'pour être certain que le serveur est toujours sur le réseau', ""))
q2_b.on_click(q2_check)
display(q2_w, q2_b, q2_o)



#### Ouf! C'est fini!

Voilà nous arrivons au bout de cette exploration de LwM2M, elle nous a permis de voir plus en détail CoAP, en particulier avec Observe, mais elle permet également d'introduire la prochaine semaine, où nous allons voir comment concevoir des modèles d'information pour récolter les informations des capteurs.

LwM2M, comme son nom l'indique veut rester simple et facile à mettre en œuvre, il définit un modèle de données centralisé qui permet aux serveurs et aux clients de connaitre la représentation des informations, leurs unités. Un serveur LwM2M aura connaissance de tous les objets qui y sont connectés. Pour connaître ce serveur et définir les clés de chiffrement des messages, LwM2M définit également un serveur de démarrage (bootstrap) qui permet de configurer ces paramètres dans les clients.

### Message ID et Token

**Message ID**

Le champ Message ID sur 2 octets sert à identifier les messages CoAP afin de détecter les duplicatas. Cette valeur est recopiée dans les acquittements pour permettre de savoir quel message est acquitté. Ils ne doivent pas être réutilisé pendant une période fixée.

Un peu d’algèbre élémentaire permet de calculer cette durée. La figure suivante montre le calcul du pire cas. On prend le délai de propagation maximum dans les deux directions et on suppose que toutes les requêtes se perdent sauf la dernière qui est complète et on ajoute un temps de traitement important dans le récepteur.

![](images/im30.png)

Le standard prévoit que, par défaut, la durée d’activité d’un *Message ID* est d’environ 5 minutes (247 s) pour les messages confirmés et 2,5 minutes (145 s) pour les messages non confirmés.

### Tokens

CoAP utilise le protocole UDP pour communiquer, contrairement à TCP, il n’y a donc pas de notion d’établissement de connexion. Il est donc difficile de faire le lien entre les établissements set les réponses, surtout si elles ne sont pas immédiates. La figure suivante illustre ce phénomène. Une requête GET est envoyée par un client à un serveur.

![](images/im31.png)

La réponse ne peut pas être immédiate (par exemple il faut lire une valeur sur un capteur qui demande d’être activé). Le message d'acquittement ne peut pas être différé sinon, le client ne voyant pas sa requête acquittée, la retransmettrait. Le serveur acquitte avec un message Ack vide. Quand le serveur peut envoyer la ressource, il le fait à son tour dans un message de type CON qui sera à son tour acquitté. Vous pouvez remarquer que les valeurs du champ Message ID sont complètement décorrélées. Pour faire le lien entre la requête et la réponse, un *token* fourni par le client est recopié par le serveur. On peut considérer une valeur de token comme une connexion entre le client et le serveur.**Pas trop trop compris le role de CoAP, et son plus par rapport a messageID, et pourquoi facultatif ?**

Le token est une séquence binaire dont la taille est comprise entre 1 et 8 octets. La longueur est indiquée au début de l’en-tête obligatoire et la valeur suit immédiatement cet en-tête obligatoire.

### Les options CoAP

# Les options CoAP

Le champ option va contenir des options qui vont soit servir à améliorer le protocole de transfert des données entre le client et le serveur, soit servir à coder les en-têtes des requêtes et des réponses, en garantissant une certaine compatibilité avec les en-têtes HTTP.

La structure utilisée est dite TLV pour Type Longueur Valeur. Chaque champ contient au moins ces deux informations de type qui va indiquer la nature de l’option et longueur indiquant la taille des données. Si ce dernier n’est pas nul, les données vont se trouver après. L’intérêt de cette notation est qu’elle permet de gérer une certaine évolution du protocole. Ainsi si un récepteur ne reconnaît pas un type, il peut très bien passer au suivant car il connaît la longueur. Cela permet également d’avoir des options de longueur variable.

![](images/im32.png)

CoAP complique un peu la chose en optant pour un codage différentiel de la valeur de l’option. Ainsi si l’on doit envoyer une option de type 5, puis deux de type 6, le codage contiendra ∆T = 5, ∆T = 1, ∆T = 0.

**Question perso, pourquoi ne pas aller jusqu'à 14 (avec 15 codage sur 2 octet de la différence)  au lieu de 13 ?**

Si la différence est supérieure ou égale à 13, la valeur 13 est mise dans le champ ∆T et un entier sur 8 bits code la différence moins 13. Par exemple, si les deux options sont de type 5 et 15, le codage sera le suivant ∆T = 5, ∆T = 13 + 2 ∆**1****0 ici par contre, dans ce cas plutot** **écrire****type 5 et 20**. De même, si la différence n’est codable que sur 2 octets, la valeur mise dans le champ est 14 et les deux octets suivants contiennent la différence entre les deux options moins 269. Dans le cas ou le premier type est 13, ou que la différenciel vaut précisément 13.... 00000000 ??

L’intérêt de cette représentation est qu’elle est dans la plupart des cas plus compacte mais aussi qu’elle impose d’ordonner les options ce qui peut simplifier leur traitement des options.

Pour la longueur on retrouve le même principe, les longueurs inférieures à 13 sont codées directement. Si elles sont supérieures ou égale à 13, la valeur moins 13 est codée dans un octet supplémentaire. Une valeur de 14 indique que deux octets sont utilisés pour coder la longueur moins 269.

La figure en haut de page illustre le codage d’une option dans l’en-tête d’un message CoAP.

Il se peut qu’il y ait des données après les options, dans ce cas un séparateur avec la valeur 0xFF est inséré. Il ne peut pas être confondu avec le codage d’une option.

S’il n’y a pas de données à transmettre (par exemple dans le cas d’une requête GET), le message CoAP se termine après les options.

### Options CoAP

Le premier bit servant a coder le type décrit, quand il est positionné à 1, si ce type doit être connu du récepteur (critique). Dans ce cas, si un destinataire reçoit une option de ce type est qu’il ne la connaît pas, il doit produire un message d’erreur. Dans la cas contraire cette option est ignorée du récepteur qui poursuit le traitement des options suivantes. Ainsi les options paires sont facultatives et les impaires critiques. Les options apparaissant en gras seront traitées plus en détail dans la suite de ce chapitre.

| Valeur | Nom | Type | Nature | répété | Commentaire |
| --- | --- | --- | --- | --- | --- |
| 0 | Réserve |  |  |  |  |
| 1 | If-Match | opaque | critique | oui | Utilisé pour indiquer a serveur de n'effectuer la requête que sous certaines conditions. |
| 3 | Uri-Host | string | critique |  | Contient le nom du serveur d'une URI (nom, adresse IPv4 ou IPv6). Généralement, il n'est pas nécessaire de le préciser puisque les messages CoAP sont envoyé à cette adresse. |
| 4 | ETag | opaque | facultative | oui | Utilisé pour gérer la mise en cache des ressources |
| 5 | If-None-Match | vide | critique |  | Utilisé pour indiquer a serveur de n'effectuer la requête que sous certaines conditions |
| **6** | **Observe** | **entier** | **facultative** |  | **Permet à un serveur d'envoyer une requête aux changement d'état d'une ressource. Dans la réponse la valeur doit toujours augmenter.** |
| 7 | Uri-Port | entier | critique |  | Contient le numéro du port UDP sur lequel CoAP est lancé. Généralement ce champ n'est pas nécessaire vu que le serveur CoAP attend déjà des messages sur ce ports augmenter. |
| 8 | Location-Path | string | facultative | oui | Utilisé en réponse à une requête POST pour indiquer un segment du chemin de la ressource. |
| **11** | **Uri-Path** | **string** | **facultative** | **oui** | **Contient un des segments de l'URI.** |
| **12** | **Content-Format** | **entier** | **facultative** |  | **Définit le format dans lequel sont codées des données.** |
| 14 | Max-Age | entier | facultative |  | Durée pendant laquelle la ressource peut être mise en cache. |
| **15** | **Uri-Query** | **string** | **critique** | **oui** | **Contient les segments d'interrogation que l'on retrouve dans les URI.** |
| **17** | **Accept** | **entier** | **critique** |  | **Indique les formats que le client peut accepter.** |
| 20 | Location-Query | string | facultative | oui | Utilisé en réponse à une requête POST pour indiquer le chemin de la ressource. |
| 35 | Proxy-Uri | string | critique |  | Contient une URI qui doit être prise en compte par le proxy. |
| 35 | Proxy-Scheme | string | critique |  | Indique le format de l'URI et le protocole dans une URL. |
| 60 | Size1 | entier | facultative |  | Indique la taille de la ressource. |
| **258** | **No-Response** | **entier** | **facultative** |  | **Limite les notifications REST.** |

### TP Mise en oeuvre de GET

# TP Mise en oeuvre de GET

**Pourquoi ce TP est-il là par rapport à la structure du cours (et non pas dans une section séparée comme le TP suivant)?**

**Ce TP fait-il partie de la formule qualifiante?**

Nous allons mettre en œuvre le protocole CoAP entre deux processus sur votre machine, puis si vous le voulez, vous pourrez le tester sur un Pycom. Côté objet nous allons utiliser une mise en œuvre simple mais compacte du protocole pour comprendre son fonctionnement. A l’autre extrémité, nous utiliserons la mise en œuvre *aiocoap* qui est très complète, mais beaucoup plus complexe et demandant plus de ressources.

### aiocoap

*aiocoap* met en œuvre coap, avec les modules *asynchronous Input/Output*, permettant un fort degrès de parallélisme. Le programme suivant (coap_basic_server1.py) donne un exemple d’un serveur simple gérant une seule ressource time :

```python
#!/usr/bin/env python3
# This file is part of the Python aiocoap library project.
#
# Copyright (c) 2012-2014 Maciej Wasilak <http://sixpinetrees.blogspot.com/>,
# 2013-2014 Christian Amsüss <c.amsuess@energyharvesting.at>
#
# aiocoap is free software, this file is published under the MIT license as
# described in the accompanying LICENSE file.
"""This is a usage example of aiocoap that demonstrates how to implement a
simple server. See the "Usage Examples" section in the aiocoap documentation
for some more information."""
import datetime
import logging
import socket
import asyncio
import aiocoap.resource as resource
import aiocoap
class TimeResource(resource.Resource):
async def render_get(self, request):
await asyncio.sleep(5)
payload = datetime.datetime.now().\
strftime("%Y-%m-%d %H:%M").encode('ascii')
return aiocoap.Message(payload=payload)
# logging setup
logging.basicConfig(level=logging.INFO)
logging.getLogger("coap-server").setLevel(logging.DEBUG)
def main():
# Resource tree creation
ip_addr = socket.gethostbyname(socket.gethostname())
port = 5683
print ("server running on ", ip_addr, "at port", port)
root = resource.Site()
root.add_resource(['time'], TimeResource())
asyncio.Task(
aiocoap.Context.create_server_context(root,
bind=(ip_addr, port)))
asyncio.get_event_loop().run_forever()
if __name__ == "__main__":
main()
```

L’utilisation de *aicoap* se traduit par l’importation des modules qui se trouvent dans le répertoire aoicoap. Dans la fonction main le programme cherche son adresse IP fixe le numéro de port à la valeur affectée à CoAP (5683).  L’utilisation d’une adresse IP et non du joker 0.0.0.0 permet de faire tourner le serveur dans un environnement Windows et MAC.

La variable root contient l’arbre des ressources. A la ligne suivante, la ressource time est associée à une classe TimeResource.

La méthode

```
asyncio.Task(aiocoap.Context.create_server_context(root, bind=(ip_addr, port)))
```

permet de lier cet arbre de ressource à l’adresse IP et au numéro de port précédemment défini. Le serveur est ensuite lancé dans une boucle sans fin.

Il est plus intéressant de voir le traitement effectué lorsque la ressource est appelée par le serveur. La classe TimeResource dérivant de la classe générique aiocoap Resource est utilisée :

```
class TimeResource(resource.Resource):
```

Pour chaque méthode CoAP une méthode peut être définie. Dans l’exemple, la méthode render_get permet de traiter les requêtes GET. Pour simuler un temps de traitement le programme commence par attendre 5 secondes puis construit la chaîne de caractères contenant la date qu’il va retourner dans un objet aoicoap Message.

Ainsi, si tout se passe correctement, la réponse à une requête (Code = 0x01) sera 2.05 (*Content*). La figure suivante résume cet échange que nous avions vu théoriquement dans le chapitre consacré aux *tokens*.

![](images/im31.png)

```
Sur votre ordinateur, lancez le programme coap_basic_server1.py dans un terminal.
```

***Les programmes pourront tourner indifféremment dans une fenêtre de votre ordinateur ou sur votre pycom. Dans la suite, nous supposerons qu'il s'agit d'une deuxième fenêtre de terminal sur votre ordinateur, mais libre à vous de le lancer sur votre Pycom en Wi-Fi (voire en LoRaWAN) pour plus de réalisme. L'utilisation du réseau Sigfox est plus problématique vu que les requêtes provenant de votre Pycom sont limitées à 12 caractères et que les réponses ne sont qu'au nombre de 4 par jour et limitée à 8 caractères.***

### Messages vides

Du coté du client, nous allons utiliser une mise en œuvre plus compacte qui nous permettra d’expérimenter le fonctionnement de CoAP en modifiant les valeurs des champs protocolaires.

Le programme coap_empty_msg.py :

```python
import CoAP
import socket
SERVER = "192.168.1.26" # change to your server's IP address
PORT = 5683
s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
coap = CoAP.Message()
coap.new_header()
coap.dump(hexa=True)
s.settimeout(10)
s.sendto (coap.to_byte(), (SERVER, 5683))
resp,addr = s.recvfrom(2000)
answer = CoAP.Message(resp)
answer.dump(hexa=True)
```

Le programme est beaucoup plus simple, dans un premier temps vous devez remplacer l’adresse IP par celle fournie par votre serveur CoAP. Le programme crée une socket UDP au travers le laquelle l’échange avec le serveur sera effectué. La première action consiste à créer un message CoAP (ligne 10) et à la ligne suivante créer un en-tête obligatoire.

Le programme affiche le message et l’envoie sur la socket. La ligne 14 permet de limiter l’attente de la réponse à 10 secondes. Cette réponse est attendue ligne 17, transformée en message CoAP ligne suivante et affichée.

Ce programme fonctionne aussi bien sur dans un terminal sur votre ordinateur que sur les Pycom avec une connexion en Wi-Fi sur votre réseau local. Vous pouvez également le tester sur les réseaux LoRaWAN en remplaçant le numéro de port 33033 utilisé par generic_relay.py par la valeur 5683. Il est beaucoup plus délicat de le faire tourner sur Sigfox car CoAP va vite remplir la limite des 12 caractères et dans un premier temps nous allons générer beaucoup de messages *downlink.*

Vous pouvez également lancer un wireshark sur votre ordinateur pour voir le trafic passant sur le port 5683 29 .

On obtient le résultat suivant :

```
b ’ 40000001 ’
CON 0 x0001 EMPTY
b ’ 70000001 ’
RST 0 x0001 EMPTY
```

Les messages suivants ont circulé sur le réseau.

```
18:38:30.381449 IP 192.168.1.26.50883 > 192.168.1.26.5683: UDP, length 4
0x0000: 4500 0020 221f 0000 4011 0000 c0a8 011a E..."...@.......
0x0010: c0a8 011a c6c3 1633 000c 83a2 4000 0001 .......3....@...
18:38:30.382107 IP 192.168.1.26.5683 > 192.168.1.26.50883: UDP, length 4
0x0000: 4500 0020 efbb 0000 4011 0000 c0a8 011a E.......@.......
0x0010: c0a8 011a 1633 c6c3 000c 83a2 7000 0001 .....3......p...
```

On retrouve dans le contenu des messages UDP, le message CoAP donné par l’application. Si l’on prend le deuxième message. Il commence par 0x70, ce qui correspond en binaire à 0b01_11_0000, soit version = 1, type = 3 et longueur du token = 0. L’octet suivant donne le code 0 *(Empty)* et les deux derniers octets contiennent le message ID. Le serveur ne sachant pas quoi faire de la requête, le rejette en envoyant un message Reset pour essayer d’arrêter le code sur le client qui envoie ce genre de requête.

GET

Nous allons modifier le programme précèdent pour qu’il récupère la ressource time sur le serveur. Le programme coap_get_time1.py est le suivant :

```python
import CoAP
import socket
SERVER = "192.168.1.77" # change to your server's IP address
PORT = 5683
s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
coap = CoAP.Message()
coap.new_header(code=CoAP.GET)
coap.add_option (CoAP.Uri_path, "time")
coap.dump(hexa=True)
s.sendto (coap.to_byte(), (SERVER, PORT))
s.settimeout(10)
resp,addr = s.recvfrom(2000)
answer = CoAP.Message(resp)
answer.dump(hexa=True)
```

Il diffère du précèdent au niveau de la création de l’en-tête où la méthode GET est spécifié et par l’utilisation de l’option uri-path pour coder le nom de la ressource.

Quand on exécute ce programme on obtient :

```
b'40010001b474696d65'
CON 0x0001 GET
> URI_path : b'time'
b'60000001'
ACK 0x0001 EMPTY
```

La réponse est toujours EMPTY, mais si l’on regarde les messages affichés par le serveur ou l’analyseur réseau, on voit un trafic apparaitre au bout de 5 secondes. C’est lié à la mise en œuvre sur le serveur, nous attendions 5 secondes avant de retourner la valeur. Le message EMPTY que le client reçoit est juste là pour acquitter la requête. On peut modifier le code du client pour attendre encore 10 secondes pour récupérer la seconde requête.

On ajoute à la fin les lignes suivantes :

```
s.settimeout(10)
resp,addr = s.recvfrom(2000)
answer = CoAP.Message(resp)
answer.dump()
```

qui forment le programme coap_get_time2.py.

On obtient:

```
'CON 0x0001 GET
> URI_path : b'time'
ACK 0x0001 EMPTY
CON 0x93BC 2.05
---CONTENT
hex: b'323032302d30392d30352031393a3533'
txt: b'2020-09-05 19:53'
```

Le champ code contient la valeur 2.05 là où HTTP aurait utilisé le code générique "200".

Si l’on obtient le résultat, on peut remarquer, en regardant ses messages de log, que le serveur continue d’envoyer la réponse. Comme la réponse est un message de type CON, il faut l’acquitter en utilisant la valeur du message ID de la réponse en ajoutant la partie suivante :

```
mid = answer.get_mid()
ack = CoAP.Message()
ack.new_header(mid=mid, type=CoAP.ACK)
ack.dump()
s.sendto (ack.to_byte(), (SERVER, PORT))
```

qui forment le programme coap_get_time3.py.

Si le client n’envoie qu’une requête, la réponse correspond évidemment à cette requête, mais pour plus de garantie, il est préférable d’utiliser un *token* qui sera indiqué lors de construction de l’en-tête.

Modifiez la ligne 10 pour ajouter le paramètre token ce qui forme le programme coap_get_time4.py.

```
coap.new_header(code=CoAP.GET, token=0x12345)
```

On voit que le token est retourné dans la réponse.

```
CON 0x0001 GET T=012345
> URI_path : b'time'
ACK 0x0001 EMPTY
CON 0x93C3 2.05 T=012345
---CONTENT
hex: b'323032302d30392d30352032303a3135'
txt: b'2020-09-05 20:15'
ACK 0x93C3 EMPTY
```

**Quiz : Blank Common Problem**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Combien de messages circulent sur le réseau pour obtenir la ressource ?**"))
q0_w = widgets.Text(layout=widgets.Layout(width="40%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(_num_ok(q0_w.value, "2"), ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)

display(Markdown("**Quel type de message transporte la ressource ?**"))
q1_w = widgets.RadioButtons(options=['CON', 'NON', 'ACK', 'RST', '2.05'], description="", layout=widgets.Layout(width="90%"))
q1_b = widgets.Button(description="Verifier")
q1_o = widgets.Output()
def q1_check(_):
    q1_o.clear_output()
    with q1_o:
        display(_feedback(q1_w.value == 'ACK', "2.05 n'est pas un type de message. La réponse étant immédiate, le serveur CoAP optimise les échanges en  retournant la réponse dans un message ACK."))
q1_b.on_click(q1_check)
display(q1_w, q1_b, q1_o)



A chaque fois que l’on a relancé le programme, on a obtenu une réponse. Si votre programme tourne sur un Pycom, cela peut sembler logique vu que le contexte étant gardé, la valeur du champ Message ID est incrémenté après chaque exécution. En revanche, dans le cas d’une exécution sur votre ordinateur, le champ Message ID reste à 0x001. En fait, le numéro de port source change à chaque exécution, *aiocoap* suppose que cela provient à chaque fois d’un client CoAP diffèrent. Nous allons donc forcer le numéro de port source grâce à la méthode bind. Ajoutez la ligne 

```
s.bind(("0.0.0.0", 33033))
```

après l'ouverture de la socket et forcer la valeur du champ Message ID au moment de la construction de l’en-tête CoAP :

```
coap.new_header(mid=0xF001, code=CoAP.GET)
```

ce qui donne le programme coap_get_time5.py.

Dans les traces du serveur, on peut voir qu’il identifie la requête comme étant déjà faite et retourne la valeur précédemment retournée sans appeler de nouveau la méthode render_get de la classe TimeResource.

Côté client vous pouvez également vous en rendre compte puisque l’heure retournée n’est pas modifiée.

```
- 17:02:31- %python3 coap\_get\_time5.py
CON 0xF001 GET
> URI_path : b'time'
ACK 0xF001 2.05
---CONTENT
hex: b'323032302d30392d30362031373a3032'
txt: b'2020-09-06 17:02'
-17:04:25- %python3 coap\_get\_time5.py
CON 0xF001 GET
> URI_path : b'time'
ACK 0xF001 2.05
---CONTENT
hex: b'323032302d30392d30362031373a3032'
txt: b'2020-09-06 17:02'
-17:08:19- %python3 coap\_get\_time5.py
CON 0xF001 GET
> URI\_path : b'time'
ACK 0xF001 2.05
---CONTENT
hex: b'323032302d30392d30362031373a3038'
txt: b'2020-09-06 17:08'
```

Les deux premières requêtes retournent 17:02 tandis que la dernière effectuée 5 minutes plus tard retourne une nouvelle valeur 17:08.

**Quiz : Blank Common Problem**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Comment expliquer ce phénomène ?**"))
q0_w = widgets.RadioButtons(options=['Le serveur fait un sleep(5) qui le rend inactif 5 minutes', "Le client a gardé dans une mémoire cache l'information reçue et ne renvoie pas de requête avant que celle-ci n'expire", "Le serveur a gardé l'information dans une mémoire cache et retourne toujours la même information avant que celle-ci n'expire.", "Le serveur a mémorisé la valeur du champ Message ID et considère les messages avec la même valueur comme des demandes de transmission car la réponse s'est perdue."], description="", layout=widgets.Layout(width="90%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value == "Le serveur a mémorisé la valeur du champ Message ID et considère les messages avec la même valueur comme des demandes de transmission car la réponse s'est perdue.", ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)



### TP Les POST

### Ressource codée en texte

Nous allons voir le traitement d’une requête POST avec *aiocoap.* Nous aurions pu ajouter ce traitement à la classe TimeResource vue précédemment, mais nous avons préféré ajouter une nouvelle ressource pour le traitement des températures (vous voyez où on veut en venir). **- euh non...**.

La réponse à la requête est le code 2.04 qui indique que la ressource été modifiée. **Quelle requète? Je ne comprends pas**

Les lignes suivantes ont été ajoutée pour traiter le POST. Notez le nom de la méthode render_post.

```python
class TemperatureResource(resource.Resource):
async def render_post(self, request):
print ("-"\*20)
print ("payload:", binascii.hexlify(request.payload))
resp = aiocoap.Message(code=aiocoap.CHANGED)
return resp
```

et lui associer une nouvelle classe dans l’arbre des ressources:

```
root.add_resource(['temp'], TemperatureResource())
```

formant le programme coap_basic_server2.py.

Coté client, le principe est le même que pour le GET. Dans le programme coap_post_temp1.py,  la méthode add_payload permet d’ajouter du contenu à la requête POST. Le programme émet sa requête et affiche le résultat. Nous allons également simplifier la gestion des réponses en utilisant la fonction send_ack du module CoAP. Cette fonction prend en argument une socket, un tuple de destination et le message CoAP à envoyer. Elle va le retransmettre au maximum 4 fois tant qu’elle n’a pas reçu l’acquittement correspondant.

```python
import CoAP
import socket
import time
SERVER = "192.168.1.26" # change to your server's IP address
PORT = 5683
destination = (SERVER, PORT)
s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
coap = CoAP.Message()
coap.new_header(code=CoAP.POST)
coap.add_option (CoAP.Uri_path, "temp")
coap.add_payload("23.5")
coap.dump()
answer = CoAP.send_ack(s, destination, coap)
answer.dump()
```

La figure suivante montre un échange avec perte de messages. Elles peuvent être simulé en arrêtant le programme serveur sur votre ordinateur.

![](images/im33.png)

###

### Ressource codée en CBOR

Ce programme est correct, mais nous avons utilisé le format ASCII de transfert par défaut (Content-format : 0), il serait préférable de revenir au format JSON ou CBOR que nous avions vu précédemment. Pour ce faire, nous devons ajouter dans l’en-tête de la requêté l’option Content-format. Comme son code est 12, elle doit être placée après l’option Uri-path (cf. tableau des options).

```python
import CoAP
import socket
import time
try:
import kpn_senml.cbor_encoder as cbor #pycom
except:
import cbor2 as cbor # terminal on computer
SERVER = "192.168.1.26" # change to your server's IP address
PORT = 5683
destination = (SERVER, PORT)
s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
coap = CoAP.Message()
coap.new_header(code=CoAP.POST)
coap.add_option (CoAP.Uri_path, "temp")
coap.add_option (CoAP.Content_format, CoAP.Content_format_CBOR)
coap.add_payload(cbor.dumps(23.5))
coap.dump()
answer = CoAP.send_ack(s, destination, coap)
answer.dump()
```

Notez que ce programme peut aussi bien s'exécuter sur votre ordinateur ou sur votre pycom. Dans le premier cas, le module cbor2 sera utilisé. Sur le pycom nous ferons appel au module de kpn_senml. Dans les deux cas, le contenu est transformé en CBOR grâce à la fonction dumps

Du coté du serveur, la méthode render_post doit connaître le format de la ressource. Pour ce faire, elle doit accéder aux options CoAP.

```python
class TemperatureResource(resource.Resource):
async def render_post(self, request):
print ("-"\*20)
ct = request.opt.content_format or \
aiocoap.numbers.media_types_rev['text/plain']
if ct == aiocoap.numbers.media_types_rev['text/plain']:
print ("text:", request.payload)
elif ct == aiocoap.numbers.media_types_rev['application/cbor']:
print ("cbor:", cbor.loads(request.payload))
else:
print ("Unknown format")
return aiocoap.Message(code=aiocoap.UNSUPPORTED_MEDIA_TYPE)
return aiocoap.Message(code=aiocoap.CHANGED)
```

Dans le programme coap_basic_server3.py, la variable ct contient la valeur de l’option Content-format si elle existe. De manière générale, *aiocoap* permet d’accéder aux valeurs de toutes les options CoAP contenues dans la requête. None indique que l’option n’est pas présente dans l’en-tête. Dans ce cas, la variable ct contiendra la valeur par défaut indiquant un texte en ASCII.

Suivant le format de la donnée, le programme affichera le texte ou le CBOR transformé en chaîne de caractères avec la fonction loads. Si le format n’est pas connu du programme, un code d’erreur est retourné au client.

### Optimisation de la voie retour

On a vu que les réseaux de type LPWAN n’aiment pas trop les acquittements qui vont produire des messages en *downlink*. On peut essayer de les supprimer en utilisant le type NON confirmé plutôt que les messages CONfirmés que nous avons utilisés jusqu’à présent.

La fonction send_ack retourne None pour un message NON confirmé. Dans ce cas nous utilisons la fonction get_msg pour récupérer le prochain message CoAP qui est envoyé au client.

```python
import CoAP
import socket
import time
try:
import kpn_senml.cbor_encoder as cbor #pycom
except:
import cbor2 as cbor # terminal on computer
SERVER = "192.168.1.26" # change to your server's IP address
PORT = 5683
destination = (SERVER, PORT)
s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
coap = CoAP.Message()
coap.new_header(type=CoAP.NON, code=CoAP.POST)
coap.add_option (CoAP.Uri_path, "temp")
coap.add_option (CoAP.Content_format, CoAP.Content_format_CBOR)
coap.add_payload(cbor.dumps(23.5))
coap.dump(hexa=True)
answer = CoAP.send_ack(s, destination, coap)
if answer != None:
answer.dump()
else:
answer = CoAP.get_msg(s)
if answer != None:
answer.dump()
```

En lançant le programme coap_post_temp3.py  côté le client, on a le résultat suivant :

```
%python3.7 coap_post_temp3.py
b'50020001b474656d70113cfffb4037800000000000'
NON 0x0001 POST
> Uri-path : b'temp'
> Content-format : 60
---CONTENT
hex: b'fb4037800000000000'
NON 0xCB18 2.04
```

L’acquittement que l’on avait précédemment s’est transformé en un message NON confirmé du serveur vers le client pour transporter la notification du POST. On va donc avoir un message en *downlink* pour chaque requête.

Pour éviter ce comportement, le [RFC 7967](https://tools.ietf.org/html/rfc7967) permet au client de signaler au serveur qu’il n’est pas intéressé par tous les acquittements. Il transporte un masque binaire sur un octet, indiquant quand le bit est à 1 que le client n’est pas intéressé par les notifications de cette catégorie. Ainsi :

* la valeur 0b0000_0010 supprime les notifications de type 2.0x, c’est à dire les notifications positives.
* la valeur 0b0000_1000 supprime les notifications de type 4.0x, c’est à dire les erreurs dues au comportement du client.
* la valeur 0b0001_0000 supprime les notifications de type 5.0x, c’est à dire les erreurs dues au serveur.

Elles peuvent être combinées entre elles. La valeur 0b0001_0010 n’acquittera que les erreurs produites par le client. Une option vide indique que le client est intéressé par toutes les notifications.

Cette option est positionnée dans le programme coap_post_temp4.py dont voici la partie construisant le message CoAP.

```
coap = CoAP.Message()
coap.new_header(type=CoAP.NON, code=CoAP.POST)
coap.add_option (CoAP.Uri_path, "temp")
coap.add_option (CoAP.Content_format, CoAP.Content_format_CBOR)
coap.add_option (CoAP.No_Response, 0b00000010)
coap.add_payload(cbor.dumps(23.5))
coap.dump()
```

Quand le client envoie une requête, il ne reçoit pas de réponse.

```
%python3.7 coap_post_temp3.py
NON 0x0001 POST
> Uri-path : b'temp'
> Content-format : 60
> No-Response : 2
---CONTENT
hex: b'fb4037800000000000'
```

**Quiz : Blank Common Problem**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Changez l’URI pour une qui n’est pas connue du serveur. Recevez vous une réponse malgré l’emploi de l’option No-Response ?========ATTENTION METTRE AU MINIMUM 4 REPONSES CAR L'APPRENANT AURA 2 TENTATIVES================**"))
q0_w = widgets.RadioButtons(options=['oui 4.04 (Not Found)', 'oui 4.05 (Method Not Allowed)', 'non'], description="", layout=widgets.Layout(width="90%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value == 'oui 4.04 (Not Found)', ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)

display(Markdown("**Quelle est la taille minimale d’un requête POST CoAP ?**"))
q1_w = widgets.Text(layout=widgets.Layout(width="40%"))
q1_b = widgets.Button(description="Verifier")
q1_o = widgets.Output()
def q1_check(_):
    q1_o.clear_output()
    with q1_o:
        display(_feedback(_num_ok(q1_w.value, "11"), ""))
q1_b.on_click(q1_check)
display(q1_w, q1_b, q1_o)

display(Markdown("**Est-il efficace de faire du CoAP sur Sigfox ? ========ATTENTION METTRE AU MINIMUM 4 REPONSES CAR L'APPRENANT AURA 2 TENTATIVES================**"))
q2_w = widgets.RadioButtons(options=['Oui', 'Non'], description="", layout=widgets.Layout(width="90%"))
q2_b = widgets.Button(description="Verifier")
q2_o = widgets.Output()
def q2_check(_):
    q2_o.clear_output()
    with q2_o:
        display(_feedback(q2_w.value == 'Non', "Si on regarde l'en-tête minimale CoAP on a 4 octets obligatoire, 2 octets Uri-path (nom sur 1 caractère), 2 octets pour Content-format et 2 pour No-Response et un octet pour le séparateur. Sur Sigfox, il ne reste plus qu'un octet pour transmettre des données."))
q2_b.on_click(q2_check)
display(q2_w, q2_b, q2_o)



```
Vous avez tous les éléments pour combiner les programmes écrits la dernière fois pour récolter les séries temporelles provenant du capteur de température humidité pression.
La dernière fois nous n’avions remonté que la température. Avec CoAP, il est possible de créer trois ressources sur le serveur pour chacune de ces mesures. Evitez d’envoyer des messages en downlink, même si vous utilisez le réseau Wi-Fi.
Comme le format CBOR que l’on utilise n’est pas très standard, vous pouvez définir votre propre valeur de Content-format (plido-cbor) pour indiquer au serveur qu’il s’agit de données CBOR codées en delta avec un précision du centième.
```

## Experimentons CoAP

### TP Mise en œuvre de GET

#### Installation

*Les programmes se trouvent dans le répertoire pycom pour l'objet et plido-tp4 pour le serveur.*

*Les programmes pourront tourner indifféremment dans une fenêtre de votre ordinateur ou sur votre Pycom. Dans la suite, nous supposerons qu'il s'agit d'une deuxième fenêtre de terminal sur votre ordinateur mais libre à vous de le lancer sur votre Pycom en Wi-Fi (voire en LoRaWAN) pour plus de réalisme. L'utilisation du réseau Sigfox est plus problématique vu que les requêtes provenant de votre Pycom sont limitées à 12 caractères et que les réponses ne sont qu'au nombre de 4 par jour et limitées à 8 caractères.*

*Pour nous concentrer sur le fonctionnement de CoAP, nous allons tout d'abord expérimenter en local sur notre ordinateur. Nous verrons par la suite comment utiliser le programme generic_relay.py pour bénéficier des réseaux LPWAN.*

*Attention, le serveur ne fonctionne pas directement sous Windows. Vous devez impérativement le faire tourner dans la machine virtuelle.*

#### Cinéma



In [ ]:
from IPython.display import YouTubeVideo
YouTubeVideo("Nieitr19VGg", width=700, height=394)

#### Démarrage

# TP Mise en œuvre de GET

Nous allons mettre en œuvre le protocole CoAP entre deux processus sur votre machine puis, si vous le voulez, vous pourrez le tester sur un Pycom. Côté objet, nous allons utiliser une mise en œuvre simple mais compacte du protocole pour comprendre son fonctionnement. À l’autre extrémité, nous utiliserons la mise en œuvre *aiocoap* qui est très complète mais beaucoup plus complexe et demandant plus de ressources.

### aiocoap

Comme son nom l'indique, *aiocoap* met en œuvre CoAP, avec les modules *asynchronous Input/Output*, permettant un fort degré de parallélisme. Le programme suivant (coap_basic_server1.py) donne un exemple d’un serveur simple gérant une seule ressource time :

```python
#!/usr/bin/env python3
# This file is part of the Python aiocoap library project.
#
# Copyright (c) 2012-2014 Maciej Wasilak <http://sixpinetrees.blogspot.com/>,
# 2013-2014 Christian Amsüss <c.amsuess@energyharvesting.at>
#
# aiocoap is free software, this file is published under the MIT license as
# described in the accompanying LICENSE file.
"""This is a usage example of aiocoap that demonstrates how to implement a
simple server. See the "Usage Examples" section in the aiocoap documentation
for some more information."""
import datetime
import logging
import socket
import asyncio
import aiocoap.resource as resource
import aiocoap
class TimeResource(resource.Resource):
async def render_get(self, request):
await asyncio.sleep(5)
payload = datetime.datetime.now().\
strftime("%Y-%m-%d %H:%M").encode('ascii')
return aiocoap.Message(payload=payload)
# logging setup
logging.basicConfig(level=logging.INFO)
logging.getLogger("coap-server").setLevel(logging.DEBUG)
def main():
# Resource tree creation
with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as s:
s.connect(("8.8.8.8", 80)) # connect outside to get local IP address
ip_addr = s.getsockname()[0]
port = 5683
print ("server running on ", ip_addr, "at port", port)
root = resource.Site()
root.add_resource(['time'], TimeResource())
asyncio.Task(
aiocoap.Context.create_server_context(root,
bind=(ip_addr, port)))
asyncio.get_event_loop().run_forever()
if __name__ == "__main__":
main()
```

L’utilisation de *aicoap* se traduit par l’importation des modules qui se trouvent dans le répertoire aoicoap.

Dans la fonction main le programme cherche son adresse IP fixe et le numéro de port à la valeur affectée à CoAP (5683).  L’utilisation d’une adresse IP et non du joker 0.0.0.0 permet de faire tourner le serveur dans un environnement Windows et MAC.

La variable root contient l’arbre des ressources. À la ligne suivante, la ressource time est associée à une classe TimeResource.

La méthode

asyncio.Task(aiocoap.Context.create_server_context(root, bind=(ip_addr, port)))

permet de lier cet arbre de ressource à l’adresse IP et au numéro de port précédemment défini. Le serveur est ensuite lancé dans une boucle sans fin.

Il est plus intéressant de voir le traitement effectué lorsque la ressource est appelée par le serveur. La classe TimeResource dérivant de la classe générique aiocoap Resource est utilisée :

```
class TimeResource(resource.Resource):
```

Pour chaque méthode CoAP, une méthode peut être définie. Dans l’exemple, la méthode render_get permet de traiter les requêtes GET. Pour simuler un temps de traitement, le programme commence par attendre 5 secondes puis construit la chaîne de caractères contenant la date qu’il va retourner dans un objet aoicoap Message.

Ainsi, si tout se passe correctement, la réponse à une requête (Code = 0x01) sera 2.05 (*Content*). La figure suivante résume cet échange que nous avions vu théoriquement dans le chapitre consacré aux *tokens*.

![](images/im31.png)

```
Sur votre ordinateur, lancez le programme coap_basic_server1.py dans un terminal. On vous rappelle que sur Windows, vous devez impérativement passer par la machine virtuelle.
```

#### Messages vides

Messages vides

Du coté du client, nous allons utiliser une mise en œuvre plus compacte qui nous permettra d’expérimenter le fonctionnement de CoAP en modifiant les valeurs des champs protocolaires.

Le programme coap_empty_msg.py :

```python
import CoAP
import socket
SERVER = "192.168.1.26" # change to your server's IP address
PORT = 5683
s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
coap = CoAP.Message()
coap.new_header()
coap.dump(hexa=True)
s.settimeout(10)
s.sendto (coap.to_byte(), (SERVER, 5683))
resp,addr = s.recvfrom(2000)
answer = CoAP.Message(resp)
answer.dump(hexa=True)
```

Le programme est beaucoup plus simple. Dans un premier temps, vous devez remplacer l’adresse IP par celle fournie par votre serveur CoAP. Le programme crée une socket UDP au travers de laquelle l’échange avec le serveur sera effectué. La première action consiste à créer un message CoAP (ligne 9) et à la ligne suivante créer un en-tête obligatoire.

Le programme affiche le message et l’envoie sur la socket. La ligne 13 permet de limiter l’attente de la réponse à 10 secondes. Cette réponse est attendue ligne 16, transformée en message CoAP ligne suivante et affichée.

Lancez une capture Wireshark pour voir le trafic passant sur le port de CoAP (udp.port==5683 dans la fenêtre de filtrage).

Lancez maintenant le programme client.

On obtient le résultat suivant :

```
b ’ 40000001 ’
CON 0 x0001 EMPTY
b ’ 70000001 ’
RST 0 x0001 EMPTY
```

Les messages suivants ont circulé sur le réseau.

```
18:38:30.381449 IP 192.168.1.26.50883 > 192.168.1.26.5683: UDP, length 4
0x0000: 4500 0020 221f 0000 4011 0000 c0a8 011a E..."...@.......
0x0010: c0a8 011a c6c3 1633 000c 83a2 4000 0001 .......3....@...
18:38:30.382107 IP 192.168.1.26.5683 > 192.168.1.26.50883: UDP, length 4
0x0000: 4500 0020 efbb 0000 4011 0000 c0a8 011a E.......@.......
0x0010: c0a8 011a 1633 c6c3 000c 83a2 7000 0001 .....3......p...
```

On retrouve dans le contenu des messages UDP, le message CoAP donné par l’application. Si l’on prend le deuxième message, il commence par 0x70 ; ce qui correspond en binaire à 0b01_11_0000, soit version = 1, type = 3 et longueur du token = 0. L’octet suivant donne le code 0 *(Empty)* et les deux derniers octets contiennent le message ID. Le serveur ne sachant pas quoi faire de la requête, le rejette en envoyant un message Reset pour essayer d’arrêter le code sur le client qui envoie ce genre de requête.

#### GET /time

### GET /time

### 

Nous laissons tourner le serveur CoAP et nous allons construire la requête CoAP du client pour qu'il demande la ressource /time. N'oubliez pas de mettre la bonne adresse IP du SERVER dans ce programme.

```python
import CoAP
import socket
SERVER = "192.168.1.XX" # change to your server's IP address
PORT = 5683
s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
coap = CoAP.Message()
coap.new_header(code=CoAP.GET)
coap.add_option (CoAP.Uri_path, "time")
coap.dump(hexa=True)
s.sendto (coap.to_byte(), (SERVER, PORT))
s.settimeout(10)
resp,addr = s.recvfrom(2000)
answer = CoAP.Message(resp)
answer.dump(hexa=True)
```

* la méthode new_header() précise le code, ici GET ;
* et on ajoute l'élément d'URI time en option.

Côté client, on obtient le résultat suivant :

```
> python3 coap_get_time1.py
False
b'40010001b474696d65'
CON 0x0001 GET
> Uri-path : b'time'
b'60000001'
ACK 0x0001 EMPTY
```

La requête CoAP commence par le mot 40010001, indiquant un message CONfirmable, sans Token, un code GET et un MID de 0x0001, suivi de l'option Uri-Path.

La réponse est un ACK du MID et le code est vide (0.00).

On n'obtient pas la réponse au GET, juste un acquittement. Pourtant, les log du serveur et l'analyse du réseau montrent bien que le serveur à répondu.

```
DEBUG:coap-server:Incoming message <aiocoap.Message at 0x7f3d997ace80: Type.CON GET (MID 1, empty token) remote <UDP6EndpointAddress 192.168.1.79:52495 (locally 192.168.1.79%lo)>, 1 option(s)>
DEBUG:coap-server:New unique message received
DEBUG:coap-server:Sending empty ACK: Response took too long to prepare
DEBUG:coap-server:Sending message <aiocoap.Message at 0x7f3d99742748: Type.ACK EMPTY (MID 1, empty token) remote <UDP6EndpointAddress 192.168.1.79:52495 (locally 192.168.1.79%lo)>>
```

![](images/capture29.png)

Comme nous avons ajouté un délai de 5 secondes avant de répondre sur le serveur CoAP dans la fonction render_get(), le serveur acquitte la requête et cherche 5 secondes plus tard à envoyer une nouvelle requête confirmée. Mais le client, ayant fermé sa socket, ne peut plus la recevoir et retourne une erreur ICMP.

La solution est d'attendre un second message et de le décoder comme le montre le listing suivant qui donne le code du programme coap_get_time2.py dans lequel vous n'aurez pas oublié de changer l'adresse IP du serveur.

```python
import CoAP
import socket
SERVER = "192.168.1.XX" # change to your server's IP address
PORT = 5683
s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
coap = CoAP.Message()
coap.new_header(code=CoAP.GET)
coap.add_option (CoAP.Uri_path, "time")
coap.dump()
s.sendto (coap.to_byte(), (SERVER, PORT))
s.settimeout(10)
resp,addr = s.recvfrom(2000)
answer = CoAP.Message(resp)
answer.dump()
s.settimeout(10)
resp,addr = s.recvfrom(2000)
answer = CoAP.Message(resp)
answer.dump()
```

Nous pouvons laisser éclater notre joie car nous avons la réponse :

```
> python3 coap_get_time2.py
False
CON 0x0001 GET
> Uri-path : b'time'
ACK 0x0001 EMPTY
CON 0x2708 2.05
---CONTENT
hex: b'323032312d30332d33302031343a3537'
txt: b'2021-03-30 14:57'
```

Mais notre joie est de courte durée car si on regarde plus attentivement le trafic sur le réseau :

![](images/capture30.png)

on voit que la réponse a été émise deux fois et que l'on retrouve ensuite une erreur ICMP. Cela est dû au fait que l'on n'acquitte pas le message venant du serveur. Le croyant perdu, il le retransmet et tombe sur une socket inexistante.

Pour bien faire, nous devons fournir un acquittement au serveur avec ce code de client avec le programme coap_get_time3.py.

```python
import CoAP
import socket
SERVER = "192.168.1.XX" # change to your server's IP address
PORT = 5683
s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
coap = CoAP.Message()
coap.new_header(code=CoAP.GET)
coap.add_option (CoAP.Uri_path, "time")
coap.dump()
s.sendto (coap.to_byte(), (SERVER, PORT))
s.settimeout(10)
resp,addr = s.recvfrom(2000)
answer = CoAP.Message(resp)
answer.dump()
s.settimeout(10)
resp,addr = s.recvfrom(2000)
answer = CoAP.Message(resp)
answer.dump()
mid = answer.get_mid()
ack = CoAP.Message()
ack.new_header(mid=mid, type=CoAP.ACK)
ack.dump()
s.sendto (ack.to_byte(), (SERVER, PORT))
```

Ici, l'exécution est parfaite :

```
> python3 coap_get_time3.py
False
CON 0x0001 GET
> Uri-path : b'time'
ACK 0x0001 EMPTY
CON 0x2709 2.05
---CONTENT
hex: b'323032312d30332d33302031353a3133'
txt: b'2021-03-30 15:13'
ACK 0x2709 EMPTY
```

si vous n'avez pas oublié de changer l'adresse IP du serveur, et l'utilisation du réseau est optimale.

![](images/capture31.png)

Enfin, on peut ajouter un token pour lier les deux transactions. Ici, il n'y a pas d’ambiguïté car nous ne demandons qu'une seule ressource. Mais si nous demandions plusieurs ressources simultanément, il faudrait pouvoir associer la réponse à la requête. Le programme coap_get_time4.py ajoute lors de la création de l'en-tête un champ token.

```python
import CoAP
import socket
SERVER = "192.168.1.XX" # change to your server's IP address
PORT = 5683
s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
coap = CoAP.Message()
coap.new_header(code=CoAP.GET, token=0x12345)
coap.add_option (CoAP.Uri_path, "time")
coap.dump()
s.sendto (coap.to_byte(), (SERVER, PORT))
s.settimeout(10)
resp,addr = s.recvfrom(2000)
answer = CoAP.Message(resp)
answer.dump()
s.settimeout(10)
resp,addr = s.recvfrom(2000)
answer = CoAP.Message(resp)
answer.dump()
mid = answer.get_mid()
ack = CoAP.Message()
ack.new_header(mid=mid, type=CoAP.ACK)
ack.dump()
s.sendto (ack.to_byte(), (SERVER, PORT))
```

L'échange est le même mais le token est répété dans la réponse.

```
> python3 coap_get_time4.py
False
CON 0x0001 GET T=012345
> Uri-path : b'time'
ACK 0x0001 EMPTY
CON 0x270A 2.05 T=012345
---CONTENT
hex: b'323032312d30332d33302031353a3232'
txt: b'2021-03-30 15:22'
ACK 0x270A EMPTY
```

**Quiz : Quiz**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Que se passe-t-il si vous utilisez une requête non confirmable pour demander la ressource /time (mettre l'argument type=CoAP.NON dans la construction de l'en-tête obligatoire).**"))
q0_w = widgets.RadioButtons(options=['Ça plante le serveur', 'Le serveur répond par une requête Confirmable', 'Le serveur répond par une requête Non Confirmable', "Le serveur retourne un RST car nous n'avons pas programmé ce cas.", "On passe en heure d'hiver"], description="", layout=widgets.Layout(width="90%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value == 'Le serveur répond par une requête Non Confirmable', ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)



**Quiz : Quiz**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Que se passe-t-il quand le client envoie une requête CONfirmable ?**"))
q0_w = widgets.RadioButtons(options=["Le serveur retourne la réponse dans l'acquittement.", 'Le serveur retourne une requête NON confirmable.', 'Le serveur attend quand même quelques secondes pour ne pas saturer le réseau.', 'Le serveur enlève le token de la réponse.'], description="", layout=widgets.Layout(width="90%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value == "Le serveur retourne la réponse dans l'acquittement.", ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)

display(Markdown("**Que se passe-t-il quand le client envoie une requête NON confirmable ?**"))
q1_w = widgets.RadioButtons(options=["Le serveur retourne la réponse dans l'acquittement.", 'Le serveur retourne une requête NON confirmable.', 'Le serveur attend quand même quelques secondes pour ne pas saturer le réseau.', 'Le serveur enlève le token de la réponse.'], description="", layout=widgets.Layout(width="90%"))
q1_b = widgets.Button(description="Verifier")
q1_o = widgets.Output()
def q1_check(_):
    q1_o.clear_output()
    with q1_o:
        display(_feedback(q1_w.value == 'Le serveur retourne une requête NON confirmable.', ""))
q1_b.on_click(q1_check)
display(q1_w, q1_b, q1_o)



### TP Les POST

#### codage en texte

### Ressource codée en texte

Nous allons voir le traitement d’une requête POST avec *aiocoap.* Nous aurions pu ajouter ce traitement à la classe TimeResource vue précédemment, mais nous avons préféré ajouter une nouvelle ressource pour le traitement des températures.

Les lignes suivantes ont été ajoutées pour traiter le POST. Notez le nom de la méthode render_post.

```python
class TemperatureResource(resource.Resource):
async def render_post(self, request):
print ("-"\*20)
print ("payload:", binascii.hexlify(request.payload))
resp = aiocoap.Message(code=aiocoap.CHANGED)
return resp
```

La réponse à la requête est le code 2.04 (aiocoap.CHANGED) qui indique que la ressource été modifiée. 

On lui associe une nouvelle classe dans l’arbre des ressources :

```
root.add_resource(['temp'], TemperatureResource())
```

Le tout forme le programme coap_basic_server2.py.

Coté client, le principe est le même que pour le GET. Dans le programme coap_post_temp1.py,  la méthode add_payload permet d’ajouter du contenu à la requête POST. Le programme émet sa requête et affiche le résultat. Nous allons également simplifier la gestion des réponses en utilisant la fonction send_ack du module CoAP. Cette fonction prend en argument une socket, un tuple de destination et le message CoAP à envoyer. Elle va le retransmettre au maximum 4 fois tant qu’elle n’a pas reçu l’acquittement correspondant.

```python
import CoAP
import socket
import time
SERVER = "192.168.1.XX" # change to your server's IP address
PORT = 5683
destination = (SERVER, PORT)
s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
coap = CoAP.Message()
coap.new_header(code=CoAP.POST)
coap.add_option (CoAP.Uri_path, "temp")
coap.add_payload("23.5")
coap.dump()
answer = CoAP.send_ack(s, destination, coap)
answer.dump()
```

La figure suivante montre un échange avec perte de messages.

![](images/im33.png)

Elles peuvent être simulées en arrêtant le programme serveur sur votre ordinateur.

* Lancez le serveur coap_basic_server2.py et tapez ctrl-Z pour stopper son exécution, tout en laissant ouverte la socket.

```
> python3 ./coap_basic_server2.py
server running on 192.168.1.79 at port 5683 ^Z
Job 2, “python3 ./coap_basic_server2.py” has stopped
>
```

* Lancez le client coap_port_temp1.py (en n'oubliant pas de changer l'adresse IP du serveur).

```
> python3.9 coap_post_temp1.py
False
CON 0x0001 POST
> Uri-path : b'temp'
---CONTENT
hex: b'32332e35'
txt: b'23.5'
timeout 2 1
timeout 4 2
```

On voit que le client ne reçoit pas l'acquittement, déclenche un temporisateur et retransmet le message. La durée du temporisateur double à chaque tentative. Le standard recommande d'ajouter un aléa lors de la transmission pour éviter que deux équipements se synchronisent et se perturbent toujours en même temps. Il n'est pas mis en œuvre dans cet exemple.

* Réactivez le serveur coap en tapant fg (foregroung)

```
> fg
DEBUG:coap-server:Incoming message <aiocoap.Message at 0x7f05d621fcc0: Type.CON POST (MID 1, empty token) remote <UDP6EndpointAddress 192.168.1.79:37286 (locally 192.168.1.79%lo)>, 1 option(s), 4 byte(s) payload>
DEBUG:coap-server:New unique message received
--------------------
payload: b'32332e35'
DEBUG:coap-server:Sending message <aiocoap.Message at 0x7f05d622df98: Type.ACK 2.04 Changed (MID 1, empty token) remote <UDP6EndpointAddress 192.168.1.79:37286 (locally 192.168.1.79%lo)>>
DEBUG:coap-server:Incoming message <aiocoap.Message at 0x7f05d621fcc0: Type.CON POST (MID 1, empty token) remote <UDP6EndpointAddress 192.168.1.79:37286 (locally 192.168.1.79%lo)>, 1 option(s), 4 byte(s) payload>
INFO:coap-server:Duplicate CON received, sending old response again
DEBUG:coap-server:Sending message <aiocoap.Message at 0x7f05d622df98: Type.ACK 2.04 Changed (MID 1, empty token) remote <UDP6EndpointAddress 192.168.1.79:37286 (locally 192.168.1.79%lo)>>
DEBUG:coap-server:Socket error recevied, details: SockExtendedErr(ee_errno=111, ee_origin=2, ee_type=3, ee_code=3, ee_pad=0, ee_info=0, ee_data=0)
DEBUG:coap-server:Incoming error 111 from <UDP6EndpointAddress 192.168.1.79:37286 (locally 192.168.1.79%lo)>
DEBUG:coap-server:Incoming message <aiocoap.Message at 0x7f05d622deb8: Type.CON POST (MID 1, empty token) remote <UDP6EndpointAddress 192.168.1.79:37286 (locally 192.168.1.79%lo)>, 1 option(s), 4 byte(s) payload>
INFO:coap-server:Duplicate CON received, sending old response again
DEBUG:coap-server:Sending message <aiocoap.Message at 0x7f05d622df98: Type.ACK 2.04 Changed (MID 1, empty token) remote <UDP6EndpointAddress 192.168.1.79:37286 (locally 192.168.1.79%lo)>>
DEBUG:coap-server:Socket error recevied, details: SockExtendedErr(ee_errno=111, ee_origin=2, ee_type=3, ee_code=3, ee_pad=0, ee_info=0, ee_data=0)
DEBUG:coap-server:Incoming error 111 from <UDP6EndpointAddress 192.168.1.79:37286 (locally 192.168.1.79%lo)>
```

On peut observer plusieurs phénomènes :

* le serveur avait gardé en mémoire les requêtes précédentes mais ne les avait pas traitées car le programme avait été stoppé. En le redémarrant, il va pouvoir les traiter. Il répond donc au client qui peut afficher la notification 2.04 ;
* les autres requêtes sont vues comme des répétitions de la première puisque le champ Message ID est identique. Le serveur se contente de retourner une copie ;
* le client ayant terminé sa transaction a fermé la socket conduisant à l'émission d'un message ICMP qui conduit à l'affichage du message de log d'erreur (Socket error received).

Ceci peut être vérifié sur la capture *Wireshark.*

![](images/capture32.png)

**Quiz : Quizz**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Combien de tentatives de REtransmission fait le client avant d'abandonner ?**"))
q0_w = widgets.Text(layout=widgets.Layout(width="40%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(_num_ok(q0_w.value, "4"), ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)



#### Codage en CBOR

### Ressource codée en CBOR

Ce programme est correct mais nous avons utilisé le format ASCII de transfert par défaut (Content-format : 0). Il serait préférable de revenir au format JSON ou CBOR que nous avions vu précédemment. Pour ce faire, nous devons ajouter dans l’en-tête de la requête l’option Content-format. Comme son code est 12, elle doit être placée après l’option Uri-path (cf. [tableau des options](https://www.iana.org/assignments/core-parameters/core-parameters.xhtml#option-numbers)).

```python
import CoAP
import socket
import time
try:
import kpn_senml.cbor_encoder as cbor #pycom
except:
import cbor2 as cbor # terminal on computer
SERVER = "192.168.1.XX" # change to your server's IP address
PORT = 5683
destination = (SERVER, PORT)
s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
coap = CoAP.Message()
coap.new_header(code=CoAP.POST)
coap.add_option (CoAP.Uri_path, "temp")
coap.add_option (CoAP.Content_format, CoAP.Content_format_CBOR)
coap.add_payload(cbor.dumps(23.5))
coap.dump()
answer = CoAP.send_ack(s, destination, coap)
answer.dump()
```

Notez que ce programme peut aussi bien s'exécuter sur votre ordinateur ou sur votre Pycom. Dans le premier cas, le module cbor2 sera utilisé. Sur le Pycom, nous ferons appel au module kpn_senml. Dans les deux cas, le contenu est transformé en CBOR grâce à la fonction dumps.

Du coté du serveur, la méthode render_post doit connaître le format de la ressource. Pour ce faire, elle doit accéder aux options CoAP.

```python
class TemperatureResource(resource.Resource):
async def render_post(self, request):
print ("-"\*20)
ct = request.opt.content_format or \
aiocoap.numbers.media_types_rev['text/plain']
if ct == aiocoap.numbers.media_types_rev['text/plain']:
print ("text:", request.payload)
elif ct == aiocoap.numbers.media_types_rev['application/cbor']:
print ("cbor:", cbor.loads(request.payload))
else:
print ("Unknown format")
return aiocoap.Message(code=aiocoap.UNSUPPORTED_MEDIA_TYPE)
return aiocoap.Message(code=aiocoap.CHANGED)
```

Dans le programme coap_basic_server3.py, la variable ct contient la valeur de l’option Content-format si elle existe. De manière générale, *aiocoap*permet d’accéder aux valeurs de toutes les options CoAP contenues dans la requête. None indique que l’option n’est pas présente dans l’en-tête. Dans ce cas, la variable ct contiendra la valeur par défaut indiquant un texte en ASCII.

Suivant le format de la donnée, le programme affichera le texte ou le CBOR transformé en chaîne de caractères avec la fonction loads. Si le format n’est pas connu du programme, un code d’erreur est retourné au client.

![](images/capture33.png)

**Quiz : Quiz**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Que reçoit-on en réponse à la requête POST ?**"))
q0_w = widgets.RadioButtons(options=['un message ACK', 'le statut 2.00', 'le statut 2.04 CHANGED', 'rien'], description="", layout=widgets.Layout(width="90%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value == 'le statut 2.04 CHANGED', ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)

display(Markdown("**Modifiez le programme client pour indiquer un content-format JSON. Quelle notification obtenez-vous ?**"))
q1_w = widgets.RadioButtons(options=['un message RST', 'le statut 4.04 NOT FOUND', 'le statut 4.15', 'le statut 5.00'], description="", layout=widgets.Layout(width="90%"))
q1_b = widgets.Button(description="Verifier")
q1_o = widgets.Output()
def q1_check(_):
    q1_o.clear_output()
    with q1_o:
        display(_feedback(q1_w.value == 'le statut 4.15', ""))
q1_b.on_click(q1_check)
display(q1_w, q1_b, q1_o)



#### No Response

### No Response

Que l'on utilise le type CONfirmable ou NON confirmable, le serveur va envoyer une notification : 2.xx quand tout se passe bien, et 4.xx ou 5.xx quand le client ou le serveur a commis une erreur.

Si un capteur veut utiliser CoAP sur un réseau LPWAN, à chaque POST qu'il va envoyer sur la voie montante (uplink), il va récupérer un acquittement dans la voie descendante (downlink). Or, on a vu que l'on devait ménager cette dernière qui pouvait être sujette à saturation ou à facturation.

L'option No response définie dans le [RFC 7967](https://datatracker.ietf.org/doc/rfc7967/) permet à un client d'informer le serveur qu'il ne souhaite pas recevoir de notifications lors d'un POST ou d'un PUT.  La valeur de l'option est 258 et elle est suivie d'un octet contenant un bitmap qui va indiquer quel type de notification va être émis :

* si le deuxième bit à partir de la droite est mis à 1, le client ne veut pas recevoir les notifications de type 2.xx ;
* si le quatrième bit à partir de la droite est mis à 1, le client ne veut pas recevoir les notifications de type 4.xx ;
* si le cinquième bit à partir de la droite est mis à 1, le client ne veut pas recevoir les notifications de type 5.xx.

Par exemple, en mettant la valeur 0x02 dans cette option, le client ne reçoit pas les acquittements positifs mais uniquement les erreurs 4.xx et 5.xx. En mettant la valeur 0b00011010 (0x1a, 26), le serveur sera complètement silencieux.

Le programme coap_port_temp4.py ajoute cette option et le type du message CoAP a été fixé à NON confirmable. Cependant, le chemin d'URI n'est pas connu du serveur.  On voit que le serveur retourne un message d'erreur.

![](images/capture34.png)

Si en revanche le POST se déroule correctement, seul le client émet une donnée et le serveur reste silencieux.

### On met tout ensemble

#### Vidéo



In [ ]:
from IPython.display import YouTubeVideo
YouTubeVideo("TFeeSEqGhPk", width=700, height=394)

#### Explications

Voilà ! nous pouvons mettre les différents concepts ensemble pour construire notre capteur de température, humidité et pression :

* côté capteur, nous allons, si le BME280 est présent, récupérer directement les données. Sinon, nous allons utiliser les mesures virtuelles. Nous allons également définir le protocole réseau : WiFi, Sigfox, LoRaWAN ;
* si nous utilisons un réseau LoRaWAN, nous devons mettre en place un programme relais pour transmettre les données vers le serveur CoAP ;
* le serveur CoAP doit pouvoir envoyer les données au serveur *beebotte* pour affichage.

Le message CoAP va être configuré de la manière suivante :

* type = NON pour éviter les acquittements des messages CoAP ;
* l'option No Response à 2 pour éviter les notifications REST positives. On garde sur la voie descendante les notifications d'erreurs pour permettre au capteur d'arrêter de transmettre ou d'augmenter sa période d'émission si le serveur n'est pas capable de traiter les données. Cela permettra de limiter l'usage du spectre et de préserver l'énergie des capteurs ;
* 3 chemins d'URI pour les 3 valeurs mesurées ;
* le codage en CBOR des séries temporelles.

Il reste un point à traiter car avec la limitation à 12 octets des trames Sigfox, il est impossible de transmettre des données si on utilise CoAP. En effet, l'en-tête CoAP va contenir :

* 4 octets pour l'en-tête obligatoire ;
* 0 octets de Token ;
* 2 octets au minimum pour l'option URI-path si on réduit à un caractère le chemin ; par exemple T, H, P pour représenter la température, l'humidité et la pression ;
* 2 octets pour l'option Content-format indispensable pour indiquer que l'on transporte du CBOR ;
* 3 octets pour l'option No-response, également indispensable pour indiquer que l'on ne veut surtout pas d'acquittement (Sigfox les limitant à 4 par jour).

Soit, au total 11 octets. Il n'en reste plus qu'un. Si la température dépasse les 23 °C, l'information ne sera pas transportable.

#### SCHC

### SCHC

*Static Context Header Compression*, défini dans le [RFC 8724](https://datatracker.ietf.org/doc/html/rfc8724), donne l'acronyme SCHC que l'on prononce Chic. SCHC propose un mécanisme de compression générique des en-têtes. SCHC se base sur un contexte commun à l'émetteur et au récepteur qui va permettre d'éliminer les informations connues dans le message et de ne transmettre que les données qui ne peuvent pas être prédéterminées.

Nous allons mettre en œuvre une version simplifiée. Dans le message CoAP, nous avons besoin du champ Message ID qui va servir à éliminer les doublons qui pourraient apparaitre dans le réseau. Les serveurs CoAP gardent en mémoire les informations concernant les Messages ID pendant 5 minutes. Donc, la numérotation des messages doit permettre des périodes plus longues avant de réutiliser les mêmes valeurs. Nous allons également transporter 3 types de ressources : /temperature, /humidity et /pressure.

On peut donc construire manuellement un en-tête SCHC avec les champs suivants :

* 2 bits pour numéroter les règles de compression. Dans notre cas, nous n'utiliserons que la règle 00 ;
* 4 bits pour numéroter les messages ID, ce qui donne 15 valeurs possibles de 1 à 15. Au pire, il faudrait émettre plus de 3 trames par minutes pour que le serveur CoAP traite deux trames différentes comme des doublons ;
* 2 bits pour désigner les chemins d'URI ; 3 seront utilisés.

Cela fait appel à plusieurs techniques de compression définies par SCHC :

* not_sent: la valeur d'un champ n'est pas envoyée sur le réseau car elle se trouve dans les règles. Cela s'appliquera à la plupart des champs ;
* Least Significant Bit : on n'envoie que les bits de poids faible. Cela s'applique au champ Message ID duquel on ne va transmettre que les 4 bits de poids faible ;
* Matching_sent : au lieu d'envoyer la valeur, on va envoyer un index sur un tableau commun. Cela s'applique à Uri-path où l'on enverra 00 pour l'élément *temperature*, 01 pour l'élément *pressure*, et 10 pour élément *humidity*.

La compression très simplifiée que nous allons effectuer se fera au moment de l'envoi des données :

```python
def send_coap_message(sock, destination, uri_path, message):
if destination[0] == "SIGFOX": # do SCHC compression
global sigfox_MID
""" SCHC compression for Sigfox, use rule ID 0 stored on 2 bits,
followed by MID on 4 bits and 2 bits for an index on Uri-path.
the SCHC header is RRMMMMUU
"""
uri_idx = ['temperature', 'pressure', 'humidity', 'memory'].index(uri_path)
schc_residue = 0x00 # ruleID in 2 bits RR
schc_residue |= (sigfox_MID << 2) | uri_idx # MMMM and UU
sigfox_MID += 1
sigfox_MID &= 0x0F # on 4 bits
if sigfox_MID == 0: sigfox_MID = 1 # never use MID = 0
msg = struct.pack("!B", schc_residue) # add SCHC header to the message
msg += cbor.dumps(message)
print ("length", len(msg), binascii.hexlify(msg))
s.send(msg)
return None # don't use downlink
# for other technologies we wend a regular CoAP message
coap = CoAP.Message()
coap.new_header(type=CoAP.NON, code=CoAP.POST)
coap.add_option (CoAP.Uri_path, uri_path)
coap.add_option (CoAP.Content_format, CoAP.Content_format_CBOR)
coap.add_option (CoAP.No_Response, 0b00000010) # block 2.xx notification
coap.add_payload(cbor.dumps(message))
coap.dump(hexa=True)
answer = CoAP.send_ack(s, destination, coap)
return answer
```

Dans le cas de Sigfox, on prend l'index en cherchant l'élément dans le tableau

```
uri_idx = ['temperature', 'pressure', 'humidity', 'memory'].index(uri_path)
```

On construit ensuite l'octet SCHC en ajoutant un numéro de règle et les 4 bits de poids faible du champ Message ID qui va donc varier de 1 à 15.

```
schc_residue = 0x00 # ruleID in 2 bits RR
schc_residue |= (sigfox_MID << 2) | uri_idx # MMMM and UU
```

Puis, l'on concatène les données CBOR a envoyer.

Pour les autres technologies de transmission, l'en-tête CoAP est construite avec les fonctions du module CoAP.py.

```
coap = CoAP.Message()
coap.new_header(type=CoAP.NON, code=CoAP.POST)
coap.add_option (CoAP.Uri_path, uri_path)
coap.add_option (CoAP.Content_format, CoAP.Content_format_CBOR)
coap.add_option (CoAP.No_Response, 0b00000010) # block 2.xx notification
coap.add_payload(cbor.dumps(message))
```

L'envoi de la trame se fait avec la fonction send_ack.

```
answer = CoAP.send_ack(s, destination, coap)
Comme le message est de type NON, cette fonction n'attend pas de réponse après l'émission des données.
```

#### SCHC

Si les données émises par coap_full_client.py passent par un réseau LPWAN, le programme generic_coap_relay.py va servir d'intermédiaire pour les envoyer au serveur CoAP. Le programme est presque identique à celui que nous avions utilisé pour transmettre dans la session précédente. Les seules différences sont les numéros de ports utilisés : 5683 au lieu de 33033 et l'ajout de la décompression SCHC pour les données venant de Sigfox.

```
SCHC_byte = payload[0]
ruleID = SCHC_byte >> 6
mID = (SCHC_byte & 0b00111100) >> 2
uri_idx = SCHC_byte & 0b0000_0011
m = aiocoap.message.Message(
mtype=NON,
code=POST,
mid=mID,
payload=payload[1:]
)
m.opt.uri_path = (
["temperature", "pressure", "humidity", "memory"][uri_idx],
)
m.opt.content_format = 60
m.opt.no_response = 0b0000_0010
```

Pour reconstruire l'en-tête CoAP, SCHC se base sur des règles pour rendre le traitement indépendant des champs employés par le protocole. Mais ici, pour faire plus simple, nous utilisons le module Message d'aoicoap pour reconstituer le message CoAP.

Dans un premier temps, on extrait du premier octet la valeur du champ message ID et l'index des URI :

```
ruleID = SCHC_byte >> 6
mID = (SCHC_byte & 0b00111100) >> 2
uri_idx = SCHC_byte & 0b0000_0011
puis, à partir de ces valeurs et de celles connues à l'avance, le message CoAP est reconstitué.
Dans tous les cas, la fonction forward_data est utilisée. Elle retourne la réponse du serveur CoAP qui est renvoyée au réseau LPWAN suivant les principes que nous avions vus lors de la session précédente. Nous ne l'avons pas mis en œuvre pour Sigfox pour éviter une erreur vu que 4 messages par jours sont autorisés.
```

#### Serveur

Finalement, le programme coap_server.py a été étendu à différents chemins d'URI pour traiter les différentes ressources que vont nous envoyer les capteurs.

```python
class temperature(resource.Resource):
async def render_post(self, request):
root = resource.Site()
# add resource processing, /proxy is not used here, see comments in generic_sensor()
root.add_resource(['temperature'], temperature())
root.add_resource(['pressure'], pressure())
root.add_resource(['humidity'], humidity())
root.add_resource(['proxy'], generic_sensor())
# associate resource tree and socket
asyncio.Task(aiocoap.Context.create_server_context(root))
# let's go forever
asyncio.get_event_loop().run_forever()
# if no content_format option set the default value
ct = request.opt.content_format or \
aiocoap.numbers.media_types_rev['text/plain']
# text will just display the value
if ct == aiocoap.numbers.media_types_rev['text/plain']:
print ("text:", request.payload)
# cbor will be displayed and processed.
elif ct == aiocoap.numbers.media_types_rev['application/cbor']:
print ("cbor:", cbor.loads(request.payload))
to_bbt("capteurs", "temperature", cbor.loads(request.payload), period=60, factor=0.01)
else:
print ("Unknown format")
return aiocoap.Message(code=aiocoap.UNSUPPORTED_MEDIA_TYPE)
return aiocoap.Message(code=aiocoap.CHANGED)
```

Et dans le traitement de la ressource, l'appel a la fonction to_bbt a été ajouté. Voici l'exemple pour la température :

```python
class temperature(resource.Resource):
async def render_post(self, request):
# if no content_format option set the default value
ct = request.opt.content_format or \
aiocoap.numbers.media_types_rev['text/plain']
# text will just display the value
if ct == aiocoap.numbers.media_types_rev['text/plain']:
print ("text:", request.payload)
# cbor will be displayed and processed.
elif ct == aiocoap.numbers.media_types_rev['application/cbor']:
print ("cbor:", cbor.loads(request.payload))
to_bbt("capteurs", "temperature", cbor.loads(request.payload), period=60, factor=0.01)
else:
print ("Unknown format")
return aiocoap.Message(code=aiocoap.UNSUPPORTED_MEDIA_TYPE)
return aiocoap.Message(code=aiocoap.CHANGED)
```

#### Si vous vous ennuyez

Vous pouvez étendre cet ensemble de programmes. Voici quelques pistes d'amélioration :

* les capteurs envoient également leur mémoire disponible. Cela peut être utile pour détecter une fuite dans la mémoire ; par exemple quand une structure n'est jamais libérée. La structure CBOR est bien envoyée mais ni coap_server.py ni beebotte n'ont été configurés pour afficher ces valeurs. Vous pouvez donc l'intégrer dans la chaîne de traitement ;
* le POST de la mémoire conduit à l'émission d'un message en *downlink* pour indiquer l'erreur 4.04. C'est une conséquence de l'utilisation de l'option CoAP no_response qui ne bloque que les notifications de type 2.xx. Vous pouvez modifier le module CoAP.py pour que la réponse soit traitée. Par exemple, en limitant à un envoi par jour de cette ressource ;
* en plus de la mémoire, il peut être intéressant d'envoyer le niveau de la batterie. La page [Correct formula for BATT monitoring on expansion board | Pycom user forum](https://forum.pycom.io/topic/1690/correct-formula-for-batt-monitoring-on-expansion-board) donne des indications pour récupérer le niveau de la batterie ;
* finalement, nous n'avons pas réglé tous les problèmes d'interopérabilité. Si, par exemple, vous voulez envoyer toutes les heures un relevé de la mémoire libre et toutes les minutes la température, vous devez également modifier le programme coap_server.py. Si ces paramètres sont transmis de manière optimale par le serveur, le programme coap_serveur peut devenir complètement indépendant de la valeur mesurée.

## OMA Lightweight M2M

### TP LwM2M

#### Installation

Ce TP est conçu pour fonctionner sur la machine virtuelle où l'environnement de travail a été configuré. Vous pouvez également le faire tourner sur votre machine physique à condition d'avoir installé java (voir installation.sh). Le répertoire de travail est lwm2m où les programmes seront téléchargés.

#### Introduction

### 

Il est temps de voir une vraie plateforme IoT qui met en œuvre ce que nous venons d'apprendre sur CoAP et REST.

*Lightweight Machine to Machine* ou LwM2M est une architecture définie par l'OMA (*Open Mobile Alliance*) à l'origine pour permettre aux opérateurs de gérer certaines ressources sur les téléphones mobiles. Mais cette architecture peut être étendue à d'autres environnements.

Les spécifications sont accessibles sur le site de l'OMA (<https://omaspecworks.org/what-is-oma-specworks/iot/lightweight-m2m-lwm2m/>). Nous allons utiliser une mise en œuvre en Java appelé *leshan.* Pas de panique ! nous n'allons pas programmer. Nous aurons juste besoin d'un navigateur Web et de *Wireshark.*

LwM2M, comme son nom l'indique, se veut léger. C'est-à-dire que les mises en œuvre ne doivent pas être trop complexes et que le trafic engendré ne doit pas non plus être trop important. LwM2M est une plate-forme et elle va donc faire plus qu'un simple trafic REST. En particulier, l'un des rôles est de permettre aux objets de s'enregistrer et de décrire leurs caractéristiques. LmM2M va également structurer fortement les ressources en imposant des règles de nommages relativement contraintes et le format des ressources.

### Architecture

Comme tout système qui se respecte, LwM2M fonctionne en mode client/serveur. Le serveur est la plate-forme de gestion des objets et les clients sont les objets connectés sur le réseau.

* Allez dans le répertoire PLIDO/lwm2m

Vous devez y trouver deux fichiers leshan-client-demo.jar et leshan-server-demo.jar téléchargés lors du script d'installation.

Dans un premier temps, lancez *Wireshark* sur l'interface *loopback* et filtrez l'affichage du trafic en le limitant au port 5683 pour UDP (celui de CoAP) en tapant udp.port==5683 comme le montre l'animation suivante :

![](images/lw_wireshark.gif)

Nous allons maintenant lancer le serveur LwM2M. Tapez :

|  |  |
| --- | --- |
| ``` ``` | ``` > java -jar ./leshan-server-demo.jar 2020-10-22 01:34:09,365 INFO LeshanServer - LWM2M server started at coap://0.0.0.0/0.0.0.0:5683 coaps://0.0.0.0/0.0.0.0:5684 2020-10-22 01:34:09,553 INFO LeshanServerDemo - Web server started at http://0.0.0.0:8080/. ``` |

Il nous indique qu'il utilise le port 5683 pour CoAP et que l'on peut superviser le serveur avec un navigateur sur le port 8080.

![](images/lw_start_server.gif)

Lancez le navigateur et tapez l'adresse http://127.0.0.1:8080. La page suivante apparaît :

![](images/lw_firefox.gif)

Vous pouvez remarquer que l'ouverture du serveur LwM2M n'a provoqué aucun affichage de trafic CoAP dans *Wireshark*.

Lancez maintenant, dans une autre fenêtre, le client LwM2M :

|  |  |
| --- | --- |
| ``` ``` | ``` > java -jar ./leshan-client-demo.jar 2020-10-22 01:49:51,063 INFO LeshanClientDemo - Commands available : - create <objectId> : to enable a new object. - delete <objectId> : to disable a new object. - update : to trigger a registration update. - w : to move to North. - a : to move to East. - s : to move to South. - d : to move to West. 2020-10-22 01:49:51,064 INFO LeshanClient - Starting Leshan client ... 2020-10-22 01:49:51,158 INFO CaliforniumEndpointsManager - New endpoint created for server coap://localhost:5683 at coap://0.0.0.0:48274 2020-10-22 01:49:51,159 INFO LeshanClient - Leshan client[endpoint:ubuntu] started. 2020-10-22 01:49:51,160 INFO DefaultRegistrationEngine - Trying to register to coap://localhost:5683 ... 2020-10-22 01:49:51,252 INFO DefaultRegistrationEngine - Registered with location '/rd/BOr5Pg7yW8'. 2020-10-22 01:49:51,252 INFO DefaultRegistrationEngine - Next registration update to coap://localhost:5683 in 53s... ``` |

Comme nous utilisons une adresse de *loopback*, il est un peu plus difficile de repérer qui est le client et qui est le serveur. Le serveur attend sur le port 5683.

![](images/lw_start_client.gif)

**Quiz : Quiz**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Quelle est la nature de la requête émise par le client ?**"))
q0_w = widgets.RadioButtons(options=['GET', 'POST', 'PUT', 'DELETE'], description="", layout=widgets.Layout(width="90%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value == 'POST', ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)

display(Markdown("**Quel est le chemin de l'URI ?**"))
q1_w = widgets.RadioButtons(options=['vide', '/rd', '/lwm2m'], description="", layout=widgets.Layout(width="90%"))
q1_b = widgets.Button(description="Verifier")
q1_o = widgets.Output()
def q1_check(_):
    q1_o.clear_output()
    with q1_o:
        display(_feedback(q1_w.value == '/rd', ""))
q1_b.on_click(q1_check)
display(q1_w, q1_b, q1_o)

display(Markdown("**Quel est le format du contenu (content-format)?**"))
q2_w = widgets.RadioButtons(options=['text', 'XML', 'JSON', 'link-format', 'CBOR'], description="", layout=widgets.Layout(width="90%"))
q2_b = widgets.Button(description="Verifier")
q2_o = widgets.Output()
def q2_check(_):
    q2_o.clear_output()
    with q2_o:
        display(_feedback(q2_w.value == 'link-format', ""))
q2_b.on_click(q2_check)
display(q2_w, q2_b, q2_o)

display(Markdown("**À quelle période, en secondes, voyez-vous s'afficher des messages CoAP ?**"))
q3_w = widgets.Text(layout=widgets.Layout(width="40%"))
q3_b = widgets.Button(description="Verifier")
q3_o = widgets.Output()
def q3_check(_):
    q3_o.clear_output()
    with q3_o:
        display(_feedback(_num_ok(q3_w.value, "53"), ""))
q3_b.on_click(q3_check)
display(q3_w, q3_b, q3_o)



#### Décodage

Nous allons décoder l'Uri-Query de la requête CoAP à l'aide du document [LwM2M Core Specification](http://www.openmobilealliance.org/release/LightweightM2M/V1_1_1-20190617-A/OMA-TS-LightweightM2M_Core-V1_1_1-20190617-A.pdf) et du tableau 6.2.1 page 40.

**Quiz : Quiz**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Que signifie b=U ?**"))
q0_w = widgets.RadioButtons(options=['Les données seront émises avec les unités (Units)', 'Le protocole sous-jacent est en mode datagramme (UDP)', 'Le référenciel des unités est la représentation Universelle', "Le client n'est pas référencé (Unreferenced)"], description="", layout=widgets.Layout(width="90%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value == 'Le protocole sous-jacent est en mode datagramme (UDP)', ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)

display(Markdown("**Que signifie lwm2m=1.1 ? Il s'agit...**"))
q1_w = widgets.RadioButtons(options=['de la version lwm2m du client.', "de la taille mémoire de l'implémentation (1.1 ko).", 'de la version lwm2m du serveur.'], description="", layout=widgets.Layout(width="90%"))
q1_b = widgets.Button(description="Verifier")
q1_o = widgets.Output()
def q1_check(_):
    q1_o.clear_output()
    with q1_o:
        display(_feedback(q1_w.value == 'de la version lwm2m du client.', ""))
q1_b.on_click(q1_check)
display(q1_w, q1_b, q1_o)

display(Markdown("**Que signifie lt=300 ?**"))
q2_w = widgets.RadioButtons(options=["C'est le nombre maximal d'objets (less than 300).", "C'est la taille maximale d'un échange (300 octets).", "C'est la durée de vie de l'enregistrement d'un objet (lifetime)."], description="", layout=widgets.Layout(width="90%"))
q2_b = widgets.Button(description="Verifier")
q2_o = widgets.Output()
def q2_check(_):
    q2_o.clear_output()
    with q2_o:
        display(_feedback(q2_w.value == "C'est la durée de vie de l'enregistrement d'un objet (lifetime).", ""))
q2_b.on_click(q2_check)
display(q2_w, q2_b, q2_o)

display(Markdown("**Que signifie ep=virtual-ilab ?**"))
q3_w = widgets.RadioButtons(options=["C'est le système d'exploitation de l'objet.", "C'est le nom de l'équipement.", "C'est l'emplacement de l'objet."], description="", layout=widgets.Layout(width="90%"))
q3_b = widgets.Button(description="Verifier")
q3_o = widgets.Output()
def q3_check(_):
    q3_o.clear_output()
    with q3_o:
        display(_feedback(q3_w.value == "C'est le nom de l'équipement.", ""))
q3_b.on_click(q3_check)
display(q3_w, q3_b, q3_o)



#### Transport

Nous allons maintenant regarder les spécifications concernant le [transport](http://www.openmobilealliance.org/release/LightweightM2M/V1_1_1-20190617-A/OMA-TS-LightweightM2M_Transport-V1_1_1-20190617-A.pdf), c'est-à-dire comment utiliser CoAP sur différents médias comme traditionnellement sur UDP. Mais il est également possible d'envoyer CoAP via des SMS. Allez page 37 au chapitre 6.4.3.

**Quiz : Quiz**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Que répond le serveur à une requête à l'URI /rd ?**"))
q0_w = widgets.RadioButtons(options=['rien', '4.04', "un nouveau URI identifiant la ressource grâce à l'option Location-Path.", "une ressource contenant les informations concernant la vision du serveur de l'objet.", "les informations nécessaires à l'échange de clés pour le chiffrement."], description="", layout=widgets.Layout(width="90%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value == "un nouveau URI identifiant la ressource grâce à l'option Location-Path.", ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)



#### Link-format

Revenons au contenu du message de la requête POST émise initialement par le client. Que veut dire *link-format* ? Nous n'avons pas encore vu ce format. Heureusement, l'IANA est notre ami et en allant chercher à quoi correspond cette valeur, sur cette page <https://www.iana.org/assignments/core-parameters/core-parameters.xhtml#content-formats>, on trouve que le[rfc 6690](https://tools.ietf.org/html/rfc6690)définit ce contenu.

Il utilise un format particulier qui est utilisé initialement par le Web pour définir des relations entre pages. Ce qui est important de remarquer, et après la lecture devient plus claire, c'est que les URI sont notées entre <>. Ensuite, on trouve des attributs liés à cet URI séparés par des points-virgules. Deux URI sont séparés par des virgules.

En suivant ces règles de notation, les données émises par le client peuvent être formatés de la manière suivante :

```
</>;rt="oma.lwm2m",
</1>;ver=1.1,
</1/0>,
</3>;ver=1.1,
</3/0>,
</6/0>,
</3303/0>
```

Le client utilise ce format pour décrire les 7 ressources qu'il possède et qui sont identifiées par ces 7 chemins d'URI. Le nommage des ressources peut paraître étrange ; nous verrons par la suite à quoi il correspond.

**Quiz : Quiz**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**À l'aide du rfc 6690, à quoi correspond l'attribut \"rt\" ?**"))
q0_w = widgets.RadioButtons(options=['resource type', 'retwit', 'retransmission timer', 'reassembly timer'], description="", layout=widgets.Layout(width="90%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value == 'resource type', ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)

display(Markdown("**À quoi sert l'attribut rt ?**"))
q1_w = widgets.RadioButtons(options=['à donner la sémantique (i.e. comment interpréter) de la ressource.', 'à aider à déboguer les échanges.', 'à indiquer la taille de la ressource en faisant référence à lwm2m.', 'à indiquer la version de lwm2m utilisée.'], description="", layout=widgets.Layout(width="90%"))
q1_b = widgets.Button(description="Verifier")
q1_o = widgets.Output()
def q1_check(_):
    q1_o.clear_output()
    with q1_o:
        display(_feedback(q1_w.value == 'à donner la sémantique (i.e. comment interpréter) de la ressource.', ""))
q1_b.on_click(q1_check)
display(q1_w, q1_b, q1_o)



#### Ressources LwM2M

LwM2M représente les ressources d'une manière assez originale pour être à la fois compact, universel (ce qui est quelquefois oxymorique) et flexible. Pour ce faire, tout est désigné par des chiffres.  Le chapitre 7 page 63 du document principal (<http://www.openmobilealliance.org/release/LightweightM2M/V1_1_1-20190617-A/OMA-TS-LightweightM2M_Core-V1_1_1-20190617-A.pdf>) contient le schéma suivant illustrant cette hiérarchie :

![](/static/Capture_d_e_cran_2020-10-22_a__14.45.47.png)

Un objet physique (i.e. notre client) est identifié par un numéro. Ce numéro est attribué par l'OMA. Par exemple, un capteur de température aura la valeur 3303. Une liste des valeurs déjà attribuées peut être trouvée à cette adresse : <http://www.openmobilealliance.org/wp/OMNA/LwM2M/LwM2MRegistry.html#resources>.

Ainsi pour l'objet Température, on trouve la définition suivante :

![](/static/Capture_d_e_cran_2020-10-22_a__14.50.53.png)

Un client peut évidemment posséder plusieurs capteurs de température. Le deuxième chiffre du chemin de l'URI indique cette instance.

Finalement, l'objet peut être complexe et contenir plusieurs informations. En cliquant sur le chiffre 3303 dans la page Web précédente, on obtient la description de l'objet. On peut voir que :

* 5700 représente la valeur mesurée par le capteur ;
* 5601 la valeur minimale ;
* 5602 la valeur maximale ;
* 5701 indique les unités ;
* 5605 permet de remettre à zéro le calcul des minima et maxima ;

Ainsi, 3303/3/5601 représentera la valeur minimale (5601) du 4e capteur (3 car on commence à 0) de la température (3303).

**Quiz : Quiz**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Que représente l'URI 3301/0/5602 ?**"))
q0_w = widgets.RadioButtons(options=['la température maximale du premier capteur', "l'humidité minimale du capteur 0", 'la luminosité maximale du capteur 0'], description="", layout=widgets.Layout(width="90%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value == 'la luminosité maximale du capteur 0', ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)

display(Markdown("**Que représente l'URI 10340/0/2 ?**"))
q1_w = widgets.RadioButtons(options=["la température en Fahrenheit de l'objet 10340", 'la longitude dans des coordonnées GPS', "le statut actif ou non d'une caméra"], description="", layout=widgets.Layout(width="90%"))
q1_b = widgets.Button(description="Verifier")
q1_o = widgets.Output()
def q1_check(_):
    q1_o.clear_output()
    with q1_o:
        display(_feedback(q1_w.value == "le statut actif ou non d'une caméra", ""))
q1_b.on_click(q1_check)
display(q1_w, q1_b, q1_o)

display(Markdown("**/1**"))
q2_w = widgets.RadioButtons(options=['Device', 'LwM2M Server', 'Location', 'Temperature'], description="", layout=widgets.Layout(width="90%"))
q2_b = widgets.Button(description="Verifier")
q2_o = widgets.Output()
def q2_check(_):
    q2_o.clear_output()
    with q2_o:
        display(_feedback(q2_w.value == 'LwM2M Server', ""))
q2_b.on_click(q2_check)
display(q2_w, q2_b, q2_o)

display(Markdown("**/3**"))
q3_w = widgets.RadioButtons(options=['Device', 'LwM2M Server', 'Location', 'Temperature'], description="", layout=widgets.Layout(width="90%"))
q3_b = widgets.Button(description="Verifier")
q3_o = widgets.Output()
def q3_check(_):
    q3_o.clear_output()
    with q3_o:
        display(_feedback(q3_w.value == 'Device', ""))
q3_b.on_click(q3_check)
display(q3_w, q3_b, q3_o)

display(Markdown("**/6**"))
q4_w = widgets.RadioButtons(options=['Device', 'LwM2M Server', 'Location', 'Temperature'], description="", layout=widgets.Layout(width="90%"))
q4_b = widgets.Button(description="Verifier")
q4_o = widgets.Output()
def q4_check(_):
    q4_o.clear_output()
    with q4_o:
        display(_feedback(q4_w.value == 'Location', ""))
q4_b.on_click(q4_check)
display(q4_w, q4_b, q4_o)

display(Markdown("**/3303**"))
q5_w = widgets.RadioButtons(options=['Device', 'LwM2M Server', 'Location', 'Temperature'], description="", layout=widgets.Layout(width="90%"))
q5_b = widgets.Button(description="Verifier")
q5_o = widgets.Output()
def q5_check(_):
    q5_o.clear_output()
    with q5_o:
        display(_feedback(q5_w.value == 'Temperature', ""))
q5_b.on_click(q5_check)
display(q5_w, q5_b, q5_o)

display(Markdown("**Combien d'instance(s) maximum pour un objet ?**"))
q6_w = widgets.Text(layout=widgets.Layout(width="40%"))
q6_b = widgets.Button(description="Verifier")
q6_o = widgets.Output()
def q6_check(_):
    q6_o.clear_output()
    with q6_o:
        display(_feedback(_num_ok(q6_w.value, "1"), ""))
q6_b.on_click(q6_check)
display(q6_w, q6_b, q6_o)



**Quiz : Quiz**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**URI :**"))
q0_w = widgets.Text(layout=widgets.Layout(width="40%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value.strip() == "3311/0/5851".strip(), ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)

display(Markdown("**Valeur Max :**"))
q1_w = widgets.Text(layout=widgets.Layout(width="40%"))
q1_b = widgets.Button(description="Verifier")
q1_o = widgets.Output()
def q1_check(_):
    q1_o.clear_output()
    with q1_o:
        display(_feedback(q1_w.value.strip() == "100".strip(), ""))
q1_b.on_click(q1_check)
display(q1_w, q1_b, q1_o)



#### Représentation des objets

L'OMA attribue des numéros pour les objets. Ces numéros sont uniques et se réfèrent toujours au même objet. De cette manière, il n'y a pas d'ambiguïté possible. Quand un serveur reçoit une valeur contenue dans le premier élément du chemin de l'URI, il connait à la fois la nature de l'objet et son format, c'est-à-dire les ressources qui le composent, leur signification et leur codage.

Si l'on reprend l'objet Température (3303), nous avons vu qu'il est composé de différentes ressources également représentées par un entier identifiant chaque ressource (ressource ID). Par exemple :

* 5700 pour la valeur courante,
* 5601 pour la valeur minimale...

Ces dernières valeurs peuvent être communes à différent objets (comme l'objet Humidité). De même, un annuaire permet de connaître leur signification et leur représentation. Si l'on va sur la page <https://devtoolkit.openmobilealliance.org/OEditor/Default>, on peut avoir la liste des *resourceID* attribués :

* |  |  |  |  |  |  |  |  |
  | --- | --- | --- | --- | --- | --- | --- | --- |
  | 5700 | Sensor Value | R | Float |  |  | IPSO | Last or Current Measured Value from the Sensor. |

* |  |  |  |  |  |  |  |  |
  | --- | --- | --- | --- | --- | --- | --- | --- |
  | 5601 | Min Measured Value | R | Float |  |  | IPSO | The minimum value measured by the sensor since power ON or reset. |
  |
* |  |  |  |  |  |  |  |  |
  | --- | --- | --- | --- | --- | --- | --- | --- |
  | 5701 | Sensor Units | R | String |  |  | IPSO | Measurement Units Definition |

Les valeurs mesurées par les capteurs seront représentées par des nombres flottants, et les unités par une chaîne de caractères.

On voit aussi que les données sont de type R (Read). Le serveur peut donc les lire mais pas les changer.

#### Premier échange

Nous allons nous focaliser sur le premier échange que nous avons commencé à analyser en regardant le message émis par le client vers le serveur LwM2M. Comme nous l'avons vu, ce premier message contient la description des ressources présentes sur le client. Les valeurs des identifiants d'objets (Object ID) et de ressources (Resource ID) permettent au serveur de savoir ce que peut faire le client, vu qu'elles sont normalisées. Le client LwM2M envoie ces informations sur un chemin d'URI bien connu /rd (pour *resource description*).

**Quiz : Quiz**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Que répond le serveur ?**"))
q0_w = widgets.RadioButtons(options=['Il acquitte juste le message.', 'Rien.', "Il renvoie une URI qui servira par la suite à identifier l'objet.", 'Il retourne les données de chiffrement des messages.'], description="", layout=widgets.Layout(width="90%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value == "Il renvoie une URI qui servira par la suite à identifier l'objet.", ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)



**Quiz : Quiz**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Si on laisse la plate-forme fonctionner sans intervenir, que voit-on sur l'analyseur réseau ?**"))
q0_w = widgets.RadioButtons(options=['Rien.', 'Le capteur envoie des informations indiquant un changement de valeur mesurée.', "Le client envoie les valeurs mesurées même s'il elles n'ont pas changé.", "Le client envoie un message vide vers le serveur pour indiquer qu'il est toujours présent.", "Le serveur envoie un message vide vers ses clients pour savoir s'ils sont toujours présents."], description="", layout=widgets.Layout(width="90%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value == "Le client envoie un message vide vers le serveur pour indiquer qu'il est toujours présent.", ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)



#### Interface graphique

Il est temps de revenir à l'interface de la plate-forme en ouvrant la page http://127.0.0.1:8080 depuis le navigateur. Vous devez avoir quelque chose de ce genre :

![](images/lw_server1.gif)

On voit qu'un objet s'y est enregistré et l'on retrouve des informations que l'on avait en disséquant l'échange. Le nom virtual-ilab était présent dans l'URI Query ep=virtual-ilab (ep est pour *end point*). Il est suivi d'une séquence (*Registration ID*) qui correspond au chemin d'URI retourné par le serveur en réponse à la demande d'enregistrement. En mettant la souris sur le i en fin de ligne, on retrouve tous les paramètres envoyés par le client.

Cliquez sur le nom de l'objet qui apparait en couleur et en gras. Une nouvelle page apparaît :

![](images/im35.png)

Le contenu ne devrait maintenant pas vous étonner. On retrouve les objets ID que l'on avait vu en analysant le message d'enregistrement et en allant sur la description des objets. On retrouve dans la deuxième colonne les URI correspondantes. En jetant un œil sur le trafic capturé par l'analyseur, on voit que l'affichage de cette page n'a produit aucun échange sur le réseau. Le bouton coap message en bas à droite de la fenêtre du navigateur nous permet d'avoir les mêmes informations que depuis l'analyseur.

## TP3 - Réalisation d'un client CoAP (invisible des apprenants)

### Architecture du service

#### Text

# Architecture du service

Avec le TP précédent, nous avons pu échanger des messages CBOR entre le pycom et le cloud. Nous allons continuer dans cette voie en ajoutant CoAP dans l'échange pour nommer l'information que nous allons transmettre et indiquer sa représentation.

**Ce TP n'est que pour l'utilisation du PyCom ou on peut utiliser la VM?**

Dans le cloud, le serveur va être décomposé en deux parties:

- un programme relay qui va prendre des requêtes HTTP venant du LNS et envoyer les données à une adresse IP. Inversement, ce programme va attendre les données retournées par cette adresse IP et les envoyer au LNS lors de l'acquittement de la requête HTTP.
- un serveur CoAP standard (aiocoap) sur lequel nous allons traiter les requêtes CoAP en provenance du pycom.

La figure suivante décrit l'architecture globale du service de collecte que nous allons mettre en place, elle est très similaire à ce que nous avons déjà réalisé.

![architecture globale, mise en place du serveur CoAP avec un relai](/static/Capture_d_e_cran_2018-08-31_a__14.06.37.png)

L'architecture se compose du pycom qui va agir comme un client CoAP et va envoyer des requêtes POST sur le réseau LoRaWAN. Grâce à la configuration du LNS, les requêtes CoAP vont être encapsulées dans des requêtes HTTP qui vont être dirigées vers une machine dans le cloud.

Cette machine dans le cloud va traiter ces requêtes. La nouveauté est qu'au lieu de répondre directement, nous allons prendre les  données et les envoyer en IP et UDP vers un serveur CoAP qui va également tourner sur cette machine. Cette articulation permettra d'utiliser un serveur CoAP standard utilisant la pile IP.

Le serveur CoAP peut retourner des données en fonction de la requête. Ces données seront renvoyées au relais. Cette réponse sera transmise dans l'acquittement de la requête HTTP qui va arriver au LNS. Ce dernier enverra les données au pycom.

Pour cette première manipulation, nous allons successivement:

1. Lancer le relais entre HTTP et IP sur la machine dans le cloud. Nous n'allons pas analyser ce programme qui a vocation à être transparent. Vous pouvez bien entendu si vous le voulez regarder dans ses entrailles.
2. Analyser le serveur CoAP qui utilise le framework *aiocoap* et découvrir les paramètres par défaut utilisés par le programme.
3. Construire pas à pas un client CoAP sur le pycom qui interagira avec le serveur.

Ceci va nous servir de base pour construire un service de mesure qui pourra envoyer différents types d'informations.

### Mise en place du serveur

#### Text

### Relais

Dans un premier temps, nous allons lancer le programme de relayage entre le monde HTTP et le monde IP.

Sur la machine dans le cloud, tapez:

python3 CoAPRelay.py -p 7002

en précisant grâce au paramètre -p le numéro de port sur lequel le programme va attendre. C'est toujours le même, fourni dans la mallette pédagogique. Ce programme affiche les messages qui circulent, il est préférable de dédier une fenêtre à son exécution.

### Serveur

Le programme CoAPServer.py est intéressant à étudier.

```python
#!/usr/bin/env python3
# This file is part of the Python aiocoap library project.
#
# Copyright (c) 2012-2014 Maciej Wasilak <http://sixpinetrees.blogspot.com/>,
# 2013-2014 Christian Amsüss <c.amsuess@energyharvesting.at>
#
# aiocoap is free software, this file is published under the MIT license as
# described in the accompanying LICENSE file.
"""This is a usage example of aiocoap that demonstrates how to implement a
simple server. See the "Usage Examples" section in the aiocoap documentation
for some more information."""
import sys, getopt
sys.path.append('./aiocoap')
import binascii
import datetime
import logging
import asyncio
import aiocoap.resource as resource
import aiocoap
import cbor2 as cbor
class BME280(resource.Resource):
"""Manage POST requests coming from the BME280 """
async def render_post(self, request):
print ("request:",request)
print ("options:", request.opt)
print ("payload:", binascii.hexlify(request.payload))
ct = request.opt.content_format or aiocoap.numbers.media_types_rev['text/plain']
if ct == aiocoap.numbers.media_types_rev['application/cbor']:
message = cbor.loads(request.payload)
else:
print ("unsupported content")
self.content = b""
return aiocoap.Message(code=aiocoap.UNSUPPORTED_MEDIA_TYPE ,
payload = self.content)
print ("Message:", message)
self.content = b""
resp = aiocoap.Message(code=aiocoap.CHANGED, payload=self.content)
return resp
def main():
defPort = 7002
try:
opts, args = getopt.getopt(sys.argv[1:],"hp:",["port="])
except getopt.GetoptError:
print ("{0} -p <port> -h".format(sys.argv[0]))
sys.exit(2)
for opt, arg in opts:
if opt == '-h':
print ("{0} -p <port> -h".format(sys.argv[0]))
sys.exit()
elif opt in ("-p", "--port"):
defPort = int(arg)
# Resource tree creation
root = resource.Site()
root.add_resource(('TPH',), BME280())
asyncio.Task(aiocoap.Context.create_server_context(root, bind=("::", 5683+defPort)))
asyncio.get_event_loop().run_forever()
if __name__ == "__main__":
main()
```

Les premières lignes:

```python
import sys, getopt
sys.path.append('./aiocoap')
```

permettent d'aller chercher les modules dans la directory ./aiocoap qui se trouve sur votre compte.

La fonction main commence comme les programmes précédents par analyser les arguments passés en option lors du lancement du programme, en particulier pour récupérer le numéro de port utilisé.

La suite est propre à CoAP.

```
root = resource.Site()
root.add_resource(('TPH',), BME280())
```

Il s'agit d'associer un chemin que l'on trouve dans un URI à un programme qui va traiter la ressource. Ici la fonction BME280 sera appelée.

L'instruction suivante:

```
asyncio.Task(aiocoap.Context.create_server_context(root, bind=("::", 5683+defPort)))
```

permet de lier les ressources précédemment définie à une adresse IP (ici "::" indique toutes les adresses de la machine) et un numéro de port. Pour éviter un conflit avec les ports utilisés par le programme relais, nous le décalons de 5683 (valeur qui correspond par ailleurs au numéro de port par défaut des serveur CoAP).

Ensuite le serveur est lancé par l'instruction suivante qui ne retourne jamais.

```
asyncio.get_event_loop().run_forever()
```

Pour plus d'info sur les API AIOCoAP [c'est par ici (in english)](https://aiocoap.readthedocs.io/en/latest/api.html)

**Quiz : QCM**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Quel devra être l'URI utilisée sur le pycom pour accéder à la ressource?**"))
q0_w = widgets.Text(layout=widgets.Layout(width="40%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value.strip().lower() == "TPH".strip().lower(), ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)



#### Text

La classe BME280 permet de gérer les requêtes venant du pycom pour cette ressource. La fonction render_post est appelée quand un POST est utilisé. Il est possible de créer d'autres fonctions pour les autres méthodes de coap (get, put, delete,...).

Les lignes suivantes nous permettent de voir ce que reçoit cette fonction.

```python
print ("request:",request)
print ("options:", request.opt)
print ("payload:", request.payload, binascii.hexlify(request.payload))
```

La variable request reçue en argument est affichée, suivie des options de la requête CoAP et de la payload.

Avec

```
ct = request.opt.content_format or aiocoap.numbers.media_types_rev['text/plain']
```

on affecte dans la variable ct la valeur du content_format, c'est-à-dire le type de l'information, si cette variable n'existe pas, alors la valeur par défaut correspond à du texte 'text/plain' est utilisée (voir l'utilisation de l'opérateur [or](https://docs.python.org/3/reference/expressions.html#or)).

Ensuite on teste cette variable pour lire correctement le contenu. Dans cet exemple simple seul un contenu indiqué en CBOR (valeur 60) est accepté. Les autres valeurs conduisent à retourner un message d'erreur UNSUPPORTED_CONTENT_FORMAT (4.15).

Si le contenu est du CBOR, les données sont transfromées en JSON

```
message = cbor.loads(request.payload)
```

et affiché après les tests, si d'autres formats avaient été possibles.

```python
print ("Message:", message)
```

Pour la réponse, le code CHANGED (2.04) est envoyé et la partie données est laissée vide.

```
self.content = b""
return aiocoap.Message(code=aiocoap.CHANGED, payload=self.content)
```

#### Text

Lancez le programme CoAPServer sur votre machine dans le cloud en précisant votre numéro de port:

$ ./CoAPServer.py -p 7002

Notez que la première ligne du programme:

```
#!/usr/bin/env python3
```

permet d'indiquer qu'il s'agit d'un programme en python3, il n'est plus nécessaire de le préciser lors de son exécution.

### Module CoAP.py

Dans cette partie, nous allons construire pas à pas un client CoAP sur le Pycom qui va pouvoir interagir avec le serveur que nous venons de lancer sur votre machine dans le cloud. Le fonctionnement est très similaire à celui mis en œuvre pour envoyer des données codées en CBOR dans le TP précédent.

### Le module CoAP

Dans le master, vous disposez d'un module CoAP.py qui contient une mise en œuvre réduite de CoAP adaptée aux réseaux LPWAN.

Nous pouvons découvrir les méthodes en envoyant directement les commandes au pycom.

Le constructeur de la classe, peut être appelé

- soit sans argument et dans ce cas un objet coap vide est créé:

>>> **import CoAP as coap**  
>>> **c1 = coap.CoAP()**

- soit avec une chaîne d'octets qui seront affecté à cet objet:

>>> **c2 = coap.CoAP(binascii.unhexlify("44020c3cd19796c1c13c"))**

L'objet CoAP peut être visualisé via un print ou en utilisation la méthode dump qui donne plus de détails.

>>> **print (c1)**  
CoAP message: Len 0 Value b''  
>>> **c1.dump()**  
empty CoAP message  
>>> **print(c2)**  
CoAP message: Len 10 Value b'44020c3cd19796c1c13c'  
>>> **c2.dump()**  
CON 0x0c3c [b'd19796c1']  
0.02 (POST)  
Content-Format:0x3c

#### Créer un en-tête CoAP

Pour rappel, l'en-tête CoAP se code sous la forme:

![Entête CoAP](images/CoAPHeader.png)

Pour créer un en-tête CoAP, on utilise la méthode new_header en fournissant le Type, le Code et si nécessaire le Token et le message ID (MID). Attention à la lettre en majuscule devant ces mots clef.

>>> **c1.new_header(Type=coap.RST, Code=coap.PUT)**  
>>> **c1.dump()**  
RST 0x0001 [b'']  
0.03 (PUT)  
>>> **c1.new_header(Type=coap.NON, Code=coap.GET, Token=0x1234, MID=None)**  
>>> **c1.dump()**  
NON 0x000e [b'3412']  
0.01 (GET)

Ensuite, si l'on veut ajouter des en-têtes, différentes méthodes peuvent être utilisées:

* add_option_URI_path (path) qui prend comme argument une chaîne de caractères contenant un élément du chemin. **Attention: elle ne doit pas contenir le caractère /.** Si vous devez indiquer un chemin du genre /element/souselement, la syntaxe sera:

**>>> c1.****add_option_URI_path("element")**  
****>>> c1.****add_option_URI_path("souselement")****

* add_option_content (val) qui prend comme argument un entier donnant la nature des données transportées. Les valeurs suivantes ont prédéfinies :

+ CONTENT_TEXT, CONTENT_JSON, CONTENT_CBOR

* add_option_noResponse (val) qui prend comme argument les drapeaux indiquant quel type de messages doivent être acquittés:

+ NO_2, NO_4 ou NO_5. Le chiffre correspond à celui utilisé par les notificiations CoAP :

- 2 notification positive du serveur, 4 erreur du client et 5 erreur du serveur.

+ Ils peuvent être combinés avec l'opérateur |. Par exemple : NO_2 | NO_4 permet de ne pas acquitter les messages 2 (notification positive du serveur) et 4 (erreur du client).

* end_option() qui est utilisé pour indiquer la fin des options.

**Ces méthodes doivent être utilisées dans l'ordre dans lequel l'en-tête CoAP les attend (i.e. l'ordre indiqué ci -dessus).**

Pour ajouter des données au message CoAP. Il faut utiliser la méthode :

* add_value (data) où data est une chaîne de caractère ou un objet CBOR.

**Quiz : Text Input**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Donnez la représentation hexadécimale de ce message.**"))
q0_w = widgets.Text(layout=widgets.Layout(width="40%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value.strip().lower() == "b'4002c007b87072657373696f6e086d696e696d616c65113cff1903e8'".strip().lower(), ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)



### Ecriture du client

#### Text

### POST non acquitté

La requête la plus simple que l'on peut envoyer au serveur est un POST non acquitté, c'est-à-dire utilisant le type NON.

Le programme send-coap-TPH.py qui se trouve dans le répertoire téléchargé de github montre comment former une requête CoAP. Sa structure est la même que celle utilisée pour émettre du CBOR.

```
post = coap.CoAP()
post.new_header(Type=coap.NON, Code=coap.POST, Token = 12)
post.add_option_URI_path("foo") #temperature pression humidity
post.end_option()
post.add_value(c)
```

La classe CoAP est appelée pour créer l'objet post.

Il commence par les champs obligatoires de CoAP. Le type CoAP est mis à NON (non comfirmable) pour indiquer que l'autre extrémité ne devra pas acquitter ce message CoAP. Le champ code est mis à POST pour écrire une ressource sur le serveur. Les autres champs obligatoires seront remplis automatiquement.

L'option URI-path est ensuite ajoutée pour indiquer le nom de la ressource suivi de la fin d'option pour indiquer le début des données qui contiendront la structure CBOR.

Le listing suivant montre un échange, la première ligne montre le message émis par le pycom et la seconde la réponse reçue par le pycom du serveur distant:

b'50020005b3666f6fff84021900c819019019012c'  
b'60840005'

Pour rappel l'entête de trames CoAP

![CoAP header](images/CoAPHeader.png)

**Quiz : QCM**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Pour rappel, vous avez des indications surl'entête CoAP ici**"))
q0_w = widgets.RadioButtons(options=['0', '1', '4', '6'], description="", layout=widgets.Layout(width="90%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value == '1', ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)

display(Markdown("**Pour rappel, vous avez des indications surl'entête CoAP ici**"))
q1_w = widgets.RadioButtons(options=['CON', 'NON', 'ACK', 'RST', 'POST', 'GET'], description="", layout=widgets.Layout(width="90%"))
q1_b = widgets.Button(description="Verifier")
q1_o = widgets.Output()
def q1_check(_):
    q1_o.clear_output()
    with q1_o:
        display(_feedback(q1_w.value == 'NON', ""))
q1_b.on_click(q1_check)
display(q1_w, q1_b, q1_o)

display(Markdown("**Pour rappel, vous avez des indications surl'entête CoAP ici**"))
q2_w = widgets.RadioButtons(options=['CON', 'NON', 'ACK', 'RST', 'POST', 'GET'], description="", layout=widgets.Layout(width="90%"))
q2_b = widgets.Button(description="Verifier")
q2_o = widgets.Output()
def q2_check(_):
    q2_o.clear_output()
    with q2_o:
        display(_feedback(q2_w.value == 'POST', ""))
q2_b.on_click(q2_check)
display(q2_w, q2_b, q2_o)

display(Markdown("**Pour rappel, vous avez des indications surl'entête CoAP ici**"))
q3_w = widgets.Text(layout=widgets.Layout(width="40%"))
q3_b = widgets.Button(description="Verifier")
q3_o = widgets.Output()
def q3_check(_):
    q3_o.clear_output()
    with q3_o:
        display(_feedback(_num_ok(q3_w.value, "5"), ""))
q3_b.on_click(q3_check)
display(q3_w, q3_b, q3_o)

display(Markdown("**Pour rappel, vous avez des indications surl'entête CoAP ici**"))
q4_w = widgets.RadioButtons(options=['2.01 Created', '2.02 Deleted', '2.03 Valid', '2.04 Changed', '2.05 Content', '4.00 Bad Request', '4.01 Unauthorized', '4.02 Bad Option', '4.03 Forbidden', '4.04 Not Found', '4.05 Method Not Allowed', '4.06 Not Acceptable', '4.12 Precondition Failed', '4.13 Request Entity Too Large', '4.15 Unsupported Content-Format', '5.00 Internal Server Error', '5.01 Not Implemented', '5.02 Bad Gateway', '5.03 Service Unavailable', '5.04 Gateway Timeout', '5.05 Proxying Not Supported'], description="", layout=widgets.Layout(width="90%"))
q4_b = widgets.Button(description="Verifier")
q4_o = widgets.Output()
def q4_check(_):
    q4_o.clear_output()
    with q4_o:
        display(_feedback(q4_w.value == '4.04 Not Found', ""))
q4_b.on_click(q4_check)
display(q4_w, q4_b, q4_o)

display(Markdown("**Question**"))
q5_w = widgets.RadioButtons(options=['redemarrer le pycom', "supprimer l'option URI-path", 'aucune, le serveur traite correctement la requête', "modifier la valeur de l'option URI-path", 'redémarrer le serveur', 'changer la valeur du Message ID'], description="", layout=widgets.Layout(width="90%"))
q5_b = widgets.Button(description="Verifier")
q5_o = widgets.Output()
def q5_check(_):
    q5_o.clear_output()
    with q5_o:
        display(_feedback(q5_w.value == "modifier la valeur de l'option URI-path", ""))
q5_b.on_click(q5_check)
display(q5_w, q5_b, q5_o)



#### Text

### Modification du programme Client sur le pycom

Comme le montre l'échange précédent et le résultat de votre exécution, le serveur retourne un message d'erreur, car le serveur n'a pas la possibilité de créer une ressource sous ce nom.

**Modifiez le programme send-coap-TPH.py pour utiliser le bon nom de ressource.**

**Quiz : QCM**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Question**"))
q0_w = widgets.RadioButtons(options=['2.01 Created', '2.02 Deleted', '2.03 Valid', '2.04 Changed', '2.05 Content', '4.00 Bad Request', '4.01 Unauthorized', '4.02 Bad Option', '4.03 Forbidden', '4.04 Not Found', '4.05 Method Not Allowed', '4.06 Not Acceptable', '4.12 Precondition Failed', '4.13 Request Entity Too Large', '4.15 Unsupported Content-Format', '5.00 Internal Server Error', '5.01 Not Implemented', '5.02 Bad Gateway', '5.03 Service Unavailable', '5.04 Gateway Timeout', '5.05 Proxying Not Supported'], description="", layout=widgets.Layout(width="90%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value == '4.15 Unsupported Content-Format', ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)

display(Markdown("**Question**"))
q1_w = widgets.RadioButtons(options=['redemarrer le pycom', "supprimer l'option URI-path", 'aucune, le serveur traite correctement la requête', "modifier la valeur de l'option URI-path", 'redémarrer le serveur', 'ajouter une option décrivant le contenu.'], description="", layout=widgets.Layout(width="90%"))
q1_b = widgets.Button(description="Verifier")
q1_o = widgets.Output()
def q1_check(_):
    q1_o.clear_output()
    with q1_o:
        display(_feedback(q1_w.value == 'ajouter une option décrivant le contenu.', ""))
q1_b.on_click(q1_check)
display(q1_w, q1_b, q1_o)



#### Text

**Modifiez le programme send-coap-TPH.py pour ajouter l'option Content-Format** grâce à la fonction suivante:

    post.add_option_content(coap.CONTENT_CBOR)

Attention à respecter l'ordre des options qui ne peuvent être mises que de manière croissante (cf. [liste des options CoAP](https://www.iana.org/assignments/core-parameters/core-parameters.xhtml#option-numbers)).

**Quiz : QCM**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Question**"))
q0_w = widgets.RadioButtons(options=['2.01 Created', '2.02 Deleted', '2.03 Valid', '2.04 Changed', '2.05 Content', '4.00 Bad Request', '4.01 Unauthorized', '4.02 Bad Option', '4.03 Forbidden', '4.04 Not Found', '4.05 Method Not Allowed', '4.06 Not Acceptable', '4.12 Precondition Failed', '4.13 Request Entity Too Large', '4.15 Unsupported Content-Format', '5.00 Internal Server Error', '5.01 Not Implemented', '5.02 Bad Gateway', '5.03 Service Unavailable', '5.04 Gateway Timeout', '5.05 Proxying Not Supported'], description="", layout=widgets.Layout(width="90%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value == '2.04 Changed', ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)

display(Markdown("**Question**"))
q1_w = widgets.RadioButtons(options=['redemarrer le pycom', 'aucune, le serveur traite correctement la requête', "modifier la valeur de l'option URI-path", 'redémarrer le serveur', 'ajouter une option décrivant le contenu.'], description="", layout=widgets.Layout(width="90%"))
q1_b = widgets.Button(description="Verifier")
q1_o = widgets.Output()
def q1_check(_):
    q1_o.clear_output()
    with q1_o:
        display(_feedback(q1_w.value == 'aucune, le serveur traite correctement la requête', ""))
q1_b.on_click(q1_check)
display(q1_w, q1_b, q1_o)



Suppression de l'acquittement

Dans les réseaux LPWAN, l'utilisation de messages descendants (du réseau vers l'objet) est coûteux. Il est préférable de ne pas acquitter un message périodique de peu d'importance.

Nous utilisons des message de type NON acquitté, donc au niveau transport nous ne recevons pas d'acquittements. Par contre au niveau REST, la requête est toujours notifiée.

Pour se faire, nous allons utiliser l'option No_Response qui permet de supprimer les notifications de type 2.XX, 4.XX ou 5.XX.

Pour ce faire nous allons rajouter l'instruction [post.add_option_noResponse()](/jump_to_id/40ddd40c56384c6f878102a2d419bb3a)dans notre programme. [Consultez ce lien pour revenir sur la section du cours consacré à cette option](/jump_to_id/40ddd40c56384c6f878102a2d419bb3a).

Pour aller plus loin sur le détail de l'option No Response dans la norme, [vous pouvez également consulter ce document (in english)](https://tools.ietf.org/html/rfc7967).

**Quiz : Text Input**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Réponse numérique**"))
q0_w = widgets.Text(layout=widgets.Layout(width="40%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(_num_ok(q0_w.value, "3"), ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)



### Acquittements

#### Objet CoAPClient

Pour prendre en compte les différents types de messages CoAP (CON, ACK et NON), nous allons utiliser l'objet CoAPClient :

client = coap.CoAPClient(s)

où s est une socket de la famille LoRaWAN dont la déclaration est identique aux programmes précédents.

Pour émettre un message CoAP, il faut utiliser la méthode send :

    client.send(post)

où post est un objet CoAP.

La méthode sleep permet ensuite d'attendre le nombre de secondes indiqué en argument.

    client.sleep(35)

Si le message CoAP est de type *confirmable* (CON), le client CoAP cherchera à le retransmettre jusqu'à qu'un *acquittement* (ACK) soit reçu. Les message de type *non confirmable* (NON) seront envoyés comme précédemment.

Vous allez à présent écrire un programme qui envoie toutes les 2 secondes un tableau CBOR contenant les mesures du capteur avec une précision au centième sur l'URI /TPH.

Si la température dépasse un seuil que vous identifierez sur votre capteur, un message d'alerte sera envoyé par le capteur à l'URI /alert. Ce message devra être confirmé.

Attention, vous devrez aussi modifier le programme CoAPServer.py sur le cloud pour y ajouter le traitement de l'URI /alert.

Pour finir, vous allez tester différents acquittements en fonction des type de message et de la présence ou non du paramètre "No response".

Qulles sont les valeurs que prennent A, B, C et D dans la matrice suivante ?

![Matrice](images/MatriceTP3_part3.PNG)

**Quiz : QCM**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Question**"))
q0_w = widgets.RadioButtons(options=["Il n'y a pas de réponse reçue", 'ACK 0.00 (EMPTY)', 'ACK 2.04 (CHANGED)', 'CON 2.04 (CHANGED)', 'NON 0.00 (EMPTY)', 'RST 0.00 (EMPTY)', 'PUT 2.04 (CHANGED)'], description="", layout=widgets.Layout(width="90%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value == 'ACK 2.04 (CHANGED)', "Explication Le type de message CON va indiquer que le message doit être acquité. L'absence de paramètre No response, indique que le serveur va envoyer un message positif (2.04 = CHANGED)."))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)

display(Markdown("**Question**"))
q1_w = widgets.RadioButtons(options=["Il n'y a pas de réponse reçue", 'ACK 0.00 (EMPTY)', 'ACK 2.04 (CHANGED)', 'CON 2.04 (CHANGED)', 'NON 0.00 (EMPTY)', 'RST 0.00 (EMPTY)', 'PUT 2.04 (CHANGED)'], description="", layout=widgets.Layout(width="90%"))
q1_b = widgets.Button(description="Verifier")
q1_o = widgets.Output()
def q1_check(_):
    q1_o.clear_output()
    with q1_o:
        display(_feedback(q1_w.value == 'ACK 2.04 (CHANGED)', "Explication Le type de message NON va indiquer que le message n'a pas besoin d'être acquité. L'absence de paramètre No response, indique que le serveur va envoyer un message positif (2.04 = CHANGED), il y aura donc un ACK ou un CON envoyé par le serveur."))
q1_b.on_click(q1_check)
display(q1_w, q1_b, q1_o)

display(Markdown("**Question**"))
q2_w = widgets.RadioButtons(options=["Il n'y a pas de réponse reçue", 'ACK 0.00 (EMPTY)', 'ACK 2.04 (CHANGED)', 'CON 2.04 (CHANGED)', 'NON 0.00 (EMPTY)', 'RST 0.00 (EMPTY)', 'PUT 2.04 (CHANGED)'], description="", layout=widgets.Layout(width="90%"))
q2_b = widgets.Button(description="Verifier")
q2_o = widgets.Output()
def q2_check(_):
    q2_o.clear_output()
    with q2_o:
        display(_feedback(q2_w.value == 'ACK 0.00 (EMPTY)', "Explication Le type de message CON va indiquer que le message doit être acquité. Le paramètre No response = N0_2, indique que le serveur ne va pas renvoyer de message positif, l'acquittement reçu sera vide."))
q2_b.on_click(q2_check)
display(q2_w, q2_b, q2_o)

display(Markdown("**Question**"))
q3_w = widgets.RadioButtons(options=["Il n'y a pas de réponse reçue", 'ACK 0.00 (EMPTY)', 'ACK 2.04 (CHANGED)', 'CON 2.04 (CHANGED)', 'NON 0.00 (EMPTY)', 'RST 0.00 (EMPTY)', 'PUT 2.04 (CHANGED)'], description="", layout=widgets.Layout(width="90%"))
q3_b = widgets.Button(description="Verifier")
q3_o = widgets.Output()
def q3_check(_):
    q3_o.clear_output()
    with q3_o:
        display(_feedback(q3_w.value == "Il n'y a pas de réponse reçue", "Explication Le type de message NON va indiquer que le message n'a pas besoin d'être acquité. Le paramètre No response = N0_2, indique que le serveur ne va pas renvoyer de message positif. Il n'y aura donc pas de message envoyé depuis le serveur."))
q3_b.on_click(q3_check)
display(q3_w, q3_b, q3_o)



### Endormir son Pycom

## Piloter mon apprentissage

### Check micro-compétences

**Quiz : Auto-évaluation des compétences**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**À la fin de cette semaine, je suis capable de :** _(plusieurs reponses possibles)_"))
q0_w = widgets.SelectMultiple(options=['Expliquer les protocoles REST et CoAP', 'Émettre des données structurées sur un réseau IoT avec un objet et recevoir ces données sur un serveur en utilisant les protocoles REST et CoAP'], layout=widgets.Layout(width="90%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(set(q0_w.value) == {'Expliquer les protocoles REST et CoAP', 'Émettre des données structurées sur un réseau IoT avec un objet et recevoir ces données sur un serveur en utilisant les protocoles REST et CoAP'}, "Si vous n'êtes pas à l'aise avec certains éléments présentés dans la liste, nous vous suggérons de revoir les vidéos et activités associées à ces compétences avant de passer à la suite."))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)



## REST, Publish-subscribe et MQTT (invisible des apprenants)

### Principe de REST

REST (Representational State Transfer) est une philosophie d’implémentation pour les services Web, qui est aussi utilisée pour les services IoT. On parle aussi d'architecture logicielle.

Selon Roy FIELDING (qui a défini REST), REST est « un ensemble de principes, de propriétés et de contraintes». REST utilise le modèle de communication client-serveur et utilise généralement le protocole HTTP (Hypertext Transfer Protocol).

Voici les 6 propriétés qui définissent REST :

- **Architecture client-serveur** : Les serveurs sont les entités qui fournissent des services (tels que l'heure, le partage de données, etc.) et les clients sont les entités qui demandent ces services.
- **Sans état (ou stateless)** : Cela signifie que l'état de l'interaction entre le client et le serveur n'est pas stocké. Chaque requête client est traitée comme une nouvelle demande indépendante. De cette façon, le serveur n'a pas besoin de stocker, suivre ou analyser les requêtes précédentes du même client.
- **Cacheability** : les périphériques intermédiaires ainsi que les clients peuvent mettre en cache les réponses pour une utilisation ultérieure. Cela conduit à améliorer de performances.
- **Système en couches** : cela signifie que les services peuvent être réalisés en utilisant des couches telles que différents serveurs responsables de différentes parties des services (stockage, recherche, etc.), mais le client ne sait pas s'il est connecté au serveur final ou intermédiaire.
- **Code à la demande** (facultatif) : les serveurs peuvent éventuellement envoyer du code exécutable aux clients.
- **Interface uniforme** : elle fait référence aux principes selon lesquelles une ressource doit avoir une seule représentation, telle que les URIs (Uniform Resource Identifier), et suivre certaines directives pour la dénomination, le format de lien et de données, etc. Il doit avoir suffisamment d'informations sur le traitement. Les ressources initiales, telles que la page d'accueil d'un site Web, devraient avoir d'autres liens leur permettant de découvrir d'autres ressources.

REST utilise généralement HTTP qui fonctionne comme un protocole de demande-réponse, comme montre le schéma ci dessous. Certaines des méthodes HTTP sont les suivantes : **mettre l'image plus haut, pour voir que cette phrase fait référence aux bullet points qui sont plus bas.. ?**

![](https://lh5.googleusercontent.com/fKnngvJDvir8JiDyjjqs2urtzn8KSf2sBYP8Hc78rxW35LUovHuVlfFW-GAuW82fl5BO3NttNT_A62FqtTI2abcZn58egTi7EmuQzsG5_Owcqo8-Z5TLbCCUcM-EYz9o_Vk-Asv1)

*Request-Response using HTTP*

- **GET** est utilisée pour récupérer la représentation d'une ressource (par exemple page web, valeur de température d’un capteur, etc.). Par exemple voici le format d’en-tête HTTP GET pour récupérer ma page web (screenshot d’outil wireshark **Petit lien descriptif de shark ou explication de son interet ?**)  :

![](https://lh3.googleusercontent.com/pul1wKIVLR6mvjcyqobMNDeD5IUMi8CyL5GsCrSWHXT7B5QOTIW0_2Hn6mQHind5VN1Y56GkZM3d4pgBziF4io0Cyn6VyQtahGLcAmowrp1iCmEF95jG1N7I_cR942fEIuplGrGA)

*Message HTTP GET*

- **HEAD** est utilisée pour récupérer uniquement les métadonnées présentes dans les en-têtes de réponse sans le corps de réponse.
- **POST** est utilisée pour spécifier un serveur **C'est a dire specifier un serveur ? Cela passe par la création d'un URI non ?** pour accepter la ressource par exemple en le stockant, en créant une nouvelle entrée etc.
- **PUT** est utilisée pour stocker une ressource à l'endroit identifié par l'URI dans la requête. Si la ressource existe déjà, elle sera modifiée.
- **DELETE** est utilisée pour supprimer la ressource définie.

### Application de REST dans notre cas

#### Text

# Application de REST dans notre cas

**Ca manque un peu d'illustrations ou schéma**

Avec notre capteur, nous pouvons définir plusieurs types de remontées de mesure. Par exemple:

- les séries temporelles, qui permettent de suivre la température dans un lieu pendant une période ;
- les remontées des trois mesures possibles : température, humidité, pression ;
- un message d'alerte si la température monte soudainement, un incendie pouvant en être à l'origine.

Si le capteur est capable d'envoyer ces trois types de mesures, il faut que la machine qui va les recevoir puisse distinguer ces trois types pour les traiter correctement.

Une solution est de les nommer, par exemple **évolution_température**, **mesures_brutes**, **alerte_incendie**. De cette manière le récepteur peut sans ambiguïté appliquer le bon traitement.

Nous avons également vu la représentation des données en CBOR. Bien qu'elle soit intéressante pour les données provenant de capteurs, elle n'est pas la seule. Il serait donc utile de pouvoir également indiquer quel est le format des données transportées. Par exemple des données au format CBOR, un texte en ASCII, du JSON, ...

Ces notions de nommage et de typage existent déjà dans les protocoles utilisés pour le Web et nous pouvons les généraliser à l'Internet des Objets. Les blocs de données que l'on a envoyé sont appelées des **ressources**. Elles sont désignées localement sur la machine par un nom, ou plutôt un chemin car le nommage peut être hiérarchique, par exemple:

- /mesures/temperature
- /mesures/pression
- /mesures/humidité
- /alerte/incendie
- ...

Cela permet d'introduire deux types d'équipements :

- les **serveurs** qui stockent les ressources ;
- les **clients** qui veulent y accéder et les manipuler.

Le client a à sa disposition un petit nombre d'actions simples. Le nom entre parenthèses indique le nom de la **primitive** permettant de réaliser cette action :

- Accéder au contenu d'une ressource sur un serveur (GET)
- Écrire le contenu d'une ressource sur un serveur (PUT)
- Modifier le contenu d'une ressource sur un serveur (PATCH)
- Créer une ressource sur un serveur (POST)
- Effacer une ressource sur un serveur (DELETE)

Un client qui manipule des ressources par un nombre limité de primitives est à la base d'une architecture appelée **REST** (Representational State Transfer) qui permet de construire des services très résistant à la charge dont nous allons voir les propriétés. Si elles sont vraies et ont démontré leur succès pour le Web, elles peuvent s'appliquer avec peu de modification à l'Internet des Objets.

### **Architecture REST**

Nous allons illustrer dans ce chapitre un certain nombre des propriétés nécessaires pour rendre un service compatible avec l'architecture REST.

#### Nommage uniforme

Pour permettre de distinguer une ressource de manière non ambiguë, on utilise un nommage appelé URI (*Universal Resource Identifier*) qui est représenté par une chaîne de caractère. Elle contient deux parties, la première spécifie le format de cette chaîne (ou schéma) par un mot clé (http, sip, ...) suivi de : puis d'une séquence propre à chaque schéma identifié, par exemple:

- https://www.iana.org/assignments/uri-schemes/uri-schemes.xhtml#uri-schemes-1 pointe sur une ressource qui décrit les schémas enregistrés
- sip:alice@atlanta.com est utilisé en téléphonie sur IP pour identifier un correspondant.

- ...

Dans ce cours nous allons nous focaliser sur les URI utilisés par le web et l'Internet des objets, cela veut dire que le schéma sera du type:

```
http://nom.du.serveur:port/chemin/vers/la/ressource?valeur=max&date=20180701
```

Le schéma, valable pour http et sa version chiffrée https, contient:

- le nom du serveur qui peut être un nom de machine, une adresse IP,
- si le port n'est pas celui par défaut (80 pour http, 443 pour https), il peut être précisé après l'adresse du serveur.
- un chemin vers la ressource, qui est composé d'éléments séparés par le caractère /

Il faut bien comprendre qu'accéder à une ressource n'est pas équivalent à lire un fichier dans un répertoire.  L'accès à une ressource peut demander une action de la part du serveur (qui peut être le pycom). Par exemple, GET /temperature peut se traduire par l’exécution d'un code (ici le serveur est le pycom) qui va retourner la température, cette valeur n'étant pas initialement stockée dans un fichier.

Il est possible d'apporter des paramètres lors de la requête pour préciser que ce l'on veut du serveur. Ces paramètres sont représentés par le caractère ? et les différents paramètres sont séparés par des &. Dans l'exemple précédent, on a précisé **Ce sont des standards ? C'est la méthode REST qui impose ca ?**

```
valeur=max&date=20180701
```

on précise dans ce cas au serveur que l'on veut la valeur max pour une date donnée. Le client doit connaître au préalable les questions (query) possibles.

Quand le serveur répond à une primitive, il ajoute des méta-données (c'est-à-dire des informations supplémentaires) à la ressource indiquant par exemple sa taille ou son type.

#### Typage des données

Le serveur informe du type de la ressource. Le client peut de cette manière connaître la nature de celle-ci et effectuer le traitement approprié. En effet on n'affiche pas une image de la même manière qu'un texte. Une méta donnée ajoutée par le serveur est le type de contenu (content type).

### Quiz 05

**Quiz : QCM**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("** Quels sont les propriétés qui définissent REST? (2 réponses correctes)** _(plusieurs reponses possibles)_"))
q0_w = widgets.SelectMultiple(options=['Architecture Client-Serveur', 'Sans-état', 'Abonnement', 'QoS'], layout=widgets.Layout(width="90%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(set(q0_w.value) == {'Architecture Client-Serveur', 'Sans-état'}, "Les 6 propriétés qui définissent REST sont : Architecture Client-Serveur, Sans-état (ou Statelessness), Cacheability, Système en couche, Code à la demande, Interface uniforme."))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)



#### Text

Nous allons analyser un certain nombre d'échange HTTP faits avec le logiciel curl. Ce dernier permet de faire simplement des requêtes en utilisant le protocole HTTP. Vous pouvez trouver facilement la documentation de curl en ligne (par exemple [ici](https://curl.haxx.se/docs/manual.html)). Vous pouvez également l'installer sur votre ordinateur pour reproduire les exemples qui vous seront donnés dans la suite de cet exercice.

#### Text

Nous allons taper la commande suivante:

%**curl -v http://outils.plido.net/index.html**

**Ca manque un peu de contexte. Où est-ce qu'on tape cette commande?**

et l'on obtient comme réponse:

 1 | \*   Trying 176.31.185.138...  
 2 | \* Connected to outils.plido.net (176.31.185.138) port 80 (#0)  
 3 | > GET /index.html HTTP/1.1  
 4 | > Host: outils.plido.net  
 5 | > User-Agent: curl/7.43.0  
 6 | > Accept: \*/\*  
 7 | >   
 8 | < HTTP/1.1 200 OK  
 9 | < Date: Sun, 01 Jul 2018 14:49:42 GMT  
10 | < Server: Apache/2.4.7 (Ubuntu)  
11 | < Last-Modified: Tue, 23 Dec 2014 12:54:12 GMT  
12 | < ETag: "b1-50ae1abee6500"  
13 | < Accept-Ranges: bytes  
14 | < Content-Length: 177  
15 | < Vary: Accept-Encoding  
16 | < Content-Type: text/html  
17 | <    
18 | <html><body><h1>It works!</h1>  
19 | <p>This is the default web page for this server.</p>  
20 | <p>The web server software is running but no content has been added, yet.</p>  
21 | </body></html>  
22 | \* Connection #0 to host outils.plido.net left intact

**Quiz : QCM**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Quelle est l'URI ? **"))
q0_w = widgets.RadioButtons(options=['/index.html', 'outils.plido.net/index.html', 'http://outils.plido.net/index.html', 'http://outils.plido.net'], description="", layout=widgets.Layout(width="90%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value == 'http://outils.plido.net/index.html', "Pour avoir l'URI, il faut concaténer un schema qui décrit le format de l'URI, ici http. Le nom du serveur (ici outils.plido.net), voire un numéro de port s'il est différent de 80 et le chemin demandé dans la requête (/index.html)"))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)

display(Markdown("**Quelle requête est envoyée au serveur ? **"))
q1_w = widgets.RadioButtons(options=['GET', 'PUT', 'POST', 'FETCH', 'DELETE'], description="", layout=widgets.Layout(width="90%"))
q1_b = widgets.Button(description="Verifier")
q1_o = widgets.Output()
def q1_check(_):
    q1_o.clear_output()
    with q1_o:
        display(_feedback(q1_w.value == 'GET', "Donné ligne 3 du listing."))
q1_b.on_click(q1_check)
display(q1_w, q1_b, q1_o)

display(Markdown("**Quel type de contenu retourne le serveur ? **"))
q2_w = widgets.RadioButtons(options=['JSON', 'CBOR', 'HTML', 'JPEG'], description="", layout=widgets.Layout(width="90%"))
q2_b = widgets.Button(description="Verifier")
q2_o = widgets.Output()
def q2_check(_):
    q2_o.clear_output()
    with q2_o:
        display(_feedback(q2_w.value == 'HTML', "Donné ligne 16 du listing."))
q2_b.on_click(q2_check)
display(q2_w, q2_b, q2_o)

display(Markdown("**A quelle ligne dans le listing précédent commence la ressource?**"))
q3_w = widgets.Text(layout=widgets.Layout(width="40%"))
q3_b = widgets.Button(description="Verifier")
q3_o = widgets.Output()
def q3_check(_):
    q3_o.clear_output()
    with q3_o:
        display(_feedback(_num_ok(q3_w.value, "18"), "Entre l'en-tête et la ressource, il y a une ligne vide."))
q3_b.on_click(q3_check)
display(q3_w, q3_b, q3_o)

display(Markdown("**A quelle ligne dans le listing précédent se termine la ressource?**"))
q4_w = widgets.Text(layout=widgets.Layout(width="40%"))
q4_b = widgets.Button(description="Verifier")
q4_o = widgets.Output()
def q4_check(_):
    q4_o.clear_output()
    with q4_o:
        display(_feedback(_num_ok(q4_w.value, "21"), "Avant le message de fermeture de la connexion."))
q4_b.on_click(q4_check)
display(q4_w, q4_b, q4_o)

display(Markdown("**Quelle est la fonction du thermomètre **"))
q5_w = widgets.RadioButtons(options=["clientOù se trouve la ressource? c'est à dire l'information", 'serveur', 'client et serveur', 'Aucun des 2'], description="", layout=widgets.Layout(width="90%"))
q5_b = widgets.Button(description="Verifier")
q5_o = widgets.Output()
def q5_check(_):
    q5_o.clear_output()
    with q5_o:
        display(_feedback(q5_w.value == 'serveur', "Si la requête provient d'un téléphone portable, le thermometre est le serveur car il possède la ressource température. A noter que le thermomètre pourrait aussi envoyer des POST, dans ce cas il serait un client."))
q5_b.on_click(q5_check)
display(q5_w, q5_b, q5_o)



### Publish-subscribe

# Publish-subscribe

Un paradigme de l’architecture de communication pour IoT est le « publish-subscribe ».

**Il faudrait expliquer comment le "publish-subscribe se situe par rapport à REST. Je ne comprends pas bien le rapport entre les 2.**

**Je trouve également que le terme de "paradigme" est assez obscur. On pourrait parler de mécanisme d'échange de données (?)**

Ce modèle fait le découplage entre l'expéditeur d'un message de son destinataire. Dans ce paradigme, il existe des « Publishers » qui produisent des données ou des messages et envoient le message à une entité généralement appelée « Broker ». En outre, les messages peuvent être classés en « Topics », contenus ou types, etc.

Ensuite, il existe des abonnés qui s’abonnent au broker, par exemple à un topic donné, afin de recevoir les messages qui les intéressent, comme montré dans le schéma. Le broker peut alors utiliser des filtres pour envoyer uniquement ces messages aux abonnés qui les intéressent.

![](https://lh6.googleusercontent.com/jEPNzMDvsYKTaBOb8vwUM8YsiAWNJ4Fk50xS-vQfrq-3qq7mKt-Ignio7vaRCZnTJB7mJE9it4fSgIXIjhgUSfZzPlmOl3zRei71F5d4JGxUCYNkrRJ5wPNYDCqJHjEgg7TJZ4Bg)

Imaginons, par exemple, que plusieurs capteurs soient installés dans deux bâtiments A et B. Certains capteurs collectent des informations sur la température et d'autres collectent des informations sur l'humidité. Ces capteurs peuvent envoyer les données régulièrement à un broker central. Les données peuvent être classées en différentes rubriques qui peuvent également être organisées en manière hiérarchiques.

Par exemple, le topic /capteur pourrait signifier toutes les données de capteur, /sensor/buildingA/pourrait signifier des données de capteurs uniquement installées dans le bâtiment A. En plus, /sensor/buildingA /temperature pourrait signifier des données de capteurs de température installés uniquement dans le bâtiment A, etc. Ensuite, certains abonnés peuvent s'abonner aux messages en fonction de leur intérêt.

Ainsi, un abonné intéressé uniquement par les données d'humidité du bâtiment B peut s'abonner au sujet /sensor/buildingB/humidité et le broker n'enverra ces données qu'à cet abonné, etc.

Les principaux avantages du paradigme publish-subscribe par rapport au paradigme client-serveur, tels que inclus en REST, sont les suivants :

1. Pas beaucoup de couplage entre émetteur et récepteur
2. Passage à l’échelle.

Absence de couplage entre l'expéditeur et le destinataire se fait en termes d'espace, de temps et de synchronisation. L’expéditeur n’a pas besoin de connaître ou de se connecter au récepteur car il n’en a qu’à envoyer au broker. Ainsi, l'expéditeur et le destinataire n'ont pas besoin de communiquer en même temps. Cela permet du duty-cycling **(un peu mal dit)**comme l'expéditeur peut envoyer le message au broker et ensuite il peut dormir en utilisant le duty-cycling.

**On a expliqué le duty cycling avant? Oui**

De même, chaque fois que les abonnés se réveillent, ils peuvent recevoir le message du broker qui, à son tour, peut mettre les messages en attente. Cela peut être avantageux pour les appareils IoT qui doivent économiser de l'énergie. Dans le scénario IoT, l'expéditeur et le destinataire peuvent être les périphériques IoT et le broker peut être un puissant ordinateur, un relais ou un nœud qui, comparativement, n’a pas de contraintes de ressources.

En comparaison, il existe un couplage entre les clients et le serveur dans le paradigme client-serveur. Ainsi, le client et le serveur doivent être en ligne pendant la communication.

Un autre avantage du publish-subscribe est le passage à l’échelle. Par rapport au paradigme client-serveur, la publish-subscribe bénéficie d'opérations en parallèle et de meilleures possibilités de mise en cache des messages pour atteindre l'évolutivité lorsque le nombre de nœuds connectés est élevé. 'il y a un grand nombre d'objet qui envoie des données sur une seule chaine. (?)

### MQTT

# MQTT

**Je propose de mettre Publish-Subscribe et MQTT dans une section aller plus loin car ça apporte de la confusion**

Certains des protocoles Publish-Subscribe sont **MQTT** (Message Queuing Telemetry Transport), AMQP (Advanced Message Queuing Protocol), JMS (Java Messaging Service) et XMPP (Extensible Messaging Protocol et Présence). **Peut etre expliquer les possibles differences entre protocoles de facon generique ? pourquoi y en a il plusieurs ?**

MQTT (Message Queuing Telemetry Transport) est un protocole publish-subscribe. Il est  très léger et conçu pour les périphériques de faible puissances. Il a une très petite empreinte logiciel et est optimisé pour fonctionner dans les environnements à faible bande passante. Cela rend MQTT idéal pour les applications IoT.

MQTT, étant un protocole publish-subscribe, utilise un broker pour diffuser les messages. Les publishers diffusent un message à broker vers un topic. Les abonnés inscrivent aux topics et le broker envoie les messages aux abonnés en fonction de leurs intérêts.

MQTT utilise le protocole TCP comme protocole de transport. TCP assure une communication fiable, néanmoins, il peut être lourd pour les communications IoT. MQTT fournit également **3 niveaux de QoS** pour la communication entre un expéditeur et un destinataire : Niveau 0 - Au plus une fois, Niveau 1 - Au moins une fois, Niveau 2 - Exactement une fois. Notez qu'un expéditeur et un destinataire, par exemple, peuvent être un client et un broker.

- Niveau 0 : avec le niveau de qualité de service le plus bas « Au plus une fois », le message est envoyé une seule fois sans utilisation d'accusé de réception. C'est le mode le moins fiable. L'expéditeur envoie le message et ne le stocke ensuite pas dans sa file d'attente d'envoi. Le message n'est pas acquitté. Le destinataire recevra soit 1 copie du message, soit aucune copie. Il ne recevra aucune copie dupliquée.
- Niveau 1 : Le niveau de qualité de service « Au moins une fois » garantit la livraison d'un message. L'expéditeur envoie le message et attend l'accusé de réception. Il continue d'envoyer le message régulièrement jusqu'à l’accusé de réception arrive. Le destinataire peut recevoir des copies d'un message.
- Niveau 2 : Le niveau de QoS le plus élevé « Exactement une fois », garantit la livraison d'un message et évite également le traitement inutile des messages en double sur le récepteur. Ceci est fait en utilisant au minimum 2 aller-retours de 4 communications. L'expéditeur envoie un message et attend l'accusé de réception. Lorsque le destinataire reçoit un message, il envoie l’accusé de réception. Si l'expéditeur ne reçoit pas d'accusé de réception, il renvoie le message après un certain intervalle. Lorsque l'expéditeur reçoit l'accusé de réception, il envoie un message « release ». Le message « release » permet au destinataire, à sa réception, de terminer tout le traitement lié au message. Le récepteur, au cas où il s'agirait d'un broker, envoie le message aux abonnés et supprime le message de sa file d'attente. Le récepteur envoie ensuite un message « complet » à l'expéditeur. Jusqu'à ce que l'expéditeur ne reçoive pas un « complet », il renvoie le « release » après un certain intervalle. Lors de la réception du « complet », l'expéditeur libère également les états associés au message et le supprime de sa file d'attente.

### Quiz 06

**Quiz : QCM**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("** 1. Qu'est-ce qui est vrai avec Publish-Subscribe? **"))
q0_w = widgets.RadioButtons(options=['Les publishers doivent connaître leurs abonnés', 'Les publishers n’ont pas besoins de connaître leurs abonnés', 'Les abonnés ne peuvent pas choisir les messages à recevoir en fonction de leurs intérêts.', 'Aucune de ces réponses'], description="", layout=widgets.Layout(width="90%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value == 'Les publishers n’ont pas besoins de connaître leurs abonnés', ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)

display(Markdown("** 2. Laquelle de ces catégories appartient au paradigme publish-subscribe? **"))
q1_w = widgets.RadioButtons(options=['Sujet', 'Contenu', 'Type', 'Les trois'], description="", layout=widgets.Layout(width="90%"))
q1_b = widgets.Button(description="Verifier")
q1_o = widgets.Output()
def q1_check(_):
    q1_o.clear_output()
    with q1_o:
        display(_feedback(q1_w.value == 'Les trois', "Sujet, Contenu et Type. Les messages peuvent être classés en sujets (topics), contenus ou types, donc les trois sont des catégories de messages du paradigme publish-subscribe."))
q1_b.on_click(q1_check)
display(q1_w, q1_b, q1_o)

display(Markdown("** 3. Quel est le niveau de QoS le plus élevé pour MQTT? **"))
q2_w = widgets.RadioButtons(options=['Au plus une fois', 'Au moins une fois', 'Exactement une fois', 'Exactement deux fois'], description="", layout=widgets.Layout(width="90%"))
q2_b = widgets.Button(description="Verifier")
q2_o = widgets.Output()
def q2_check(_):
    q2_o.clear_output()
    with q2_o:
        display(_feedback(q2_w.value == 'Exactement une fois', "Exactement une fois à un niveau de qualité de service 2. Ce niveau est le plus élevé pour MQTT."))
q2_b.on_click(q2_check)
display(q2_w, q2_b, q2_o)

display(Markdown("** 4. MQTT utilise quel protocole de transport? **"))
q3_w = widgets.RadioButtons(options=['UDP', 'TCP', 'RTP', 'RTCP'], description="", layout=widgets.Layout(width="90%"))
q3_b = widgets.Button(description="Verifier")
q3_o = widgets.Output()
def q3_check(_):
    q3_o.clear_output()
    with q3_o:
        display(_feedback(q3_w.value == 'TCP', "TCP est le protocole de transport utilisé par MQTT."))
q3_b.on_click(q3_check)
display(q3_w, q3_b, q3_o)



## Info cours (invisible des apprenants)

### Info cours

## CoAP-old (invisible des apprenants)

### Découvrir CoAP

#### Text

### Limites de la philosophie REST

La philosophie d'implémentation REST est devenue très populaire en raison de à sa grande résistance à la charge. Les serveurs ont des informations appelées ressource qui sont manipulées par des clients grâce au protocole HTTP. Ceci est très bien adapté au monde du Web. Appliquer cette architecture au monde de l'Internet des Objets présente des avantages. Les données mesurées par des capteurs, les ordres envoyés à des actionneurs peuvent se modéliser également sous forme de ressources, offrant ainsi une grande interopérabilité.

Mais quand on pense un serveur, on a souvent à l'esprit une machine ou un ensemble de machines puissantes, placées dans des salles climatisées, capable de traiter plusieurs milliers de transactions la seconde. En fait, un serveur est l'équipement qui a une ressource qui peut être lue ou écrite; donc, le capteur ou l'actionneur. Dans ce cas, le client qui peut être un ordinateur, une tablette ou un smartphone a généralement beaucoup plus de capacité de traitement, d'autonomie électrique, de mémoire que le serveur. D'où la nécessité de simplifier aux maximum les fonctions du serveur.  
  
HTTP utilise TCP pour fiabiliser les communications entre le client et le serveur. Or, TCP est gourmand en ressources. Il faut de la mémoire pour stocker les paquets non acquittés ou hors séquence, un grand nombre d'heuristiques doivent être mises en œuvre pour améliorer ses performances. Pour récupérer une page Web très simple, une vingtaine de paquets peuvent être nécessaires.

HTTP souffre également d'une mise en œuvre assez complexe. Pour offrir plus de souplesse, un en-tête HTTP est composé de chaînes de caractères de taille quelconque.   
Si l'on reprend un exemple simple de requête HTTP, on voit se dessiner le format des échanges. Le client ouvre une connexion TCP vers le serveur et envoie les données suivantes:

GET /index.html HTTP/1.1  
Host: outils.plido.net  
User-Agent: curl/7.43.0  
Accept: \*/\*

Le serveur répond en envoyant les données suivantes:  
  
HTTP/1.1 200 OK  
Date: Sun, 01 Jul 2018 14:49:42 GMT  
Server: Apache/2.4.7 (Ubuntu)  
Last-Modified: Tue, 23 Dec 2014 12:54:12 GMT  
ETag: "b1-50ae1abee6500"  
Accept-Ranges: bytes  
Content-Length: 177  
Vary: Accept-Encoding  
Content-Type: text/html  
   
<html><body><h1>It works!</h1>  
<p>This is the default web page for this server.</p>  
<p>The web server software is running but no content has been added, yet.</p>  
</body></html>  
  
HTTP privilégie la flexibilité en définissant les en-têtes protocolaires en caractères ASCII. Un en-tête est composé d'un mot clé suivit de ":"suivit d'une valeur. Par exemple:

User-Agent: curl/7.43.0

Le serveur doit donc analyser le texte pour y retrouver les informations nécessaires. Le serveur doit également envoyer une réponse suivant ce format qui peut s'avérer aussi de grande taille et donc occuper plus de ressources sur le lien radio.

### CoAP

Le protocole CoAP a donc été créé pour les applications contraintes. Il va simplifier le traitement des requêtes sur le serveur et réduire l'information qui devra être transportée sur le lien radio. De plus CoAP utilise le protocole UDP qui nécessite une empreinte mémoire nettement inférieure à celle de TCP. Il est principalement défini dans le [rfc 7252](https://tools.ietf.org/html/rfc7252).

Comme son nom l'indique ***Co**nstrained **A**pplication **P**rotocol*, ce protocole est conçu pour des équipements contraints en taille mémoire et en puissance de calcul. Comme son nom ne l'indique pas CoAP est l'équivalent de HTTP pour l'Internet des Objets. Au lieu de favoriser une représentation textuelle de l'en-tête (comme http), CoAP va revenir à un certain nombre de champs, codés en binaires. De plus, la représentation en CoAP est beaucoup plus compacte et la plupart de l'information se concentre sur un mot de 32 bits. Un jeu d'options va permettre de le compléter.

CoAP repose principalement sur UDP, bien qu'une mise en œuvre sur TCP ou sur SMS existe. Dans notre cas, nous l'utiliserons directement sur LoRaWAN.

#### Types de messages

Aucun de ces supports n'offre une garantie que le récepteur recevra le message, or il peut être parfois nécessaire de fiabiliser le transfert des données. Pour ce faire, CoAP dispose d'une sous-couche implantant un protocole de type Send and Wait. Chaque message contient un champ message ID et trois types de trames sont disponibles :

- les messages de type CON pour *confirmable* indiquent qu'ils doivent être acquitté par le récepteur,
- les message de type ACK pour *acknowledgment*contiennent cet acquittement, le champ message ID du message à acquitter est recopié dans ce message. L'ACK peut également contenir des données,
- les message de type NON pour *non confirmable* sont de purs datagrammes, ils ne seront pas acquittés par le récepteur, leur perte ne sera pas détectée par le protocole CoAP. Par contre, le champ message ID permet de détecter des messages dupliqués.

Un message de type RST (*Reset*) s'ajoute aux trois types précédents, il peut être émis par exemple quand un des nœuds a perdu son contexte suite un redémarrage et ne sait plus traiter les réponses qu'il reçoit.

![confirmable exchange](/static/Capture_d_e_cran_2018-09-14_a__23.13.59.png)  
  
La figure précédente montre un échange fiabilisé avec le protocole CoAP. Les messages de type CON impliquent un acquittement en retour. Dans un premier échange, la station de gauche émet un message de type CON avec la valeur 0x1234 pour le message ID. En même temps, elle arme un temporisateur. Un message de type ACK arrive avant l'expiration de ce temporisateur contenant la même valeur de Message ID. L'émetteur voit sa donnée confirmée, désarme le temporisateur. Dans le second échange, avec la valeur de Message ID 0x1235, aucun acquittement n'arrive à l'émetteur. Dans ce cas le temporisateur de l'émetteur expire et la donnée est réémise. A noter que dans ce cas, le récepteur reçoit deux fois le message 0x1235. Il acquitte le message, mais ne traite pas son contenu. Le récepteur mémorise pendant à peu près 5 minutes les valeurs du Message ID associé à l'émetteur pour éviter de traiter plusieurs fois les retransmission.  
  
Un message émis à moins de 5 minutes d'intervalle sera donc ignoré par le récepteur.

#### Token

Le serveur n'a pas toujours la réponse de manière instantanée. Si le serveur attend trop avant de répondre, le temporisateur du client va se déclencher et la même requête va être inutilement envoyée de nouveau au serveur. Pour dé-corréler acquittement des messages et réponse à ce message, le standard prévoit l'utilisation d'un token (jeton) qui est recopié dans la réponse à une requête comme le montre la figure suivante:

![token](/static/Capture_d_e_cran_2018-09-14_a__23.27.07.png)  
Le client émet une requête vers le serveur, elle est fiabilisée car le message est de type CONfirmable et la valeur du message ID est 0x1234. Comme le serveur ne peut immédiatement répondre à la requête, il ne fait que l'acquitter. A noter que la requête contient un Token ayant la valeur 12.  
Quelques instants plus tard, il obtient la valeur à répondre. Il envoie un message de type CONfirmable vers le client. Les valeurs du champ Message ID sont différentes puisque ces valeurs sont choisies localement, par contre le token contient la même valeur que pour la requête. De cette manière, le client peut corréler la réponse avec la requête.

### Quiz 07

#### En-tête principale de CoAP

![CoAP Header](images/CoAPHeader.png)

La figure précédente montre le début d'un en-tête CoAP:

* Le champ Ver, sur 2 bits, contient le numéro de version du protocole qui vaut 01 dans la version actuelle
* Le champ T pour Type, également sur 2 bits, indique la nature du message (00 : CONfirmable, 01 : NON confirmable, 10 : Acquittement, 11 : Reset).
* Le champ TKL, sur 4 bits, donne la longueur en octets du champ token démarrant au deuxième mot de 32 bits. Si la valeur est à 0, ce champ est absent. Les valeurs de 1 à 8 indiquent la longueur en octets, les valeurs de 9 à 15 ne sont pas autorisées.
* Nous détaillerons le champ Code par la suite. Il fait 8 bits et contient la nature des requêtes et celles des réponses. Leur codage est compatible avec celui utilisé par HTTP.
* Le champ Message ID sur 16 bits contient une valeur différente à chaque nouveau message pour les messages CON et NON, et recopie cette valeur pour un acquittement.

**Quiz : QCM**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Quelle est le type de ce message (indiquez la valeur numérique)?**"))
q0_w = widgets.Text(layout=widgets.Layout(width="40%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value.strip().lower() == "0".strip().lower(), ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)

display(Markdown("**44020c3cd19796c1c13c**"))
q1_w = widgets.Dropdown(options=['CON', 'NON', 'ACK', 'RST'], description="", layout=widgets.Layout(width="90%"))
q1_b = widgets.Button(description="Verifier")
q1_o = widgets.Output()
def q1_check(_):
    q1_o.clear_output()
    with q1_o:
        display(_feedback(q1_w.value == 'CON', ""))
q1_b.on_click(q1_check)
display(q1_w, q1_b, q1_o)

display(Markdown("**Combien vaut le champ Message ID (en hexadécimal)?**"))
q2_w = widgets.Text(layout=widgets.Layout(width="40%"))
q2_b = widgets.Button(description="Verifier")
q2_o = widgets.Output()
def q2_check(_):
    q2_o.clear_output()
    with q2_o:
        display(_feedback(q2_w.value.strip().lower() == "0c3c".strip().lower(), ""))
q2_b.on_click(q2_check)
display(q2_w, q2_b, q2_o)

display(Markdown("**Quelle est la longueur, en octets, du champ token ?**"))
q3_w = widgets.Text(layout=widgets.Layout(width="40%"))
q3_b = widgets.Button(description="Verifier")
q3_o = widgets.Output()
def q3_check(_):
    q3_o.clear_output()
    with q3_o:
        display(_feedback(q3_w.value.strip().lower() == "4".strip().lower(), ""))
q3_b.on_click(q3_check)
display(q3_w, q3_b, q3_o)

display(Markdown("**Et la valeur du champ token (en hexadécimal)?**"))
q4_w = widgets.Text(layout=widgets.Layout(width="40%"))
q4_b = widgets.Button(description="Verifier")
q4_o = widgets.Output()
def q4_check(_):
    q4_o.clear_output()
    with q4_o:
        display(_feedback(q4_w.value.strip().lower() == "d1 97 96 c1".strip().lower(), ""))
q4_b.on_click(q4_check)
display(q4_w, q4_b, q4_o)



### Le champ code

![CoAP header simple](images/CoAPHeaderSimple.png)

Le champ Code, sur 1 octet, permet un codage assez subtil de la nature de la requête ou de la réponse. Dans beaucoup de protocoles applicatifs comme FTP ou le Web, le serveur rend un code sur 3 caractères indiquant si la requête s’est exécutée correctement.

Les codes commençant par le chiffre :

- 1 informent que la requête est en train d’être traitée normalement. Ce type de notification n’est pas pris en compte avec CoAP.
- 2 indiquent que la requête a été acceptée et traitée correctement.
- 3 permettent d’indiquer une indirection.
- 4 font référence à une erreur du coté client, soit due à une mauvaise syntaxe ou la requête ne peut pas être traitée. Ainsi la célèbre erreur 404 indique que le client a demandé une page qui n’existe pas sur le serveur. Donc la faute est de son coté.
- 5 désignent une erreur du coté du serveur.

Les deux chiffres suivants indiquent plus précisément la nature de la confirmation ou de l'erreur.

Pour permettre une représentation plus compacte, CoAP va coder cette chaîne de 3 caractères dans un seul octet. Les trois bits de gauche servent pour indiquer la nature du code et les 5 suivant à droite donneront la raison. Ainsi le code d’erreur HTTP "415" (*Unsupported Media Type*) se note en CoAP 4.15, s’écrit en binaire 100.01111 et en décimal 143.

Cette notation, concerne les réponses aux requêtes, mais elle laisse de la place pour coder également les requêtes. En effet, si le premier champ vaut 0,  il ne code pas une notification (qui utilise des valeurs de 1 à 5). Alors, plusieurs requêtes compatibles avec l’architecture REST peuvent être codées dans la deuxième partie du champ code :

- GET, codé 0x01, retrouve la valeur d’une ressource présente sur le serveur et désignée par un URI.
- POST, codé 0x02, stocke une valeur sur une ressource existante présente sur le serveur.
- PUT, codé 0x03, crée une ressource sur le serveur et lui affecte une valeur.
- DELETE, codé 0x04, supprime une ressource sur le serveur.

D'autres codes sont également possibles.

La liste des codes utilisés par CoAP est disponible [ici](https://www.iana.org/assignments/core-parameters/core-parameters.xhtml#codes).

**Quiz : QCM**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

def _feedback(ok, explanation=None):
    msg = "**Correct !**" if ok else "**Pas tout a fait.**"
    if explanation:
        msg += "\n\n" + explanation
    return Markdown(msg)
def _num_ok(given, expected):
    try:
        return abs(float(given.strip().replace(",", ".")) - float(expected.strip().replace(",", "."))) < 1e-6
    except Exception:
        return given.strip() == expected.strip()

display(Markdown("**Valeur binaire de la première partie du champ code (sur 3 bits)**"))
q0_w = widgets.Text(layout=widgets.Layout(width="40%"))
q0_b = widgets.Button(description="Verifier")
q0_o = widgets.Output()
def q0_check(_):
    q0_o.clear_output()
    with q0_o:
        display(_feedback(q0_w.value.strip().lower() == "100".strip().lower(), ""))
q0_b.on_click(q0_check)
display(q0_w, q0_b, q0_o)

display(Markdown("**Valeur binaire de la seconde partie du champ code (sur 5 bits)**"))
q1_w = widgets.Text(layout=widgets.Layout(width="40%"))
q1_b = widgets.Button(description="Verifier")
q1_o = widgets.Output()
def q1_check(_):
    q1_o.clear_output()
    with q1_o:
        display(_feedback(q1_w.value.strip().lower() == "00100".strip().lower(), ""))
q1_b.on_click(q1_check)
display(q1_w, q1_b, q1_o)

display(Markdown("**A quelle valeur décimale cela correspond ?**"))
q2_w = widgets.Text(layout=widgets.Layout(width="40%"))
q2_b = widgets.Button(description="Verifier")
q2_o = widgets.Output()
def q2_check(_):
    q2_o.clear_output()
    with q2_o:
        display(_feedback(q2_w.value.strip().lower() == "132".strip().lower(), ""))
q2_b.on_click(q2_check)
display(q2_w, q2_b, q2_o)

display(Markdown("**44 02 0c 3c d1 97 96 c1 c1 3c**"))
q3_w = widgets.Dropdown(options=['GET', 'POST', 'PUT', 'DELETE', 'Page Not Found'], description="", layout=widgets.Layout(width="90%"))
q3_b = widgets.Button(description="Verifier")
q3_o = widgets.Output()
def q3_check(_):
    q3_o.clear_output()
    with q3_o:
        display(_feedback(q3_w.value == 'POST', ""))
q3_b.on_click(q3_check)
display(q3_w, q3_b, q3_o)



### Codage des options

Le codage des options, qui suivent l'en-tête et le token, est surement la partie la plus complexe de CoAP et aussi la plus efficace en terme de compacité. En contre partie, le traitement de ces options par une machine est relativement simple et se fait en quelques lignes de codes.

Le champ option va contenir des options qui vont soit servir à améliorer le protocole de transfert des données entre le client et le serveur, soit servir à coder les en-têtes des requêtes et des réponses, en garantissant une certaine compatibilité avec les en-têtes HTTP.

![option coap](/static/Capture_d_e_cran_2018-09-19_a__13.06.07.png)

La structure utilisée est dite TLV pour Type Longueur Valeur. Chaque champ contient au moins les deux premières informations de type qui vont indiquer la nature de l’option et sa longueur indiquant la taille des données. Si ce dernier n’est pas nul, les données vont se trouver après. L’interêt de cette notation est qu’elle permet de gérer une certaine évolutivité du protocole. Ainsi si un récepteur ne reconnaît pas un type, il peut très bien passer au suivant grâce à l’information de longueur. Cela permet également d’avoir des options de longueur variable.

CoAP complique un peu la chose en optant pour les différences entre les types consécutifs. Ainsi si l’on doit envoyer une option de type 5, puis deux de type 6, le codage contiendra ∆T =5, ∆T =1, ∆T =0.

Si la différence est supérieure ou égale à 13, la valeur 13 (0xd) est mise dans le champ ∆T et un entier sur 8 bit code la différence moins 13. Par exemple, si les deux options sont de type 5 et 15, le codage sera le suivant ∆T = 5, ∆T = 13 + 2. De même, si la différence n’est codable que sur 2 octets, la valeur mise dans le champ est 14 (0xe) et les deux octets contiennent la différence entre les deux options moins 269.

L’intérêt de cette représentation est qu’elle est dans la plupart des cas plus compacte mais aussi qu’elle impose d’ordonner les options ce qui peut simplifier leur traitement du côté du serveur.

Pour la longueur on retrouve le même principe, les longueurs inférieures à 13 sont codées directement. Si elles sont supérieures ou égale à 13, la valeur moins 13 est codée dans un octet supplémentaire. Une valeur de 14 indique que deux octets sont utilisé pour coder la longueur moins 269.

La valeur 0xFF dans les champs Delta Type et Longueur annoncent la fin des options et le début des données du message CoAP. Si le champ donnée pouvant être vide, elle n'est pas transmise.

### Les options de CoAP

Les options sont distinguées par un entier qui sera codé de manière relatives via les delta dans les en-têtes CoAP.

Le premier bit sert à coder le type décrit. Quand il est positionné à 1, ce type doit être connu du récepteur (critique). Dans ce cas, si un destinataire reçoit une option de ce type est qu’il ne la connaît pas, il doit produire un message d’erreur. Dans le cas contraire (premier bit à 0), cette option inconnue est ignorée du récepteur qui poursuit le traitement des options suivantes. Ainsi les options paires sont facultatives et les impaires critiques.

Le nombre d'option de CoAP évolue continuellement. Une liste complète peut être trouvée [ici](https://www.iana.org/assignments/core-parameters/core-parameters.xhtml#option-numbers) avec les documents qui les décrivent, plus particulièrement, nous trouvons :

* **Uri-Host (3)** Contient dans la partie valeur, sous forme d'une chaîne de caractères, le nom du serveur d’une URI (nom, adresse IPv4 ou IPv6). Généralement, il n’est pas nécessaire de le préciser puisque les messages CoAP sont envoyé à cette adresse.
* **Observe (6)** Peut contenir un entier. Un serveur peut de cette manière informer de l'état d'une ressource, périodiquement, lorsque qu'elle change ou atteint une seuil critique, un client qui s'est préalablement abonné pour cette ressource. Dans la réponse, la valeur dans l'option Observe doit toujours augmenter. Nous consacrerons une section à l'utilisation d'Observe.
* **Uri-Port (7)** Contient le numéro du port UDP sur lequel CoAP est lancé. Généralement ce champ n’est pas nécessaire vu que le serveur CoAP attend déjà des messages sur ce port. Comme pour l'option Uri-Host, elle est peut présente dans les en-têtes CoAP.
* **Location-Path (8)** Utilisé en réponse à une requête POST. Elle contient une chaîne de caractères indiquant un segment du chemin de la ressource identifiant la ressource que le serveur a créée.
* **Uri-Path (11)** Contient une chaîne de caractères donnant un des segments de l'URI. Cette option peut être répétée plusieurs fois dans l'en-tête CoAP.
* **Content-Format (12)** Contient un entier qui définit le codage de l'information dans la ressource. Si elle n'est pas présente, la ressource est considérée comme text/plain (0). Par exemple (la liste complète est [ici](https://www.iana.org/assignments/core-parameters/core-parameters.xhtml#content-formats)) :

+ 0 : text/plain; charset=utf-8
+ 40 : application/link-format
+ 50 : application/json
+ 60 : application/cbor

* **Max-Age (14)** Indique dans un entier la durée, en seconde, pendant laquelle la ressource peut être maintenue en cache.
* **Uri-Query (15)** Contient les segments d’interrogation, sous forme de chaîne de caractères, que l’on retrouve dans les URI. Cette option peut être répétée plusieurs fois dans l'en-tête CoAP.
* **Accept (17)** Indique le format que le client attend du serveur pour coder la ressource.
* **Location-Query (20)** Utilisé en réponse à une requête POST pour indiquer le chemin de la ressource.
* **Proxy-Uri (35)** Contient une URI qui doit être prise en compte par le proxy.
* **Proxy-Scheme (39)** Indique le protocole que l'on trouve en début d'URI.
* **Size1 (60)** indique la taille de la ressource.

### Représentation d’une URI

On comprend mieux la signification de certaines options listée ci-dessus, quand la syntaxe d’un URI est connue (cf. [RFC 2396](https://tools.ietf.org/html/rfc3986)). La syntaxe générale est :

<scheme>:<scheme-specific-part>

où scheme va définir le schéma de notation (une liste exhaustive est donnée [ici](http://www.iana.org/assignments/ uri-schemes/uri-schemes.xhtml)). La partie spécifique est plus connue pour des schémas qui font appel à un serveur comme http ou https voire CoAp, elle peut s’écrire de la manière suivante :

<scheme>://*userinfo@*host*:port*/path?query

où les champs en italique userinfo@ et :port sont facultatifs. Ils contiennent respectivement le nom de l’utilisateur et le numéro de port sur lequel tourne le service. path va être composé d’une série de segments séparés par des caractères / qui localisent la ressource sur le serveur. L’URI peut se terminer des questions, c’est-à-dire une chaîne de caractères qui sera interprétée par le serveur pour la ressource précédemment désignée. Une question peut contenir plusieurs parties séparées par le caractère &.

Ceci peut être vérifié par le codage dans un en-tête CoAP de l’URI suivante :

coap://192.168.1.52/capteur1/temperature?max_value&date=20131206

Elle utilise le schéma de nommage de CoAp, le serveur est 192.168.1.52, le chemin (path) est composé de deux segments, suivi par deux questions.   
40 01 BE BF|B8 63 61 70 74  - 65 75 72 31|0B 74 65 @.?Àücapt eur1.te  
6D 70 65 72 61 74 75 72 65  - 49|6D 61 78 5F 76 61 mperature Imax_va  
6C 75 65|0D 00 64 61 74 65  - 3D 32 30 31 33 31 32 lue..date =201312  
30 36                                              06  
ver:1 Type = 0 (CON) Token Length = 0 code 1 (GET) Msg id = BEBF  
Option = 11 (+11) length = 8  
Uri-Path capteur1  
Option = 11 (+0) length = 11  
Uri-Path temperature  
Option = 15 (+4) length = 9  
Uri-Query max_value  
Option = 15 (+0) length = 13  
Uri-Query date=20131206  
  
On notera que ni le schéma, ni le serveur (et le numéro de port s'il avait été présent) ne sont codé dans l'en-tête car le serveur connaît son adresse IP et le numéro de port sur lequel s’exécute le serveur CoAP. Ils pourraient être utiles en cas de virtualisation du serveur, c’est-à-dire si plusieurs instances de CoAP tournaient, soit à des noms différents, soit sur des numéros de port différents. Si cette possibilité existe et est très utilisée en HTTP, pour l’instant, la faible capacité des ressources ne pousse pas vers une virtualisation des capteurs.

Par contre le chemin est décomposé en deux parties, avec la répétition des options Uri-path. Il en va de même avec les questions (Uri-Query). Le codage par delta fait que la première option contient le delta T 11 (0xb). Comme pour la deuxième option le delta T est de 0, on reste sur une option Uri-path. Le delta T pour la troisième option est 4, il s'agit de l'option 11+4 soit Uri-Query (15).

### Observe

### Observe

Avec l’architecture REST, le serveur répond toujours aux requêtes d’un client. CoAP permet également au serveur d’envoyer périodiquement un rafraîchissement de la valeur d’une ressource, par exemple quand celle-ci change de valeur ou atteint un seuil. La figure suivante illustre ce fonctionnement. Le client envoie une requête GET avec le type Observe vide. Si le serveur l’accepte, il retourne la valeur de la ressource en indiquant une valeur dans le type Observe, et il répète le token donné par le client. La valeur donnée dans le champ Observe doit croître pour permettre au client de re-séquencer les réponses qui peuvent plus ou moins être retardées sur le réseau. Dans l’exemple, la troisième réponse avec la valeur d’Observe de 30 est retardée et arrive chez le client après la quatrième. Le client la rejette. 

![observe](/static/Capture_d_e_cran_2018-09-22_a__20.16.28.png)

### No Response

L'option No response définit dans le [RFC 7967](https://datatracker.ietf.org/doc/rfc7967/) permet à un client d'informer le serveur qu'il ne souhaite pas recevoir de notifications lors d'un POST ou d'un PUT.  Ceci peut être utile quand le canal de communication est limité (par exemple, les communications entre les véhicules, les réseaux LPWAN) ou quand une requête est envoyée en diffusion et produirait trop de réponses.

La valeur de l'option est 258 et elle est suivie d'un octet qui va indiquer quel type de notification va être émis :

+ 2 : acquittement par le serveur
+ 4 : erreur du client
+ 5 : erreur du serveur